# Frozen reference-profile slope workflow

This is the active, end-to-end investigation and QA notebook for the promoted base-network slope workflow. It works from frozen realised networks, their single-centreline reference sections, and filtered multi-pass RiverSP/SWOT observations. It is safe to rerun: it does not change masks, topology, widths, or network geometry.

The prior candidate-only review notebook is retained under `notebooks/legacy/`.

## Scope and durable implementation

- Corridor/profile inputs: `rapid_tools.reference_profile_inputs.build_reference_profile_inputs`.
- Profile fit/artifacts: `rapid_tools.reference_profile.fit_reference_profile`, `select_reference_profile`, and `write_reference_profile_artifacts`.
- Base-domain solve: `network_variants.reference_profile_candidate.solve_boundary_conditioned_domain`.
- Controlled promotion: `network_variants.run_promote_slope_refit_candidate_contexts`.

The 16 promoted base contexts retain their prior contexts in a recoverable promotion backup. Child-state contexts, equivalent resistance, and RAPID must be refreshed subsequently from these base profiles.

## Workflow overview

1. **Freeze the realised network and longitudinal corridor.** Project realised inlet/outlet and retained SWORD nodes onto the frozen single centreline; use the connected SWORD reach corridor with the reviewed 1 km endpoint buffer.
2. **Filter and assemble multi-pass observations.** Each valid RiverSP WSE observation is \(z_{ij}\) at profile node \(i\) and acquisition \(j\); SWORD P10/P50/P90 and MAD are QA evidence, not independent head observations.
3. **Fit a smooth monotone reference profile.** Estimate \(g(x_i)\) and zero-centred acquisition offsets \(a_j\) from \(z_{ij}=g(x_i)+a_j+\epsilon_{ij}\), with curvature regularisation and the \(0.01S_{\rm ref}\) profile-gradient floor. Select smoothing per example with held-out-acquisition CV and the one-SE rule.
4. **Assign trusted external heads.** Interpolate \(g(x)\) only at the network inlet/outlet and external simple-unit or assembled-compound boundaries. Internal compound nodes are solver unknowns.
5. **Solve each base domain.** With boundary heads fixed, solve \[\min_H\sum_{e\in D}\left[\frac{(H_{u,e}-H_{d,e})/L_e-S_{\rm ref,D}}{S_{\rm ref,D}}\right]^2,\quad S_e\ge0.01S_{\rm ref,D}.\] Every directed path must preserve its prescribed boundary head loss.
6. **Treat unassigned connectors profile-directly.** Their heads are interpolated from \(g(x)\); historic factor screens are QA only, not corrections.
7. **Retain child-collapse boundary-conditioned solves.** Refresh them after base promotion; do not rebuild raster/network products.
8. **Validate and promote in layers.** Check positivity, `len` use, head/slope reconstruction, domain path loss, floor activity, and celerity sensitivity before reversible promotion.

## 0. Imports and production-function map

In [ ]:
import json
import sys
from collections import deque
from itertools import islice
from pathlib import Path

import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from scipy import sparse
from scipy.interpolate import PPoly, UnivariateSpline, make_smoothing_spline
from scipy.sparse.linalg import eigsh
from scipy.optimize import LinearConstraint, brentq, minimize


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'network_variants').is_dir() and (candidate / 'RAPID').is_dir():
            return candidate
    raise FileNotFoundError('Could not find the river-hierarchy repository root.')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RAPID_SRC = REPO_ROOT / 'RAPID' / 'src'
for import_root in (REPO_ROOT, RAPID_SRC):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from network_variants.hydraulic_context import (
    SlopeConfig,
    _fit_monotone_increasing_profile,
    _load_reference_section_support,
    _project_nodes_to_reference_section,
)
from network_variants.sword_matching import _maybe_import_sword_io, load_sword_node_reference
from rapid_tools.reference_profile import (
    ReferenceProfileConfig, fit_reference_profile, select_reference_profile,
    write_reference_profile_artifacts,
)
from rapid_tools.reference_profile_inputs import build_reference_profile_inputs
from network_variants.reference_profile_candidate import solve_boundary_conditioned_domain

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)


## 1. Select a frozen promoted base network

Start with Example 41 / Bubble 13 for the graph-potential demonstration. Then repeat
the same diagnostic for every example, including corrected Examples 13, 45, and 48.
The promoted canonical roots are the intended inputs; do not use historical archives.


In [ ]:
OUTPUT_ROOT = Path('/Volumes/PhD/river_hierarchy/network_variants/output')
EXAMPLE_ID = '11'
EXPERIMENT_NAME = f'{EXAMPLE_ID}_indep__anchored'
STATE_ID = 'S000_base'

# Diagnostic tolerances only; none is a production rule yet.
REFERENCE_NODE_MAX_OFFSET_M = 60.0
SWORD_CENTERLINE_MAX_OFFSET_M = 350.0
SWORD_NODE_SOURCE = Path('/Volumes/PhD/SWORD/v17c')
NUMERICAL_SLOPE_FLOOR = 1e-10
MAX_PATHS_TO_DISPLAY = 25

# A provisional screen only. It will be calibrated from the full diagnostic set,
# not copied into production code without review.
PROVISIONAL_CONNECTOR_FACTOR = 5.0

# Select either a simple base unit or an assembled compound bubble for an old/new comparison.
# Example 41 defaults: simple unit 1, or compound bubble 13.
DEMO_DOMAIN_KIND = 'compound_bubble'  # 'compound_bubble' or 'simple_unit'
DEMO_DOMAIN_ID = 13

experiment_dir = OUTPUT_ROOT / EXPERIMENT_NAME
state_dir = experiment_dir / 'states' / STATE_ID
assert experiment_dir.exists(), experiment_dir
assert state_dir.exists(), state_dir
print(f'Experiment: {experiment_dir}')
print(f'State:      {state_dir}')


## 2. Load frozen realised products, promoted contexts, and hierarchy boundaries

In [ ]:
def one_path(directory: Path, pattern: str) -> Path:
    matches = sorted(path for path in directory.glob(pattern) if not path.name.startswith('._'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected one {pattern!r} in {directory}; found {matches}')
    return matches[0]


variant_dir = state_dir / 'variant'
directed_dir = variant_dir / 'directed'
hydraulic_dir = state_dir / 'hydraulic_context'
links = gpd.read_file(one_path(directed_dir, '*directed_links.gpkg'))
nodes = gpd.read_file(one_path(directed_dir, '*directed_nodes.gpkg'))
link_context = pd.read_csv(hydraulic_dir / 'link_hydraulic_context.csv')
node_context = pd.read_csv(hydraulic_dir / 'node_hydraulic_context.csv')

reference_summary_path = experiment_dir / 'reference_section' / 'reference_section_summary.json'
assert reference_summary_path.exists(), reference_summary_path
support = _load_reference_section_support(str(reference_summary_path))
reference_line = support['line']

unit_summary_path = next((path for path in [variant_dir / 'hierarchy' / 'unit_summary.csv', state_dir / 'hierarchy' / 'unit_summary.csv'] if path.exists()), None)
if unit_summary_path is None:
    raise FileNotFoundError('Could not resolve unit_summary.csv.')
unit_summary = pd.read_csv(unit_summary_path)
bubble_metrics = pd.read_csv(state_dir / 'hierarchy' / 'bubble_equivalent_metrics.csv')

print(f'Links: {len(links):,}; nodes: {len(nodes):,}; centreline length: {reference_line.length:,.1f} m')
print('Hierarchy:', unit_summary_path)
display(unit_summary.head())


## 3. Build and fit the multi-pass monotone reference profile

All valid SWORD/SWOT observations near the frozen single centerline contribute to this
profile. The profile is a reach-scale reference, not an assertion that every short
connector has an independently resolved observed head loss.


In [ ]:
# Build the SWORD population from the two realised network endpoints, not from
# all historical nearest-node matches. This duplicates the reviewed corridor
# construction used in sword_corridor_investigation.ipynb.
matching_metadata = json.loads((variant_dir / 'summary' / 'variant_manifest.json').read_text())['sword_matching']
sword_region = str(matching_metadata['candidate_region']).strip().lower()
eligible_reach_ids = [int(value) for value in matching_metadata['candidate_reach_ids']]
reach_path = SWORD_NODE_SOURCE / f'sword_{sword_region.upper()}_v17c_reaches.parquet'
eligible_reaches = gpd.read_parquet(reach_path, columns=['reach_id', 'rch_id_up_main', 'rch_id_dn_main', 'geometry'])
eligible_reaches['reach_id'] = pd.to_numeric(eligible_reaches['reach_id'], errors='coerce').astype('Int64')
eligible_reaches = eligible_reaches.loc[eligible_reaches['reach_id'].isin(eligible_reach_ids)].drop_duplicates('reach_id').to_crs(support['centerline_crs']).copy()
eligible_reaches['reach_id'] = eligible_reaches['reach_id'].astype(int)

degree = pd.concat([links.groupby('id_us_node').size().rename('out_degree'), links.groupby('id_ds_node').size().rename('in_degree')], axis=1).fillna(0)
endpoint_nodes = nodes.to_crs(support['centerline_crs'])[[ 'id_node', nodes.geometry.name ]].join(degree, on='id_node').fillna({'in_degree': 0, 'out_degree': 0})
endpoint_nodes['anchor_role'] = np.select([endpoint_nodes['in_degree'].eq(0), endpoint_nodes['out_degree'].eq(0)], ['network source', 'network sink'], default='')
endpoint_nodes = endpoint_nodes.loc[endpoint_nodes['anchor_role'].ne('')].copy()
if len(endpoint_nodes) != 2:
    raise ValueError(f'Expected exactly two network endpoints, found {len(endpoint_nodes)}.')

def _reach_id(value):
    value = pd.to_numeric(value, errors='coerce')
    return int(value) if pd.notna(value) and int(value) > 0 else None

def _orient_upstream_at_start(reach_geometry, reach_nodes):
    valid = reach_nodes.loc[pd.to_numeric(reach_nodes['sword_dist_out'], errors='coerce').notna()]
    if valid.empty:
        return True
    upstream_node = valid.loc[pd.to_numeric(valid['sword_dist_out'], errors='coerce').idxmax()].geometry
    return reach_geometry.interpolate(0).distance(upstream_node) <= reach_geometry.interpolate(reach_geometry.length).distance(upstream_node)

# Project source and sink onto the closest reach inside the reviewed station corridor plus its local buffer.
projection_rows = []
for anchor in endpoint_nodes.itertuples(index=False):
    distances = eligible_reaches.geometry.distance(anchor.geometry)
    reach = eligible_reaches.loc[distances.idxmin()]
    projection_rows.append({'anchor_role': anchor.anchor_role, 'id_node': int(anchor.id_node), 'anchor_geometry': anchor.geometry, 'sword_reach_id': int(reach.reach_id), 'projection_m': float(reach.geometry.project(anchor.geometry)), 'projection_distance_m': float(distances.loc[reach.name])})
endpoint_projections = pd.DataFrame(projection_rows).set_index('anchor_role', drop=False)
reach_lookup = eligible_reaches.set_index('reach_id')

# Follow the directed main path between the projected endpoint reaches.
corridor_reach_ids = [int(endpoint_projections.loc['network source', 'sword_reach_id'])]
while corridor_reach_ids[-1] != int(endpoint_projections.loc['network sink', 'sword_reach_id']):
    next_reach = _reach_id(reach_lookup.loc[corridor_reach_ids[-1], 'rch_id_dn_main'])
    if next_reach is None or next_reach not in reach_lookup.index or next_reach in corridor_reach_ids:
        raise ValueError('Could not trace the directed main SWORD path between projected network endpoints.')
    corridor_reach_ids.append(next_reach)
core_reach_ids = list(corridor_reach_ids)

# Add one outward reach where a projection lies within 1 km of its endpoint reach boundary.
endpoint_sword_nodes = load_sword_node_reference(SWORD_NODE_SOURCE, sword_wse_field='wse_obs_p50', regions=[sword_region], reach_ids=[int(endpoint_projections.loc['network source', 'sword_reach_id']), int(endpoint_projections.loc['network sink', 'sword_reach_id'])]).to_crs(support['centerline_crs'])
for role, neighbour_field, prepend in [('network source', 'rch_id_up_main', True), ('network sink', 'rch_id_dn_main', False)]:
    projected = endpoint_projections.loc[role]
    reach = reach_lookup.loc[int(projected.sword_reach_id)]
    on_reach = endpoint_sword_nodes.loc[endpoint_sword_nodes['sword_reach_id'].eq(int(projected.sword_reach_id))]
    upstream_at_start = _orient_upstream_at_start(reach.geometry, on_reach)
    outward_end_m = 0.0 if ((role == 'network source') == upstream_at_start) else reach.geometry.length
    neighbour = _reach_id(reach[neighbour_field])
    if abs(float(projected.projection_m) - outward_end_m) < 1_000.0 and neighbour is not None and neighbour in reach_lookup.index:
        corridor_reach_ids = ([neighbour] + corridor_reach_ids) if prepend else (corridor_reach_ids + [neighbour])

corridor_reaches = eligible_reaches.set_index('reach_id').loc[corridor_reach_ids].reset_index()
corridor_reaches = gpd.GeoDataFrame(corridor_reaches, geometry=eligible_reaches.geometry.name, crs=eligible_reaches.crs)
actual_sword = load_sword_node_reference(SWORD_NODE_SOURCE, sword_wse_field='wse_obs_p50', regions=[sword_region], reach_ids=corridor_reach_ids).to_crs(support['centerline_crs']).drop_duplicates('sword_node_id').copy()

# Build directed corridor chainage, then remove nodes outside the two projected endpoints.
corridor_lookup = corridor_reaches.set_index('reach_id')
offsets, orientations, running = {}, {}, 0.0
for reach_id in corridor_reaches['reach_id'].astype(int):
    reach = corridor_lookup.loc[reach_id]
    on_reach = actual_sword.loc[actual_sword['sword_reach_id'].eq(reach_id)]
    offsets[reach_id] = running
    orientations[reach_id] = _orient_upstream_at_start(reach.geometry, on_reach)
    running += reach.geometry.length

def _chainage(reach_id, position_m):
    reach = corridor_lookup.loc[int(reach_id)]
    local = float(position_m) if orientations[int(reach_id)] else reach.geometry.length - float(position_m)
    return offsets[int(reach_id)] + local

actual_sword['_on_reach_m'] = [corridor_lookup.loc[int(row.sword_reach_id)].geometry.project(row.geometry) for row in actual_sword.itertuples(index=False)]
actual_sword['_path_chainage_m'] = [_chainage(reach_id, position_m) for reach_id, position_m in zip(actual_sword['sword_reach_id'], actual_sword['_on_reach_m'])]
source_chainage = _chainage(endpoint_projections.loc['network source', 'sword_reach_id'], endpoint_projections.loc['network source', 'projection_m'])
sink_chainage = _chainage(endpoint_projections.loc['network sink', 'sword_reach_id'], endpoint_projections.loc['network sink', 'projection_m'])
if source_chainage > sink_chainage:
    raise ValueError('Endpoint projection ordering is inconsistent with the traced SWORD path.')
actual_sword = actual_sword.loc[actual_sword['_path_chainage_m'].between(source_chainage, sink_chainage)].copy()
matched_reach_ids = corridor_reach_ids
matched_regions = [sword_region]
actual_sword['centerline_offset_m'] = actual_sword.geometry.distance(reference_line)
actual_sword['centerline_station_m'] = actual_sword.geometry.map(reference_line.project)
actual_sword['reference_section_dist_out_m'] = np.where(
    support['outlet_at_start'], actual_sword['centerline_station_m'],
    reference_line.length - actual_sword['centerline_station_m'],
)
# The production loader deliberately retains only matching fields. Load the SWORD
# observation-distribution columns separately for this diagnostic, then join by node.
# `p10`/`p90` describe the observed multi-pass distribution; they are not alternative
# node-by-node elevations to substitute into one longitudinal profile.
load_sword_nodes, scan_sword_parquet_dir = _maybe_import_sword_io()
sword_catalog = scan_sword_parquet_dir(SWORD_NODE_SOURCE)
quality_field_names = ['wse_obs_p10', 'wse_obs_p50', 'wse_obs_p90', 'wse_obs_range', 'wse_obs_mad', 'n_obs']
raw_sword_quality = load_sword_nodes(
    sword_catalog, regions=matched_regions or None, reach_ids=matched_reach_ids or None,
    columns=['node_id', *quality_field_names],
)
raw_sword_quality = pd.DataFrame(raw_sword_quality.drop(columns='geometry', errors='ignore')).rename(columns={'node_id': 'sword_node_id'})
raw_sword_quality['sword_node_id'] = pd.to_numeric(raw_sword_quality['sword_node_id'], errors='coerce').astype('Int64')
# Some source extracts repeat a node at a file boundary. Collapse only identical node IDs
# before joining; repeated realised-network matches are retained until the existing screen.
raw_sword_quality = raw_sword_quality.groupby('sword_node_id', as_index=False)[quality_field_names].mean()
actual_sword = actual_sword.merge(raw_sword_quality, on='sword_node_id', how='left', validate='m:1')

profile_candidates = actual_sword.loc[
    actual_sword['sword_wse_fill_method'].astype(str).eq('requested_field')
    & pd.to_numeric(actual_sword['sword_wse'], errors='coerce').notna()
    & actual_sword['centerline_offset_m'].le(SWORD_CENTERLINE_MAX_OFFSET_M)
].drop_duplicates(subset='sword_node_id').copy()
profile_aggregation = {
    'raw_wse_m': ('sword_wse', 'mean'),
    'centerline_offset_m': ('centerline_offset_m', 'min'),
}
for field_name in ['wse_obs_p10', 'wse_obs_p50', 'wse_obs_p90', 'wse_obs_range', 'wse_obs_mad']:
    if field_name in profile_candidates.columns:
        profile_aggregation[field_name] = (field_name, 'mean')
if 'n_obs' in profile_candidates.columns:
    profile_aggregation['n_obs'] = ('n_obs', 'sum')
profile_table = profile_candidates.groupby('reference_section_dist_out_m', as_index=False).agg(**profile_aggregation).sort_values('reference_section_dist_out_m')
if len(profile_table) < 2:
    raise RuntimeError('Fewer than two valid SWORD/SWOT profile anchors survive screening.')
profile_table['monotone_wse_m'] = _fit_monotone_increasing_profile(
    profile_table['reference_section_dist_out_m'].to_numpy(float),
    profile_table['raw_wse_m'].to_numpy(float),
)
xp = profile_table['reference_section_dist_out_m'].to_numpy(float)
yp = profile_table['monotone_wse_m'].to_numpy(float)
network_reference_slope = float((yp[-1] - yp[0]) / (xp[-1] - xp[0]))
print(f'Profile anchors={len(profile_table)}; global reference slope={network_reference_slope:.8g}')

fig = go.Figure()
profile_x = profile_table['reference_section_dist_out_m']
has_observation_envelope = {'wse_obs_p10', 'wse_obs_p90'}.issubset(profile_table.columns)
if has_observation_envelope:
    envelope = profile_table[['wse_obs_p10', 'wse_obs_p90']].apply(pd.to_numeric, errors='coerce')
    fig.add_scatter(x=profile_x, y=envelope['wse_obs_p10'], mode='lines', line={'width': 0}, showlegend=False, hoverinfo='skip')
    fig.add_scatter(
        x=profile_x, y=envelope['wse_obs_p90'], mode='lines', line={'width': 0},
        fill='tonexty', fillcolor='rgba(120, 120, 120, 0.20)', name='SWORD observed P10–P90 envelope',
    )
fig.add_scatter(x=profile_x, y=profile_table['raw_wse_m'], mode='markers', marker={'size': 5}, name='raw SWORD/SWOT P50')
fig.add_scatter(x=profile_x, y=profile_table['monotone_wse_m'], mode='lines', line={'width': 2.5}, name='current equal-weight PAVA')
fig.update_layout(
    title='Frozen single-centerline WSE profile and observed SWORD spread',
    xaxis_title='Distance to outlet [m]', yaxis_title='WSE [m]',
)
fig.show()

quality_fig = go.Figure()
if 'wse_obs_range' in profile_table.columns:
    quality_fig.add_scatter(x=profile_x, y=profile_table['wse_obs_range'], mode='lines+markers', name='P10–P90 range')
if 'wse_obs_mad' in profile_table.columns:
    quality_fig.add_scatter(x=profile_x, y=profile_table['wse_obs_mad'], mode='lines+markers', name='WSE MAD')
if 'n_obs' in profile_table.columns:
    quality_fig.add_bar(x=profile_x, y=profile_table['n_obs'], name='number of observations', yaxis='y2', opacity=0.30)
quality_fig.update_layout(
    title='SWORD observation diagnostics at retained profile anchors',
    xaxis_title='Distance to outlet [m]', yaxis={'title': 'WSE spread [m]'},
    yaxis2={'title': 'number of observations', 'overlaying': 'y', 'side': 'right', 'showgrid': False},
)
quality_fig.show()

quality_review_columns = [
    column for column in ['reference_section_dist_out_m', 'raw_wse_m', 'monotone_wse_m', 'centerline_offset_m',
                           'wse_obs_p10', 'wse_obs_p50', 'wse_obs_p90', 'wse_obs_range', 'wse_obs_mad', 'n_obs']
    if column in profile_table.columns
]
profile_quality_review = profile_table[quality_review_columns].copy()
if 'wse_obs_mad' in profile_quality_review.columns:
    print('Largest observed WSE-MAD anchors (review only; not a rejection rule):')
    display(profile_quality_review.sort_values('wse_obs_mad', ascending=False).head(20))


### 3.1 Filtered multi-pass RiverSP observations and quality evidence

This is a diagnostic input reconstruction only. It reads the locally retained audit cache
of actual RiverSP node observations, reapplies the established filter, and calculates a
node-level median, spread, and pass count. It does not alter any production SWORD or
hydraulic-context product.


In [ ]:
RIVERSP_AUDIT_OBSERVATIONS = REPO_ROOT / 'data_validation' / 'swot_sword_17c' / 'target_observations.parquet'
PASS_SUPPORT_TARGET = 5.0

if not RIVERSP_AUDIT_OBSERVATIONS.exists():
    raise FileNotFoundError(f'Missing direct RiverSP audit cache: {RIVERSP_AUDIT_OBSERVATIONS}')

riversp_observations = pd.read_parquet(RIVERSP_AUDIT_OBSERVATIONS)
for column in ['node_id', 'reach_id']:
    riversp_observations[column] = pd.to_numeric(riversp_observations[column], errors='coerce').astype('Int64')
for column in ['wse_sm', 'wse_sm_u', 'dark_frac', 'xtrk_dist']:
    riversp_observations[column] = pd.to_numeric(riversp_observations[column], errors='coerce')

# This is deliberately the same filter used to create/validate the local RiverSP audit cache.
passes_existing_riversp_filter = (
    riversp_observations['wse_sm_u'].le(1.0)
    & riversp_observations['dark_frac'].le(0.5)
    & riversp_observations['xtrk_dist'].abs().between(10_000, 60_000)
    & riversp_observations['wse_sm'].gt(0.0)
    & np.isfinite(riversp_observations[['wse_sm', 'wse_sm_u', 'dark_frac', 'xtrk_dist']]).all(axis=1)
)

direct_pass_summary = (
    riversp_observations.loc[passes_existing_riversp_filter]
    .groupby('node_id', as_index=False)
    .agg(
        direct_pass_p50_m=('wse_sm', 'median'),
        direct_pass_p10_m=('wse_sm', lambda values: values.quantile(0.10)),
        direct_pass_p90_m=('wse_sm', lambda values: values.quantile(0.90)),
        direct_pass_mad_m=('wse_sm', lambda values: np.median(np.abs(values - np.median(values)))),
        direct_pass_n_obs=('wse_sm', 'size'),
        direct_pass_wse_u_p50_m=('wse_sm_u', 'median'),
    )
)

direct_pass_profile = profile_candidates.merge(
    direct_pass_summary, left_on='sword_node_id', right_on='node_id', how='left', validate='1:1',
).sort_values('reference_section_dist_out_m').copy()
direct_pass_profile['direct_pass_count_weight'] = np.minimum(
    1.0, direct_pass_profile['direct_pass_n_obs'].fillna(0.0) / PASS_SUPPORT_TARGET,
)
direct_pass_profile['direct_minus_sword_p50_m'] = (
    direct_pass_profile['direct_pass_p50_m'] - direct_pass_profile['sword_wse']
)

direct_coverage = direct_pass_profile['direct_pass_p50_m'].notna()
print(
    f'Direct filtered RiverSP coverage: {int(direct_coverage.sum())}/{len(direct_pass_profile)} profile nodes; '
    f'pass-support target: {PASS_SUPPORT_TARGET:g}.'
)
display(
    direct_pass_profile.loc[direct_coverage, [
        'sword_node_id', 'sword_reach_id', 'reference_section_dist_out_m', 'sword_wse',
        'direct_pass_p50_m', 'direct_pass_p10_m', 'direct_pass_p90_m', 'direct_pass_mad_m',
        'direct_pass_n_obs', 'direct_pass_count_weight', 'direct_minus_sword_p50_m',
    ]].sort_values('direct_minus_sword_p50_m', key=lambda values: values.abs(), ascending=False).head(30)
)

direct_profile_fig = go.Figure()
direct_profile_fig.add_scatter(
    x=direct_pass_profile['reference_section_dist_out_m'], y=direct_pass_profile['sword_wse'],
    mode='markers', marker={'size': 4, 'color': 'rgba(45, 45, 45, 0.40)'}, name='current SWORD P50',
)
direct_profile_fig.add_scatter(
    x=direct_pass_profile.loc[direct_coverage, 'reference_section_dist_out_m'],
    y=direct_pass_profile.loc[direct_coverage, 'direct_pass_p10_m'],
    mode='lines', line={'width': 0}, showlegend=False, hoverinfo='skip',
)
direct_profile_fig.add_scatter(
    x=direct_pass_profile.loc[direct_coverage, 'reference_section_dist_out_m'],
    y=direct_pass_profile.loc[direct_coverage, 'direct_pass_p90_m'],
    mode='lines', line={'width': 0}, fill='tonexty',
    fillcolor='rgba(76, 120, 168, 0.18)', name='direct filtered RiverSP P10–P90',
)
direct_profile_fig.add_scatter(
    x=direct_pass_profile.loc[direct_coverage, 'reference_section_dist_out_m'],
    y=direct_pass_profile.loc[direct_coverage, 'direct_pass_p50_m'],
    mode='markers', marker={'size': 5, 'color': '#4C78A8'}, name='direct filtered RiverSP P50',
    customdata=direct_pass_profile.loc[direct_coverage, ['sword_node_id', 'direct_pass_n_obs', 'direct_pass_count_weight', 'direct_pass_mad_m']],
    hovertemplate='distance=%{x:.0f} m<br>direct P50=%{y:.3f} m<br>node=%{customdata[0]}<br>n=%{customdata[1]}<br>count weight=%{customdata[2]:.2f}<br>MAD=%{customdata[3]:.3f} m<extra></extra>',
)
direct_profile_fig.update_layout(
    title='Current SWORD P50 versus directly reconstructed filtered RiverSP profile',
    xaxis_title='Distance to outlet [m]', yaxis_title='WSE [m]', height=690,
)
direct_profile_fig.show()


### 3.2 SWORD reach corridor and profile-node audit

Before interpreting an irregular profile as a real hydraulic feature, verify that the retained SWORD reaches form the intended longitudinal corridor. This audit checks every consecutive selected reach against SWORD's `rch_id_up_main` / `rch_id_dn_main` topology, distinguishes omitted mainstem bridge reaches from a selected secondary branch, displays nearby spatial context, and compares each selected branch with SWORD's designated mainstem.

The local SWORD v17c node files contain multi-pass summaries (`wse_obs_p10` … `wse_obs_p90`, MAD, and `n_obs`). Individual filtered RiverSP node observations are also retained locally in `data_validation/swot_sword_17c/target_observations.parquet`; the acquisition-level audit below uses those records rather than treating the aggregate SWORD summary as a single observation.


In [ ]:
# Read the current v17c reach tables directly, rather than the compatibility loader:
# their mainstem topology fields are needed to audit the selected profile corridor.
SWORD_CORRIDOR_EXTENSION_STEPS = 3
SWORD_OBSTRUCTION_LABELS = {0: 'none', 1: 'dam', 2: 'lock', 3: 'low-head', 4: 'falls'}
reach_static_columns = [
    'reach_id', 'rch_id_up_main', 'rch_id_dn_main', 'n_rch_up', 'n_rch_down',
    'dist_out', 'reach_length', 'river_name', 'obstr_type', 'grod_id', 'hfalls_id',
    'lakeflag', 'wse_obs_p50', 'wse_obs_mad', 'swot_obs',
]

selected_profile_reach_ids = set(pd.to_numeric(profile_candidates['sword_reach_id'], errors='coerce').dropna().astype(int))
if not selected_profile_reach_ids:
    raise RuntimeError('No retained SWORD reach IDs are available for the corridor audit.')

all_static_reaches = []
for region in sorted({str(value).strip().upper() for value in matched_regions if str(value).strip()}):
    current_reach_path = SWORD_NODE_SOURCE / f'sword_{region}_v17c_reaches.parquet'
    if not current_reach_path.exists():
        raise FileNotFoundError(f'Current SWORD v17c reach table not found: {current_reach_path}')
    frame = pd.read_parquet(current_reach_path, columns=reach_static_columns)
    frame['sword_region'] = region.lower()
    all_static_reaches.append(frame)
all_static_reaches = pd.concat(all_static_reaches, ignore_index=True).drop_duplicates('reach_id')
all_static_reaches['reach_id'] = pd.to_numeric(all_static_reaches['reach_id'], errors='coerce').astype('Int64')
all_static_reaches = all_static_reaches.dropna(subset=['reach_id']).copy()
all_static_reaches['reach_id'] = all_static_reaches['reach_id'].astype(int)
reach_lookup = all_static_reaches.set_index('reach_id')

# One row per retained profile reach, in the downstream-to-upstream frozen-reference order.
profile_reach_summary = profile_candidates.groupby('sword_reach_id', as_index=False).agg(
    first_reference_station_m=('reference_section_dist_out_m', 'min'),
    last_reference_station_m=('reference_section_dist_out_m', 'max'),
    n_retained_nodes=('sword_node_id', 'nunique'),
    p50_min_m=('sword_wse', 'min'), p50_max_m=('sword_wse', 'max'),
    median_p50_m=('sword_wse', 'median'), median_MAD_m=('wse_obs_mad', 'median'),
    total_n_obs=('n_obs', 'sum'),
).sort_values('first_reference_station_m').reset_index(drop=True)
profile_reach_summary['sword_reach_id'] = pd.to_numeric(profile_reach_summary['sword_reach_id'], errors='raise').astype(int)
profile_reach_summary = profile_reach_summary.merge(
    all_static_reaches, left_on='sword_reach_id', right_on='reach_id', how='left', validate='1:1',
).drop(columns='reach_id')
profile_reach_summary['obstruction'] = pd.to_numeric(profile_reach_summary['obstr_type'], errors='coerce').map(SWORD_OBSTRUCTION_LABELS).fillna('unclassified')

SWORD_MAINSTEM_GAP_MAX_STEPS = 6
def _unselected_mainstem_bridge(downstream_id, upstream_id):
    """Return intervening mainstem reaches, or None when no short mainstem route exists."""
    bridge = []
    current = downstream_id
    for _ in range(SWORD_MAINSTEM_GAP_MAX_STEPS):
        if current not in reach_lookup.index:
            return None
        next_id = pd.to_numeric(pd.Series([reach_lookup.loc[current, 'rch_id_up_main']]), errors='coerce').iloc[0]
        if pd.isna(next_id):
            return None
        next_id = int(next_id)
        if next_id == upstream_id:
            return bridge
        bridge.append(next_id)
        current = next_id
    return None

continuity = ['downstream end of selected profile']
mainstem_bridge_ids = ['']
for index in range(1, len(profile_reach_summary)):
    downstream = profile_reach_summary.iloc[index - 1]
    upstream = profile_reach_summary.iloc[index]
    downstream_id, upstream_id = int(downstream['sword_reach_id']), int(upstream['sword_reach_id'])
    expected_main = pd.to_numeric(pd.Series([downstream['rch_id_up_main']]), errors='coerce').iloc[0]
    upstream_down = pd.to_numeric(pd.Series([upstream['rch_id_dn_main']]), errors='coerce').iloc[0]
    if pd.notna(expected_main) and int(expected_main) == upstream_id:
        continuity.append('mainstem continuation')
        mainstem_bridge_ids.append('')
    elif (bridge := _unselected_mainstem_bridge(downstream_id, upstream_id)) is not None:
        continuity.append(f'mainstem continuation with {len(bridge)} unselected reach(es)')
        mainstem_bridge_ids.append(', '.join(map(str, bridge)))
    elif pd.notna(upstream_down) and int(upstream_down) == downstream_id:
        continuity.append('selected secondary upstream branch')
        mainstem_bridge_ids.append('')
    else:
        continuity.append('not connected by SWORD main-route fields')
        mainstem_bridge_ids.append('')
profile_reach_summary['continuity_from_downstream_reach'] = continuity
profile_reach_summary['unselected_mainstem_bridge_reach_ids'] = mainstem_bridge_ids

# Inspect WSE summaries on skipped, but topologically required, mainstem reaches. A large
# centreline offset means that their omission is geometric; usable nearby nodes instead show
# that our SWORD matching list is incomplete rather than that the SWORD corridor is broken.
bridge_reach_ids = {
    int(reach_id) for bridge in mainstem_bridge_ids if bridge
    for reach_id in bridge.split(', ') if reach_id
}
if bridge_reach_ids:
    bridge_sword = load_sword_node_reference(
        SWORD_NODE_SOURCE, sword_wse_field='wse_obs_p50',
        regions=matched_regions or None, reach_ids=sorted(bridge_reach_ids),
    ).to_crs(support['centerline_crs'])
    bridge_sword['centerline_offset_m'] = bridge_sword.geometry.distance(reference_line)
    bridge_sword['centerline_station_m'] = bridge_sword.geometry.map(reference_line.project)
    bridge_sword['reference_section_dist_out_m'] = np.where(
        support['outlet_at_start'], bridge_sword['centerline_station_m'],
        reference_line.length - bridge_sword['centerline_station_m'],
    )
    bridge_sword = bridge_sword.loc[
        bridge_sword['sword_wse_fill_method'].astype(str).eq('requested_field')
        & pd.to_numeric(bridge_sword['sword_wse'], errors='coerce').notna()
    ].copy()
    bridge_raw_quality = load_sword_nodes(
        sword_catalog, regions=matched_regions or None, reach_ids=sorted(bridge_reach_ids),
        columns=['node_id', *quality_field_names],
    )
    bridge_raw_quality = pd.DataFrame(bridge_raw_quality.drop(columns='geometry', errors='ignore')).rename(columns={'node_id': 'sword_node_id'})
    bridge_raw_quality['sword_node_id'] = pd.to_numeric(bridge_raw_quality['sword_node_id'], errors='coerce').astype('Int64')
    bridge_raw_quality = bridge_raw_quality.groupby('sword_node_id', as_index=False)[quality_field_names].mean()
    bridge_sword = bridge_sword.merge(bridge_raw_quality, on='sword_node_id', how='left', validate='m:1')
    bridge_wse_summary = bridge_sword.groupby('sword_reach_id', as_index=False).agg(
        n_valid_nodes=('sword_node_id', 'nunique'),
        p50_min_m=('sword_wse', 'min'), p50_max_m=('sword_wse', 'max'),
        median_MAD_m=('wse_obs_mad', 'median'), total_n_obs=('n_obs', 'sum'),
        min_centerline_offset_m=('centerline_offset_m', 'min'),
        median_centerline_offset_m=('centerline_offset_m', 'median'),
        n_nodes_within_profile_offset=('centerline_offset_m', lambda values: int(pd.Series(values).le(SWORD_CENTERLINE_MAX_OFFSET_M).sum())),
    )
    print('Unselected mainstem bridge-reach WSE audit:')
    display(bridge_wse_summary)
else:
    bridge_wse_summary = pd.DataFrame()

# At a bifurcation, show the selected upstream branch beside SWORD's designated upstream
# mainstem. This distinguishes a genuine route-selection choice from a noisy node series.
branch_rows = []
for index, row in profile_reach_summary.iterrows():
    if row['continuity_from_downstream_reach'] != 'selected secondary upstream branch' or index == 0:
        continue
    downstream = profile_reach_summary.iloc[index - 1]
    expected_main = pd.to_numeric(pd.Series([downstream['rch_id_up_main']]), errors='coerce').iloc[0]
    if pd.notna(expected_main):
        branch_rows.append({
            'downstream_junction_reach_id': int(downstream['sword_reach_id']),
            'selected_branch_reach_id': int(row['sword_reach_id']),
            'sword_designated_mainstem_reach_id': int(expected_main),
        })
branch_choice_audit = pd.DataFrame(branch_rows)
if not branch_choice_audit.empty:
    branch_candidate_ids = sorted(set(branch_choice_audit['selected_branch_reach_id']) | set(branch_choice_audit['sword_designated_mainstem_reach_id']))
    branch_sword = load_sword_node_reference(
        SWORD_NODE_SOURCE, sword_wse_field='wse_obs_p50',
        regions=matched_regions or None, reach_ids=branch_candidate_ids,
    ).to_crs(support['centerline_crs'])
    branch_sword['centerline_offset_m'] = branch_sword.geometry.distance(reference_line)
    branch_sword = branch_sword.loc[
        branch_sword['sword_wse_fill_method'].astype(str).eq('requested_field')
        & pd.to_numeric(branch_sword['sword_wse'], errors='coerce').notna()
    ].copy()
    branch_raw_quality = load_sword_nodes(
        sword_catalog, regions=matched_regions or None, reach_ids=branch_candidate_ids,
        columns=['node_id', *quality_field_names],
    )
    branch_raw_quality = pd.DataFrame(branch_raw_quality.drop(columns='geometry', errors='ignore')).rename(columns={'node_id': 'sword_node_id'})
    branch_raw_quality['sword_node_id'] = pd.to_numeric(branch_raw_quality['sword_node_id'], errors='coerce').astype('Int64')
    branch_raw_quality = branch_raw_quality.groupby('sword_node_id', as_index=False)[quality_field_names].mean()
    branch_sword = branch_sword.merge(branch_raw_quality, on='sword_node_id', how='left', validate='m:1')
    branch_wse_summary = branch_sword.groupby('sword_reach_id', as_index=False).agg(
        n_valid_nodes=('sword_node_id', 'nunique'),
        p50_min_m=('sword_wse', 'min'), p50_max_m=('sword_wse', 'max'),
        p10_min_m=('wse_obs_p10', 'min'), p90_max_m=('wse_obs_p90', 'max'),
        median_MAD_m=('wse_obs_mad', 'median'), total_n_obs=('n_obs', 'sum'),
        min_centerline_offset_m=('centerline_offset_m', 'min'),
        median_centerline_offset_m=('centerline_offset_m', 'median'),
        n_nodes_within_profile_offset=('centerline_offset_m', lambda values: int(pd.Series(values).le(SWORD_CENTERLINE_MAX_OFFSET_M).sum())),
    )
    selected_branch_summary = branch_wse_summary.add_prefix('selected_branch_').rename(columns={'selected_branch_sword_reach_id': 'selected_branch_reach_id'})
    designated_mainstem_summary = branch_wse_summary.add_prefix('designated_mainstem_').rename(columns={'designated_mainstem_sword_reach_id': 'sword_designated_mainstem_reach_id'})
    branch_choice_audit = branch_choice_audit.merge(selected_branch_summary, on='selected_branch_reach_id', how='left').merge(designated_mainstem_summary, on='sword_designated_mainstem_reach_id', how='left')
    print('Selected-branch versus SWORD-designated-mainstem WSE audit:')
    display(branch_choice_audit)

# Context contains selected reaches, their mainstem upstream/downstream continuations, and
# every direct upstream branch at a selected junction. It makes a branch choice visible.
context_reach_ids = set(selected_profile_reach_ids)
frontier = set(selected_profile_reach_ids)
for _ in range(SWORD_CORRIDOR_EXTENSION_STEPS):
    next_frontier = set()
    for reach_id in frontier:
        if reach_id not in reach_lookup.index:
            continue
        row = reach_lookup.loc[reach_id]
        for field in ('rch_id_up_main', 'rch_id_dn_main'):
            neighbour = pd.to_numeric(pd.Series([row[field]]), errors='coerce').iloc[0]
            if pd.notna(neighbour):
                next_frontier.add(int(neighbour))
        next_frontier.update(all_static_reaches.loc[pd.to_numeric(all_static_reaches['rch_id_dn_main'], errors='coerce').eq(reach_id), 'reach_id'].astype(int))
    next_frontier -= context_reach_ids
    context_reach_ids.update(next_frontier)
    frontier = next_frontier

context_static = all_static_reaches.loc[all_static_reaches['reach_id'].isin(context_reach_ids)].copy()
context_static['is_selected_profile_reach'] = context_static['reach_id'].isin(selected_profile_reach_ids)

# Spatial context makes an incorrect branch or a missing mainstem connection inspectable,
# rather than relying only on reach-ID fields in the table below.
context_geometry_frames = []
for region in sorted({str(value).strip().upper() for value in matched_regions if str(value).strip()}):
    current_reach_path = SWORD_NODE_SOURCE / f'sword_{region}_v17c_reaches.parquet'
    geometry_frame = gpd.read_parquet(current_reach_path, columns=['reach_id', 'geometry'])
    geometry_frame['reach_id'] = pd.to_numeric(geometry_frame['reach_id'], errors='coerce')
    geometry_frame = geometry_frame.dropna(subset=['reach_id']).copy()
    geometry_frame['reach_id'] = geometry_frame['reach_id'].astype(int)
    context_geometry_frames.append(geometry_frame.loc[geometry_frame['reach_id'].isin(context_reach_ids)])
if context_geometry_frames:
    context_geometries = gpd.GeoDataFrame(
        pd.concat(context_geometry_frames, ignore_index=True), geometry='geometry',
        crs=context_geometry_frames[0].crs,
    ).drop_duplicates('reach_id').to_crs(support['centerline_crs'])
else:
    context_geometries = gpd.GeoDataFrame({'reach_id': []}, geometry=[], crs=support['centerline_crs'])

continuity_by_reach = profile_reach_summary.set_index('sword_reach_id')['continuity_from_downstream_reach'].to_dict()
def _line_parts(geometry):
    if geometry is None or geometry.is_empty:
        return []
    if geometry.geom_type == 'LineString':
        return [geometry]
    if geometry.geom_type == 'MultiLineString':
        return list(geometry.geoms)
    return []

corridor_map_fig = go.Figure()
reference_x, reference_y = reference_line.xy
corridor_map_fig.add_scatter(x=list(reference_x), y=list(reference_y), mode='lines', name='frozen single centreline', line={'color': '#1f77b4', 'dash': 'dash', 'width': 3})
for row in context_geometries.itertuples(index=False):
    reach_id = int(row.reach_id)
    selected = reach_id in selected_profile_reach_ids
    continuity_label = continuity_by_reach.get(reach_id, 'nearby SWORD context')
    if not selected:
        color, width = '#b8b8b8', 1.5
    elif continuity_label == 'mainstem continuation':
        color, width = '#2ca02c', 3.5
    elif continuity_label.startswith('mainstem continuation with'):
        color, width = '#17becf', 3.5
    elif continuity_label == 'selected secondary upstream branch':
        color, width = '#ff7f0e', 4
    elif continuity_label == 'not connected by SWORD main-route fields':
        color, width = '#d62728', 4
    else:
        color, width = '#2ca02c', 3.5
    for part in _line_parts(row.geometry):
        x, y = part.xy
        corridor_map_fig.add_scatter(
            x=list(x), y=list(y), mode='lines', showlegend=False,
            line={'color': color, 'width': width},
            hovertemplate=f'reach={reach_id}<br>{continuity_label}<extra></extra>',
        )
corridor_map_fig.update_layout(
    title='SWORD reach context: green = direct mainstem, cyan = mainstem with unselected bridge reaches, orange = branch, red = disconnected transition',
    xaxis_title='Projected easting [m]', yaxis_title='Projected northing [m]',
    height=720, yaxis={'scaleanchor': 'x', 'scaleratio': 1},
)
corridor_map_fig.show()

audit_columns = [
    'sword_reach_id', 'continuity_from_downstream_reach', 'first_reference_station_m',
    'last_reference_station_m', 'n_retained_nodes', 'p50_min_m', 'p50_max_m',
    'median_MAD_m', 'total_n_obs', 'river_name', 'obstruction', 'hfalls_id',
    'rch_id_up_main', 'rch_id_dn_main', 'unselected_mainstem_bridge_reach_ids', 'n_rch_up', 'n_rch_down',
]
display(profile_reach_summary[[column for column in audit_columns if column in profile_reach_summary.columns]])
print(profile_reach_summary['continuity_from_downstream_reach'].value_counts().to_string())

obstruction_rows = profile_reach_summary.loc[
    ~profile_reach_summary['obstruction'].eq('none')
    | pd.to_numeric(profile_reach_summary['hfalls_id'], errors='coerce').fillna(0).ne(0)
].copy()
if obstruction_rows.empty:
    print('No SWORD obstruction flag is present on the retained profile reaches.')
else:
    print('SWORD-flagged obstruction reaches in the retained profile:')
    display(obstruction_rows[[column for column in audit_columns if column in obstruction_rows.columns]])

corridor_profile_fig = go.Figure()
if {'wse_obs_p10', 'wse_obs_p90'}.issubset(profile_candidates.columns):
    corridor_profile_fig.add_scatter(x=profile_candidates['reference_section_dist_out_m'], y=profile_candidates['wse_obs_p10'], mode='lines', line={'width': 0}, showlegend=False, hoverinfo='skip')
    corridor_profile_fig.add_scatter(x=profile_candidates['reference_section_dist_out_m'], y=profile_candidates['wse_obs_p90'], mode='lines', line={'width': 0}, fill='tonexty', fillcolor='rgba(120,120,120,0.18)', name='SWORD P10–P90 envelope')
for reach_id, group in profile_candidates.groupby('sword_reach_id', sort=False):
    group = group.sort_values('reference_section_dist_out_m')
    corridor_profile_fig.add_scatter(
        x=group['reference_section_dist_out_m'], y=group['sword_wse'], mode='lines+markers',
        marker={'size': 4}, line={'width': 1.5}, name=f'reach {int(reach_id)}',
        customdata=np.column_stack([group['wse_obs_mad'], group['n_obs'], group['centerline_offset_m']]),
        hovertemplate='station=%{x:.0f} m<br>P50=%{y:.3f} m<br>MAD=%{customdata[0]:.3f} m<br>n_obs=%{customdata[1]:.0f}<br>offset=%{customdata[2]:.1f} m<extra></extra>',
    )
corridor_profile_fig.update_layout(title='Raw SWORD multi-pass WSE summaries by retained reach', xaxis_title='Distance to outlet on frozen reference section [m]', yaxis_title='WSE [m]', height=720, legend={'orientation': 'h', 'y': -0.30})
corridor_profile_fig.show()


### 3.3 Historical profile candidates and why the joint model was selected

All candidates use the same retained SWORD nodes, the same frozen centreline coordinate, and
the same downstream-to-upstream convention. They differ only in how they infer a longitudinal
latent WSE profile from multi-pass node observations. `wse_obs_p10`, `wse_obs_p90`,
`wse_obs_range`, `wse_obs_mad`, and `n_obs` are shown above as evidence about observation
spread; they are not interchangeable instantaneous WSE values. In particular, choosing P10
or P90 independently at each node would combine different hydrologic states into an artificial
profile.

Let `y_i` be P50 at centreline coordinate `x_i`, `H_i` the latent WSE, `s_i` a spread
scale derived from MAD, and `w_i` a clipped `sqrt(n_obs_i) / s_i` reliability weight.
All formulations have the monotonic downstream-to-upstream constraint `H[i+1] >= H[i]`.

1. **Current equal-weight PAVA (control).**
   `min_H sum_i (H_i - y_i)^2`, subject to `H[i+1] >= H[i]`.
   Every retained P50 has equal influence. The result is a monotone, piecewise-constant
   profile; it permits plateaux and does not use observation spread, count, or smoothness.
   This is exactly the current production profile estimator and remains the baseline.
2. **Weighted robust isotonic fit.**
   `min_H sum_i w_i rho((H_i-y_i)/s_i)`, subject to `H[i+1] >= H[i]`, where `rho` is
   the Huber loss. The notebook solves this by iteratively reweighted weighted-PAVA.
   High-MAD nodes and large residuals exert less leverage, but the fit remains monotone
   and piecewise constant.
3. **Monotone smoothing, no gradient floor.**
   The implemented smoothing-spline form is `min_f integral(f''(x)^2 dx)`, subject to
   `sum_i w_i^2 (f(x_i)-y_i)^2 <= s`; `s` is the selected weighted residual budget.
   This is equivalent to the more familiar penalised form with `lambda` for a corresponding
   choice of multiplier. The notebook then projects `f(x_i)` onto the same weighted monotonic
   cone with PAVA. It is a transparent two-stage approximation to a joint constrained smoothing
   fit and may still contain flat runs.
4. **Monotone smoothing with a small positive gradient floor.**
   Use the same smooth `f`, but project `f(x_i)-S_floor*x_i` with PAVA and add
   `S_floor*x_i` back. Equivalently it enforces `H[i+1]-H[i] >= S_floor*(x[i+1]-x[i])`.
   The floor is a sensitivity setting to prevent exact artificial plateaux, not an asserted
   physical minimum river slope; it is shown as 1% of the network reference slope here.

The next notebook step should fit all four candidates for the same profile, compare residuals
against P50 and the P10–P90 envelope, and test whether each candidate preserves the fixed
network-scale head loss. No candidate is promoted into hydraulic-context production code until
those comparisons have been reviewed.


In [ ]:
# These settings define a diagnostic sensitivity comparison, not a production calibration.
PROFILE_MAD_FLOOR_M = 0.15
PROFILE_WEIGHT_CAP_RATIO = 30.0
HUBER_TUNING_CONSTANT = 1.345
ROBUST_ISOTONIC_ITERATIONS = 20
# 0.025 is a middle active diagnostic value; the sensitivity cell shows why s/N=1 is over-smoothed here.
SPLINE_SMOOTHING_PER_ANCHOR = 0.025
SLOPE_FLOOR_FRACTION_OF_NETWORK_REFERENCE = 0.01


def weighted_pava_increasing(values, weights):
    """Weighted PAVA: argmin sum(w_i * (H_i-y_i)^2), subject to H increasing."""
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    blocks = []
    for index, (value, weight) in enumerate(zip(values, weights)):
        blocks.append([index, index, float(weight), float(weight * value)])
        while len(blocks) >= 2:
            left, right = blocks[-2], blocks[-1]
            if left[3] / left[2] <= right[3] / right[2]:
                break
            blocks[-2:] = [[left[0], right[1], left[2] + right[2], left[3] + right[3]]]
    fitted = np.empty_like(values)
    for start, end, weight, weighted_sum in blocks:
        fitted[start:end + 1] = weighted_sum / weight
    return fitted


def clipped_profile_weights(table, mad_floor_m=PROFILE_MAD_FLOOR_M):
    mad = pd.to_numeric(table.get('wse_obs_mad'), errors='coerce').to_numpy(float)
    n_obs = pd.to_numeric(table.get('n_obs'), errors='coerce').to_numpy(float)
    finite_mad = mad[np.isfinite(mad) & (mad > 0)]
    fallback_mad = float(np.nanmedian(finite_mad)) if finite_mad.size else mad_floor_m
    scale = np.where(np.isfinite(mad) & (mad > 0), mad, fallback_mad)
    scale = np.maximum(scale, mad_floor_m)
    coverage = np.where(np.isfinite(n_obs) & (n_obs > 0), n_obs, 1.0)
    weights = np.sqrt(coverage) / scale
    median_weight = float(np.nanmedian(weights[np.isfinite(weights) & (weights > 0)]))
    weights = np.clip(weights / median_weight, 1.0 / PROFILE_WEIGHT_CAP_RATIO, PROFILE_WEIGHT_CAP_RATIO)
    return weights, scale


def robust_weighted_isotonic(values, base_weights, scale):
    fitted = weighted_pava_increasing(values, base_weights)
    for _ in range(ROBUST_ISOTONIC_ITERATIONS):
        standardized_residual = np.abs((values - fitted) / scale)
        huber_weight = np.minimum(1.0, HUBER_TUNING_CONSTANT / np.maximum(standardized_residual, 1e-12))
        updated = weighted_pava_increasing(values, base_weights * huber_weight)
        if np.max(np.abs(updated - fitted)) < 1e-7:
            fitted = updated
            break
        fitted = updated
    return fitted


candidate_x = profile_table['reference_section_dist_out_m'].to_numpy(float)
candidate_y = profile_table['raw_wse_m'].to_numpy(float)
candidate_q = (candidate_x - candidate_x[0]) / (candidate_x[-1] - candidate_x[0])
candidate_weights, candidate_scale = clipped_profile_weights(profile_table)

# 1. Exact control: the current production equal-weight PAVA profile.
candidate_current_pava = profile_table['monotone_wse_m'].to_numpy(float)
# 2. Same monotonic cone, but uncertainty/count weighting plus Huber IRLS.
candidate_robust_isotonic = robust_weighted_isotonic(candidate_y, candidate_weights, candidate_scale)
# 3. Weighted smoothing spline followed by the same weighted monotone projection.
#    This is a transparent two-stage proxy for a fully joint constrained smoothing fit.
spline = UnivariateSpline(
    candidate_q, candidate_y, w=candidate_weights,
    s=len(candidate_y) * SPLINE_SMOOTHING_PER_ANCHOR,
)
candidate_spline_unconstrained = spline(candidate_q)
candidate_smoothed = weighted_pava_increasing(candidate_spline_unconstrained, candidate_weights)
# 4. Transform before PAVA to impose dH/dx >= S_floor exactly after back-transformation.
candidate_slope_floor = SLOPE_FLOOR_FRACTION_OF_NETWORK_REFERENCE * network_reference_slope
candidate_trend = candidate_slope_floor * (candidate_x - candidate_x[0])
candidate_smoothed_floor = weighted_pava_increasing(
    candidate_spline_unconstrained - candidate_trend, candidate_weights,
) + candidate_trend

candidate_profile_table = pd.DataFrame({
    'reference_section_dist_out_m': candidate_x,
    'raw_wse_m': candidate_y,
    'current_equal_weight_pava': candidate_current_pava,
    'weighted_robust_isotonic': candidate_robust_isotonic,
    'monotone_smoothed_no_floor': candidate_smoothed,
    'monotone_smoothed_with_floor': candidate_smoothed_floor,
})
candidate_profile_table['network_end_to_end_reference'] = (
    candidate_current_pava[0] + network_reference_slope * (candidate_x - candidate_x[0])
)

candidate_fig = go.Figure()
if {'wse_obs_p10', 'wse_obs_p90'}.issubset(profile_table.columns):
    candidate_fig.add_scatter(x=candidate_x, y=profile_table['wse_obs_p10'], mode='lines', line={'width': 0}, showlegend=False, hoverinfo='skip')
    candidate_fig.add_scatter(
        x=candidate_x, y=profile_table['wse_obs_p90'], mode='lines', line={'width': 0},
        fill='tonexty', fillcolor='rgba(120, 120, 120, 0.18)', name='SWORD observed P10–P90 envelope',
    )
candidate_fig.add_scatter(x=candidate_x, y=candidate_y, mode='markers', marker={'size': 4, 'color': 'rgba(30, 30, 30, 0.45)'}, name='raw SWORD P50')
candidate_fig.add_scatter(x=candidate_x, y=candidate_current_pava, mode='lines', line={'color': '#444444', 'dash': 'dot'}, name='1. current equal-weight PAVA')
candidate_fig.add_scatter(x=candidate_x, y=candidate_robust_isotonic, mode='lines', line={'color': '#F58518'}, name='2. weighted robust isotonic')
candidate_fig.add_scatter(x=candidate_x, y=candidate_smoothed, mode='lines', line={'color': '#4C78A8', 'width': 2.5}, name='3. monotone smoothing, no floor')
candidate_fig.add_scatter(x=candidate_x, y=candidate_smoothed_floor, mode='lines', line={'color': '#54A24B', 'width': 2.5}, name='4. monotone smoothing + floor')
candidate_fig.add_scatter(x=candidate_x, y=candidate_profile_table['network_end_to_end_reference'], mode='lines', line={'color': '#777777', 'dash': 'dash'}, name='end-to-end reference slope')
candidate_fig.update_layout(
    title='Four candidate longitudinal WSE profiles against the SWORD P10–P90 envelope',
    xaxis_title='Distance to outlet [m]', yaxis_title='WSE [m]',
    legend={'orientation': 'h', 'y': -0.22}, height=720,
)
candidate_fig.show()

def profile_fit_audit(name, fitted):
    residual = fitted - candidate_y
    within_envelope = np.nan
    if {'wse_obs_p10', 'wse_obs_p90'}.issubset(profile_table.columns):
        lower = pd.to_numeric(profile_table['wse_obs_p10'], errors='coerce').to_numpy(float)
        upper = pd.to_numeric(profile_table['wse_obs_p90'], errors='coerce').to_numpy(float)
        valid = np.isfinite(lower) & np.isfinite(upper)
        within_envelope = float(np.mean((fitted[valid] >= lower[valid]) & (fitted[valid] <= upper[valid])))
    return {
        'candidate': name,
        'RMSE_from_P50_m': float(np.sqrt(np.mean(residual ** 2))),
        'median_abs_deviation_from_P50_m': float(np.median(np.abs(residual))),
        'fraction_within_P10_P90': within_envelope,
        'minimum_slope': float(np.min(np.diff(fitted) / np.diff(candidate_x))),
        'head_loss_m': float(fitted[-1] - fitted[0]),
    }

candidate_audit = pd.DataFrame([
    profile_fit_audit('1. current equal-weight PAVA', candidate_current_pava),
    profile_fit_audit('2. weighted robust isotonic', candidate_robust_isotonic),
    profile_fit_audit('3. monotone smoothing, no floor', candidate_smoothed),
    profile_fit_audit('4. monotone smoothing + floor', candidate_smoothed_floor),
])
print(f'Candidate slope floor = {candidate_slope_floor:.8g} ({SLOPE_FLOOR_FRACTION_OF_NETWORK_REFERENCE:.1%} of network reference slope).')
display(candidate_audit)


### 3.4 Smoothing, uncertainty-floor, and CV sensitivity

`N` is fixed: it is the number of retained profile anchors. The tunable quantity is the
dimensionless multiplier `m = s / N`. Smaller `m` holds the spline closer to P50; larger
`m` permits more smoothing. The MAD-floor audit is deliberately shown before interpreting
any floor sensitivity: a floor below every valid MAD has no effect at all.


In [ ]:
# These values span the active part of the smoothing path for Example 45.
SMOOTHING_BUDGET_MULTIPLIERS = (0.0025, 0.0050,0.011, 0.0100, 0.0250, )
MAD_FLOOR_SENSITIVITY_M = (0.05, 0.15, 0.50)

raw_mad = pd.to_numeric(profile_table['wse_obs_mad'], errors='coerce').to_numpy(float)
valid_mad = np.isfinite(raw_mad) & (raw_mad > 0)
if not valid_mad.any():
    raise RuntimeError('No positive SWORD WSE-MAD values are available for the floor audit.')
mad_floor_audit = pd.DataFrame([
    {
        'mad_floor_m': floor_m,
        'n_profile_anchors': len(raw_mad),
        'n_below_floor': int(np.sum(valid_mad & (raw_mad < floor_m))),
        'fraction_below_floor': float(np.mean(valid_mad & (raw_mad < floor_m))),
        'minimum_observed_mad_m': float(np.nanmin(raw_mad[valid_mad])),
        'median_observed_mad_m': float(np.nanmedian(raw_mad[valid_mad])),
    }
    for floor_m in MAD_FLOOR_SENSITIVITY_M
])
print('MAD-floor audit: a zero count means that floor cannot affect any current profile weight.')
display(mad_floor_audit)

def fitted_smoothed_monotone_profile(mad_floor_m, smoothing_budget_multiplier):
    weights, _ = clipped_profile_weights(profile_table, mad_floor_m=mad_floor_m)
    fitted_spline = UnivariateSpline(
        candidate_q, candidate_y, w=weights,
        s=len(candidate_y) * smoothing_budget_multiplier,
    )
    spline_values = fitted_spline(candidate_q)
    return weighted_pava_increasing(spline_values, weights), float(fitted_spline.get_residual())

smoothing_sensitivity_profiles = {}
smoothing_sensitivity_records = []
for multiplier in SMOOTHING_BUDGET_MULTIPLIERS:
    label = f's/N={multiplier:g}; MAD floor={PROFILE_MAD_FLOOR_M:g} m'
    fitted, spline_residual = fitted_smoothed_monotone_profile(PROFILE_MAD_FLOOR_M, multiplier)
    smoothing_sensitivity_profiles[label] = fitted
    audit = profile_fit_audit(label, fitted)
    requested_budget = len(candidate_y) * multiplier
    audit.update({
        'smoothing_budget_s_over_N': multiplier, 'mad_floor_m': PROFILE_MAD_FLOOR_M,
        'requested_weighted_residual_budget': requested_budget, 'achieved_spline_weighted_residual': spline_residual,
        'spline_saturated_at_maximum_smoothing': bool(spline_residual < 0.999 * requested_budget),
    })
    smoothing_sensitivity_records.append(audit)

# Only show alternative MAD-floor curves when at least one retained node is affected.
for floor_m in MAD_FLOOR_SENSITIVITY_M:
    affected = int(mad_floor_audit.loc[mad_floor_audit['mad_floor_m'].eq(floor_m), 'n_below_floor'].iloc[0])
    if np.isclose(floor_m, PROFILE_MAD_FLOOR_M) or affected == 0:
        continue
    label = f's/N=0.025; MAD floor={floor_m:g} m'
    fitted, spline_residual = fitted_smoothed_monotone_profile(floor_m, 0.025)
    smoothing_sensitivity_profiles[label] = fitted
    audit = profile_fit_audit(label, fitted)
    requested_budget = len(candidate_y) * 0.025
    audit.update({
        'smoothing_budget_s_over_N': 0.025, 'mad_floor_m': floor_m,
        'requested_weighted_residual_budget': requested_budget, 'achieved_spline_weighted_residual': spline_residual,
        'spline_saturated_at_maximum_smoothing': bool(spline_residual < 0.999 * requested_budget),
    })
    smoothing_sensitivity_records.append(audit)

smoothing_sensitivity_audit = pd.DataFrame(smoothing_sensitivity_records)
display(smoothing_sensitivity_audit.sort_values(['mad_floor_m', 'smoothing_budget_s_over_N']))

sensitivity_fig = go.Figure()
if {'wse_obs_p10', 'wse_obs_p90'}.issubset(profile_table.columns):
    sensitivity_fig.add_scatter(x=candidate_x, y=profile_table['wse_obs_p10'], mode='lines', line={'width': 0}, showlegend=False, hoverinfo='skip')
    sensitivity_fig.add_scatter(
        x=candidate_x, y=profile_table['wse_obs_p90'], mode='lines', line={'width': 0},
        fill='tonexty', fillcolor='rgba(120, 120, 120, 0.18)', name='SWORD observed P10–P90 envelope',
    )
sensitivity_fig.add_scatter(x=candidate_x, y=candidate_y, mode='markers', marker={'size': 4, 'color': 'rgba(30, 30, 30, 0.45)'}, name='raw SWORD P50')
sensitivity_fig.add_scatter(x=candidate_x, y=candidate_profile_table['network_end_to_end_reference'], mode='lines', line={'color': '#111111', 'dash': 'dash', 'width': 2}, name='end-to-end reference slope')
palette = ['#4C78A8', '#72B7B2', '#54A24B', '#F58518', '#E45756']
for index, multiplier in enumerate(SMOOTHING_BUDGET_MULTIPLIERS):
    label = f's/N={multiplier:g}; MAD floor={PROFILE_MAD_FLOOR_M:g} m'
    sensitivity_fig.add_scatter(
        x=candidate_x, y=smoothing_sensitivity_profiles[label], mode='lines',
        line={'color': palette[index], 'width': 2.4}, name=f'monotone spline: s/N={multiplier:g}',
    )
for label, fitted in smoothing_sensitivity_profiles.items():
    if not label.startswith('s/N=0.025;'):
        continue
    floor_m = float(label.split('MAD floor=')[1].split(' m')[0])
    sensitivity_fig.add_scatter(
        x=candidate_x, y=fitted, mode='lines', line={'color': '#B279A2', 'dash': 'dot', 'width': 2.4},
        name=f'MAD-floor sensitivity: {floor_m:g} m',
    )
sensitivity_fig.update_layout(
    title='Representative multi-pass WSE profile: smoothing-budget and MAD-floor sensitivity',
    xaxis_title='Distance to outlet [m]', yaxis_title='WSE [m]',
    legend={'orientation': 'h', 'y': -0.28}, height=760,
)
sensitivity_fig.show()


### 3.5 Natural-spline grid diagnostic

This is the proposed replacement for the adaptive `UnivariateSpline` sensitivity above. A classical natural cubic smoothing spline uses every retained unique SWORD coordinate as a *potential* knot. It is still one continuous profile, and has no selected knot spacing or number-of-knots setting.

For a fixed, mean-normalised reliability weight \(w_i^2\), the grid is expressed directly as the residual budget per retained anchor, \(m=s/N\):

\[\sum_i w_i^2[y_i-f(x_i)]^2 \le mN.\]

Internally SciPy parameterises the same problem by \(\lambda\), the roughness penalty. The code below solves for the corresponding \(\lambda\) for each displayed `m`, so `m` remains the only profile-complexity control exposed to this workflow. The selected grid row is the first one whose unconstrained natural spline is monotone over the full continuous profile and therefore needs no PAVA correction. The grid reports two distances from monotonicity: signed minimum continuous slope (a negative value is a local reversal) and maximum PAVA WSE correction required at any anchor. The grid is an inspection aid; final production calibration should bracket that first admissible `m` rather than depend on grid resolution.


In [ ]:
# No knot spacing is chosen here: all retained profile coordinates are used.
# The grid is deliberately coarse for inspection; the eventual calibration brackets the
# first monotone value rather than selecting an arbitrary listed row.
NATURAL_SPLINE_M_GRID = (0.0005, 0.001, 0.0025, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 1)
MONOTONICITY_TOLERANCE = 1e-8  # WSE m per unit normalised reference-section distance
PAVA_ZERO_TOLERANCE_M = 1e-7

# make_smoothing_spline uses sum(w * residual**2). The existing diagnostic weights
# enter our written objective as w_i**2, then are mean-normalised so m retains the
# interpretable unit of average weighted squared WSE residual per anchor.
natural_objective_weights = np.square(candidate_weights)
natural_objective_weights = natural_objective_weights / np.mean(natural_objective_weights)

def natural_spline_min_derivative(spline, *, x_lower=0.0, x_upper=1.0):
    """Exact minimum derivative candidates: endpoints and roots of f'' on each piece."""
    piecewise = PPoly.from_spline(spline)
    first_derivative = piecewise.derivative()
    second_derivative = first_derivative.derivative()
    critical_x = np.concatenate([piecewise.x, second_derivative.roots(extrapolate=False)])
    critical_x = np.unique(critical_x[(critical_x >= x_lower) & (critical_x <= x_upper)])
    return float(np.min(first_derivative(critical_x)))

def natural_spline_for_m(m_value):
    """Fit the natural smoothing spline whose weighted RSS equals s=m*N when reachable."""
    target_rss = float(m_value * len(candidate_y))

    def evaluate(lam):
        fitted_spline = make_smoothing_spline(
            candidate_q, candidate_y, w=natural_objective_weights, lam=float(lam),
        )
        fitted = fitted_spline(candidate_q)
        rss = float(np.sum(natural_objective_weights * np.square(candidate_y - fitted)))
        return fitted_spline, fitted, rss

    _, _, residual_at_zero = evaluate(0.0)
    if target_rss <= residual_at_zero + 1e-12:
        spline, fitted, rss = evaluate(0.0)
        return spline, fitted, rss, 0.0, False

    lambda_upper = 1e-12
    spline_upper, fitted_upper, rss_upper = evaluate(lambda_upper)
    while rss_upper < target_rss and lambda_upper < 1e6:
        lambda_upper *= 100.0
        spline_upper, fitted_upper, rss_upper = evaluate(lambda_upper)
    if rss_upper < target_rss:
        # The requested budget exceeds the stable, attainable smoothing path. Retain
        # the limiting fit and flag it rather than silently changing the definition of m.
        return spline_upper, fitted_upper, rss_upper, lambda_upper, True

    fitted_lambda = brentq(
        lambda value: evaluate(value)[2] - target_rss,
        0.0, lambda_upper, xtol=1e-13, rtol=1e-10,
    )
    spline, fitted, rss = evaluate(fitted_lambda)
    return spline, fitted, rss, float(fitted_lambda), False

natural_grid_profiles = {}
natural_grid_records = []
for m_value in NATURAL_SPLINE_M_GRID:
    natural_spline, natural_fitted, achieved_rss, fitted_lambda, is_saturated = natural_spline_for_m(m_value)
    pava_fitted = weighted_pava_increasing(natural_fitted, natural_objective_weights)
    pava_adjustment = float(np.max(np.abs(pava_fitted - natural_fitted)))
    min_derivative = natural_spline_min_derivative(natural_spline)
    min_physical_slope = min_derivative / (candidate_x[-1] - candidate_x[0])
    head_loss = float(natural_fitted[-1] - natural_fitted[0])
    label = f'm={m_value:g}'
    natural_grid_profiles[label] = natural_fitted
    natural_grid_records.append({
        'm_equals_s_over_N': m_value,
        'N_profile_anchors': len(candidate_y),
        'potential_natural_spline_knots': len(candidate_y),
        'target_weighted_RSS': float(m_value * len(candidate_y)),
        'achieved_weighted_RSS': achieved_rss,
        'lambda_internal': fitted_lambda,
        'smoothing_path_saturated': is_saturated,
        'RMSE_from_raw_P50_m': float(np.sqrt(np.mean(np.square(natural_fitted - candidate_y)))),
        'head_loss_m': head_loss,
        'network_reference_slope': head_loss / (candidate_x[-1] - candidate_x[0]),
        'minimum_continuous_slope_m_per_m': min_physical_slope,
        # A positive deficit is the steepest remaining local upstream WSE reversal.
        'monotonicity_deficit_m_per_m': max(0.0, -min_physical_slope),
        'minimum_derivative_normalised_coordinate': min_derivative,
        'max_PAVA_adjustment_m': pava_adjustment,
        'is_monotone_without_PAVA': bool(min_derivative >= -MONOTONICITY_TOLERANCE),
        'PAVA_is_numerically_zero': bool(pava_adjustment <= PAVA_ZERO_TOLERANCE_M),
    })

natural_spline_grid = pd.DataFrame(natural_grid_records)
admissible_natural_rows = natural_spline_grid.loc[
    natural_spline_grid['is_monotone_without_PAVA']
    & natural_spline_grid['PAVA_is_numerically_zero']
].copy()
selected_natural_grid_row = admissible_natural_rows.head(1)
if selected_natural_grid_row.empty:
    print('No displayed m is yet monotone; expand the grid before treating this profile as admissible.')
else:
    selected_m = float(selected_natural_grid_row['m_equals_s_over_N'].iloc[0])
    print(f'First displayed monotone grid value: m=s/N={selected_m:g}. Inspect the full grid: monotonicity can be non-monotone along the smoothing path, so final calibration must select a stable monotone interval rather than rely on this first crossing alone.')

display(natural_spline_grid)

natural_grid_fig = go.Figure()
if {'wse_obs_p10', 'wse_obs_p90'}.issubset(profile_table.columns):
    natural_grid_fig.add_scatter(
        x=candidate_x, y=pd.to_numeric(profile_table['wse_obs_p10'], errors='coerce'),
        mode='lines', line={'width': 0}, showlegend=False, hoverinfo='skip',
    )
    natural_grid_fig.add_scatter(
        x=candidate_x, y=pd.to_numeric(profile_table['wse_obs_p90'], errors='coerce'),
        mode='lines', line={'width': 0}, fill='tonexty',
        fillcolor='rgba(120, 120, 120, 0.18)', name='SWORD observed P10–P90 envelope',
    )
natural_grid_fig.add_scatter(
    x=candidate_x, y=candidate_y, mode='markers',
    marker={'size': 4, 'color': 'rgba(30, 30, 30, 0.35)'}, name='raw SWORD P50',
)
natural_palette = ['#4C78A8', '#72B7B2', '#54A24B', '#F58518', '#E45756', '#B279A2']
for index, m_value in enumerate(NATURAL_SPLINE_M_GRID):
    row = natural_spline_grid.loc[natural_spline_grid['m_equals_s_over_N'].eq(m_value)].iloc[0]
    natural_grid_fig.add_scatter(
        x=candidate_x, y=natural_grid_profiles[f'm={m_value:g}'], mode='lines',
        line={'color': natural_palette[index % len(natural_palette)], 'width': 2},
        name=(f'm={m_value:g}; deficit={row.monotonicity_deficit_m_per_m:.3g} m/m'),
    )
natural_grid_fig.update_layout(
    title='Natural smoothing-spline grid: all SWORD locations retained; m=s/N is the sole complexity control',
    xaxis_title='Distance to outlet [m]', yaxis_title='WSE [m]',
    legend={'orientation': 'h', 'y': -0.28}, height=760,
)
natural_grid_fig.show()

# These are complementary distances from monotonicity. The slope deficit is the
# maximum local reversal in physical units; the PAVA adjustment is the maximum
# vertical WSE correction that monotonic projection would still require at an anchor.
monotonicity_grid_fig = go.Figure()
monotonicity_grid_fig.add_hline(y=0.0, line={'color': '#222222', 'width': 1})
monotonicity_grid_fig.add_scatter(
    x=natural_spline_grid['m_equals_s_over_N'],
    y=natural_spline_grid['monotonicity_deficit_m_per_m'],
    mode='lines+markers+text', name='local slope deficit (zero = monotone)',
    line={'color': '#4C78A8'}, marker={'size': 8},
    text=[f'{value:.1e}' if value > 0 else '' for value in natural_spline_grid['monotonicity_deficit_m_per_m']],
    textposition='top center',
    hovertemplate='m=%{x:g}<br>local slope deficit=%{y:.4g} m/m<extra></extra>',
)
monotonicity_grid_fig.add_scatter(
    x=natural_spline_grid['m_equals_s_over_N'],
    y=natural_spline_grid['max_PAVA_adjustment_m'],
    mode='lines+markers', name='maximum PAVA WSE adjustment', yaxis='y2',
    line={'color': '#E45756'}, marker={'size': 8},
    hovertemplate='m=%{x:g}<br>maximum PAVA adjustment=%{y:.4g} m<extra></extra>',
)
monotonicity_grid_fig.update_layout(
    title='Distance from monotonicity across the natural-spline grid',
    xaxis_title='m = s / N', yaxis={'title': 'local slope deficit [m/m]; zero = monotone', 'zeroline': True, 'rangemode': 'tozero'},
    yaxis2={'title': 'maximum PAVA adjustment [m]', 'overlaying': 'y', 'side': 'right', 'rangemode': 'tozero'},
    height=480, legend={'orientation': 'h', 'y': -0.23},
)
monotonicity_grid_fig.show()


### 3.6 Count-support diagnostic

This is the deliberately simple candidate: SWORD P50 anchors are weighted only by
their observation count, smoothed on a broad \(m=s/N\) grid, then projected once
with weighted PAVA and a small positive slope floor. Huber and direct-pass values
are not used in this candidate.


In [ ]:
COUNT_SUPPORT_TARGET = 5.0
COUNT_WEIGHTED_M_GRID = (0.01, 0.015, 0.025, 0.04, 0.06, 0.10, 0.15, 0.25, 0.40, 0.60, 1.0)

count_weighted_x = profile_table['reference_section_dist_out_m'].to_numpy(float)
count_weighted_q = (count_weighted_x - count_weighted_x[0]) / (count_weighted_x[-1] - count_weighted_x[0])
count_weighted_y = profile_table['raw_wse_m'].to_numpy(float)
count_weighted_n_obs = pd.to_numeric(profile_table['n_obs'], errors='coerce').fillna(1.0).to_numpy(float)
count_support_weight = np.minimum(1.0, np.maximum(count_weighted_n_obs, 1.0) / COUNT_SUPPORT_TARGET)
count_objective_weight = count_support_weight / np.mean(count_support_weight)

def count_weighted_natural_spline_for_m(m_value):
    target_rss = float(m_value * len(count_weighted_y))

    def evaluate(lam):
        fitted_spline = make_smoothing_spline(
            count_weighted_q, count_weighted_y, w=count_objective_weight, lam=float(lam),
        )
        fitted = fitted_spline(count_weighted_q)
        rss = float(np.sum(count_objective_weight * np.square(count_weighted_y - fitted)))
        return fitted_spline, fitted, rss

    _, _, rss_at_zero = evaluate(0.0)
    if target_rss <= rss_at_zero + 1e-12:
        spline, fitted, rss = evaluate(0.0)
        return spline, fitted, rss, 0.0, False

    lambda_upper = 1e-12
    spline_upper, fitted_upper, rss_upper = evaluate(lambda_upper)
    while rss_upper < target_rss and lambda_upper < 1e8:
        lambda_upper *= 100.0
        spline_upper, fitted_upper, rss_upper = evaluate(lambda_upper)
    if rss_upper < target_rss:
        return spline_upper, fitted_upper, rss_upper, lambda_upper, True

    fitted_lambda = brentq(
        lambda value: evaluate(value)[2] - target_rss,
        0.0, lambda_upper, xtol=1e-13, rtol=1e-10,
    )
    spline, fitted, rss = evaluate(fitted_lambda)
    return spline, fitted, rss, float(fitted_lambda), False

def count_weighted_natural_min_derivative(spline):
    piecewise = PPoly.from_spline(spline)
    first_derivative = piecewise.derivative()
    second_derivative = first_derivative.derivative()
    critical_q = np.unique(np.concatenate([piecewise.x, second_derivative.roots(extrapolate=False)]))
    critical_q = critical_q[(critical_q >= 0.0) & (critical_q <= 1.0)]
    return float(np.min(first_derivative(critical_q)))

count_slope_floor = SLOPE_FLOOR_FRACTION_OF_NETWORK_REFERENCE * network_reference_slope
count_floor_trend = count_slope_floor * (count_weighted_x - count_weighted_x[0])
count_weighted_profiles = {}
count_weighted_records = []
previous_final = None
for m_value in COUNT_WEIGHTED_M_GRID:
    spline, unconstrained, rss, fitted_lambda, saturated = count_weighted_natural_spline_for_m(m_value)
    final_profile = weighted_pava_increasing(
        unconstrained - count_floor_trend, count_objective_weight,
    ) + count_floor_trend
    nodal_slopes = np.diff(final_profile) / np.diff(count_weighted_x)
    label = f'm={m_value:g}'
    count_weighted_profiles[label] = {'unconstrained': unconstrained, 'final': final_profile}
    count_weighted_records.append({
        'm_equals_s_over_N': m_value,
        'target_weighted_RSS': float(m_value * len(count_weighted_y)),
        'achieved_weighted_RSS': rss,
        'lambda_internal': fitted_lambda,
        'smoothing_path_saturated': saturated,
        'RMSE_from_SWORD_P50_m': float(np.sqrt(np.mean(np.square(unconstrained - count_weighted_y)))),
        'unconstrained_minimum_slope_m_per_m': count_weighted_natural_min_derivative(spline) / (count_weighted_x[-1] - count_weighted_x[0]),
        'max_PAVA_adjustment_m': float(np.max(np.abs(final_profile - unconstrained))),
        'final_minimum_nodal_slope_m_per_m': float(np.min(nodal_slopes)),
        'final_head_loss_m': float(final_profile[-1] - final_profile[0]),
        'max_change_from_previous_final_m': np.nan if previous_final is None else float(np.max(np.abs(final_profile - previous_final))),
    })
    previous_final = final_profile

count_weighted_grid = pd.DataFrame(count_weighted_records)
print(
    f'Count support: {int(np.sum(count_support_weight < 1.0))}/{len(count_support_weight)} anchors below '
    f'{COUNT_SUPPORT_TARGET:g} observations; slope floor={count_slope_floor:.4g} m/m.'
)
display(count_weighted_grid)

count_weighted_fig = go.Figure()
count_weighted_fig.add_scatter(
    x=count_weighted_x, y=count_weighted_y, mode='markers',
    marker={'size': 4, 'color': 'rgba(30, 30, 30, 0.35)'}, name='SWORD P50',
)
count_palette = ['#4C78A8', '#72B7B2', '#54A24B', '#F58518', '#E45756', '#B279A2']
for index, m_value in enumerate(COUNT_WEIGHTED_M_GRID):
    row = count_weighted_grid.loc[count_weighted_grid['m_equals_s_over_N'].eq(m_value)].iloc[0]
    count_weighted_fig.add_scatter(
        x=count_weighted_x, y=count_weighted_profiles[f'm={m_value:g}']['final'], mode='lines',
        line={'color': count_palette[index % len(count_palette)], 'width': 2},
        name=f'm={m_value:g}; PAVA={row.max_PAVA_adjustment_m:.3g} m',
    )
count_weighted_fig.update_layout(
    title='Count-weighted SWORD profile: broad smoothing grid and final monotone profiles',
    xaxis_title='Distance to outlet [m]', yaxis_title='WSE [m]', height=720,
    legend={'orientation': 'h', 'y': -0.25},
)
count_weighted_fig.show()


### 3.7 Acquisition-level RiverSP slope audit

This checks whether an apparent profile reversal is present within individual SWOT RiverSP acquisitions, rather than being created by aggregation across different river stages. The primary RiverSP filter is exactly the filter used above. One physical acquisition is the Node granule identified by its file name and acquisition time, **not** a common `time_str`: node timestamps vary along an overpass. Where the local cache contains multiple product versions of the same acquisition, only the newest version is retained.

Distance-to-outlet increases upstream, so a physically downstream-decreasing longitudinal profile has a positive fitted slope. The OLS and endpoint slopes below are whole-window summaries; the local-negative fraction is retained only as a measurement-noise diagnostic.


In [ ]:
# Select one reviewed suspect window. The default is the Example 11 knickpoint;
# Example 22 also has a sparse 300--315 km diagnostic window.
RAW_PASS_WINDOWS_M = {
    '11': [(38_000.0, 47_000.0)],
    '22': [(190_000.0, 210_000.0), (300_000.0, 315_000.0)],
}
RAW_PASS_WINDOW_INDEX = 0
raw_window_m = RAW_PASS_WINDOWS_M.get(
    str(EXAMPLE_ID),
    [(float(profile_candidates['reference_section_dist_out_m'].min()), float(profile_candidates['reference_section_dist_out_m'].max()))],
)[RAW_PASS_WINDOW_INDEX]

raw_pass_records = riversp_observations.loc[passes_existing_riversp_filter].copy()
raw_pass_records['version'] = pd.to_numeric(raw_pass_records['version'], errors='coerce')
# Strip e.g. _PID0_02 or _PGD0_02: these are processing versions of the same acquisition.
raw_pass_records['acquisition_id'] = raw_pass_records['source_file'].astype(str).str.replace(
    r'_[A-Z]+\d+_\d+\.parquet$', '', regex=True,
)
latest_version = raw_pass_records.groupby('acquisition_id')['version'].transform('max')
raw_pass_records = raw_pass_records.loc[raw_pass_records['version'].eq(latest_version)].copy()

raw_pass_profile = raw_pass_records.merge(
    profile_candidates[['sword_node_id', 'reference_section_dist_out_m']],
    left_on='node_id', right_on='sword_node_id', how='inner', validate='m:1',
).drop_duplicates(['acquisition_id', 'node_id']).copy()
raw_pass_window = raw_pass_profile.loc[
    raw_pass_profile['reference_section_dist_out_m'].between(*raw_window_m)
].copy()

def _summarise_raw_acquisition(group):
    group = group.sort_values('reference_section_dist_out_m')
    # Multiple RiverSP nodes can project to one centreline coordinate. Average
    # only for this longitudinal-slope diagnostic, preventing zero-length
    # segments; the joint fit retains every original observation.
    longitudinal = group.groupby('reference_section_dist_out_m', as_index=False).agg(
        wse_sm=('wse_sm', 'mean'),
    ).sort_values('reference_section_dist_out_m')
    x = longitudinal['reference_section_dist_out_m'].to_numpy(float)
    y = longitudinal['wse_sm'].to_numpy(float)
    if len(x) < 2:
        local_slopes = np.array([], dtype=float)
        ols_slope = endpoint_slope = np.nan
    else:
        local_slopes = np.diff(y) / np.diff(x)
        ols_slope = np.polyfit(x, y, 1)[0]
        endpoint_slope = (y[-1] - y[0]) / (x[-1] - x[0])
    return pd.Series({
        'cycle': int(group['cycle'].iloc[0]),
        'pass': int(group['pass'].iloc[0]),
        'first_node_time': str(group['time_str'].min()),
        'n_nodes': len(group),
        'distance_min_km': x.min() / 1_000.0,
        'distance_max_km': x.max() / 1_000.0,
        'ols_slope_m_per_m': ols_slope,
        'endpoint_slope_m_per_m': endpoint_slope,
        'n_negative_local_segments': int((local_slopes < 0).sum()),
        'fraction_negative_local_segments': float((local_slopes < 0).mean()),
        'median_wse_uncertainty_m': group['wse_sm_u'].median(),
    })

raw_pass_summary = (
    raw_pass_window.groupby('acquisition_id', group_keys=False)
    .filter(lambda group: len(group) >= 3)
    .groupby('acquisition_id', as_index=False)
    .apply(lambda group: _summarise_raw_acquisition(group), include_groups=False)
    .reset_index(drop=True)
)
if raw_pass_summary.empty:
    raise RuntimeError('No individual RiverSP acquisition has at least three retained nodes in this window.')

print(
    f'Window: {raw_window_m[0] / 1_000:.1f}--{raw_window_m[1] / 1_000:.1f} km; '
    f'acquisitions with >=3 nodes: {len(raw_pass_summary)}; '
    f'negative OLS slopes: {(raw_pass_summary["ols_slope_m_per_m"] < 0).sum()}.'
)
display(raw_pass_summary.sort_values('ols_slope_m_per_m'))

raw_pass_plot = go.Figure()
raw_pass_centered_plot = go.Figure()
for acquisition_id, group in raw_pass_window.groupby('acquisition_id', sort=True):
    group = group.sort_values('reference_section_dist_out_m')
    if len(group) < 3:
        continue
    label = f"cycle {int(group['cycle'].iloc[0])}, pass {int(group['pass'].iloc[0])}, {str(group['time_str'].min())[:10]}"
    x_km = group['reference_section_dist_out_m'] / 1_000.0
    customdata = group[['node_id', 'wse_sm_u', 'dark_frac', 'xtrk_dist']].to_numpy()
    trace_kwargs = dict(
        x=x_km, mode='lines+markers', name=label,
        marker={'size': 5}, line={'width': 1.5},
        customdata=customdata,
        hovertemplate=(
            'distance=%{x:.3f} km<br>WSE=%{y:.3f} m<br>node=%{customdata[0]}'
            '<br>wse_sm_u=%{customdata[1]:.3f} m<br>dark_frac=%{customdata[2]:.3f}'
            '<br>|xtrk_dist|=%{customdata[3]:.0f} m<extra></extra>'
        ),
    )
    raw_pass_plot.add_scatter(y=group['wse_sm'], **trace_kwargs)
    raw_pass_centered_plot.add_scatter(y=group['wse_sm'] - group['wse_sm'].median(), **trace_kwargs)

for figure, title, y_title in [
    (raw_pass_plot, 'Filtered raw RiverSP profiles by individual acquisition', 'WSE [m]'),
    (raw_pass_centered_plot, 'Same acquisitions after removing each acquisition median stage', 'WSE relative to acquisition median [m]'),
]:
    figure.update_layout(
        title=title, xaxis_title='Distance to outlet [km]', yaxis_title=y_title,
        height=720, legend={'itemclick': 'toggle', 'itemdoubleclick': 'toggleothers'},
    )
    figure.show()


### 3.8 Joint monotone profile and acquisition-stage-offset fit

For retained observations \(z_{ij}\), at corridor station \(x_i\) and physical acquisition \(j\), production estimates
\[z_{ij}=g(x_i)+a_j+\epsilon_{ij},\qquad\sum_j a_j=0.\]
For a chosen \(\lambda\), it minimises
\[\sum_{ij}w_{ij}[z_{ij}-g(x_i)-a_j]^2+\lambda\int[g''(x)]^2\,dx,\]
subject to \(g(x_{i+1})-g(x_i)\ge0.01S_{\rm ref}(x_{i+1}-x_i)\). The implementation alternates stage-offset estimation with a curvature-penalised profile update and projects that update onto the constrained monotone cone. Grouped held-out-acquisition CV and the one-SE rule select \(\lambda\).

In [ ]:
# Run Section 11 first. It builds raw_pass_profile from the raw RiverSP cache using
# the established quality filter and newest product version per physical acquisition.
STAGE_MODEL_WEIGHT_MODE = 'equal'  # 'equal' is the primary diagnostic; use 'capped_inverse_variance' only as a sensitivity check.
STAGE_MODEL_UNCERTAINTY_FLOOR_M = 0.15
STAGE_MODEL_LAMBDA = 1_000.0      # diagnostic only; selected by acquisition-level validation later
STAGE_MODEL_MINIMUM_SLOPE_FRACTION = 0.01  # 1% of the full-corridor reference slope.
STAGE_MODEL_MAX_OUTER_ITERATIONS = 100
STAGE_MODEL_MAX_INNER_ITERATIONS = 500
STAGE_MODEL_TOLERANCE_M = 1e-7

stage_model_observations = raw_pass_profile.copy().sort_values('reference_section_dist_out_m')

# The fit lives on the *complete* fixed SWORD corridor grid, rather than only at
# locations with a retained raw-pass observation. Add the exact realised network
# inlet/outlet projections so the reference head loss and profile extent are exact.
endpoint_reference_dist_out_m = endpoint_nodes.geometry.map(reference_line.project).to_numpy(float)
if not support['outlet_at_start']:
    endpoint_reference_dist_out_m = reference_line.length - endpoint_reference_dist_out_m
endpoint_reference_dist_out_m = np.sort(endpoint_reference_dist_out_m)
if len(endpoint_reference_dist_out_m) != 2 or endpoint_reference_dist_out_m[1] <= endpoint_reference_dist_out_m[0]:
    raise ValueError('Expected two ordered realised endpoint coordinates on the frozen reference line.')

static_x_m = profile_table['reference_section_dist_out_m'].to_numpy(float)
static_pava_wse_m = profile_table['monotone_wse_m'].to_numpy(float)
def _linear_interp_extrapolate(query_x, x, y):
    query_x = np.asarray(query_x, dtype=float)
    fitted = np.interp(query_x, x, y)
    below = query_x < x[0]
    above = query_x > x[-1]
    if below.any():
        fitted[below] = y[0] + (query_x[below] - x[0]) * (y[1] - y[0]) / (x[1] - x[0])
    if above.any():
        fitted[above] = y[-1] + (query_x[above] - x[-1]) * (y[-1] - y[-2]) / (x[-1] - x[-2])
    return fitted

stage_model_x_m = np.unique(np.r_[static_x_m, endpoint_reference_dist_out_m])
stage_model_x_m.sort()
stage_model_reference_profile_m = _linear_interp_extrapolate(stage_model_x_m, static_x_m, static_pava_wse_m)
endpoint_reference_heads_m = _linear_interp_extrapolate(
    endpoint_reference_dist_out_m, static_x_m, static_pava_wse_m,
)
stage_model_network_length_m = float(np.diff(endpoint_reference_dist_out_m)[0])
stage_model_reference_slope_m_per_m = float(
    np.diff(endpoint_reference_heads_m)[0] / stage_model_network_length_m
)
if not np.isfinite(stage_model_reference_slope_m_per_m) or stage_model_reference_slope_m_per_m <= 0.0:
    raise ValueError('Full-corridor static reference slope must be strictly positive.')
stage_model_minimum_slope_m_per_m = float(
    STAGE_MODEL_MINIMUM_SLOPE_FRACTION * stage_model_reference_slope_m_per_m
)
stage_model_endpoint_node_index = np.searchsorted(stage_model_x_m, endpoint_reference_dist_out_m)
if not np.allclose(
    stage_model_x_m[stage_model_endpoint_node_index], endpoint_reference_dist_out_m,
    rtol=0.0, atol=1e-8,
):
    raise ValueError('Exact realised endpoint coordinates are missing from the fixed corridor grid.')
stage_model_node_lookup = {value: index for index, value in enumerate(stage_model_x_m)}
stage_model_acquisitions = pd.Index(sorted(stage_model_observations['acquisition_id'].unique()), name='acquisition_id')
stage_model_acquisition_lookup = {value: index for index, value in enumerate(stage_model_acquisitions)}
stage_model_node_index = stage_model_observations['reference_section_dist_out_m'].map(stage_model_node_lookup)
if stage_model_node_index.isna().any():
    raise ValueError('A retained raw-pass observation could not be placed on the fixed SWORD corridor grid.')
stage_model_node_index = stage_model_node_index.to_numpy(int)
stage_model_observations['_stage_model_node_index'] = stage_model_node_index
stage_model_acquisition_index = stage_model_observations['acquisition_id'].map(stage_model_acquisition_lookup).to_numpy(int)
stage_model_wse_m = stage_model_observations['wse_sm'].to_numpy(float)

if STAGE_MODEL_WEIGHT_MODE == 'equal':
    stage_model_weights = np.ones(len(stage_model_observations), dtype=float)
elif STAGE_MODEL_WEIGHT_MODE == 'capped_inverse_variance':
    # The cap prevents one implausibly small reported uncertainty from dominating a pass.
    stage_model_sigma_m = np.maximum(
        stage_model_observations['wse_sm_u'].to_numpy(float), STAGE_MODEL_UNCERTAINTY_FLOOR_M,
    )
    stage_model_weights = 1.0 / np.square(stage_model_sigma_m)
else:
    raise ValueError(f'Unknown STAGE_MODEL_WEIGHT_MODE: {STAGE_MODEL_WEIGHT_MODE}')

def _pava_nondecreasing(values):
    """Euclidean projection of a vector onto the non-decreasing cone."""
    block_values, block_weights, block_sizes = [], [], []
    for value in np.asarray(values, dtype=float):
        block_values.append(float(value))
        block_weights.append(1.0)
        block_sizes.append(1)
        while len(block_values) > 1 and block_values[-2] > block_values[-1]:
            combined_weight = block_weights[-2] + block_weights[-1]
            block_values[-2] = (
                block_values[-2] * block_weights[-2] + block_values[-1] * block_weights[-1]
            ) / combined_weight
            block_weights[-2] = combined_weight
            block_sizes[-2] += block_sizes[-1]
            block_values.pop(); block_weights.pop(); block_sizes.pop()
    return np.repeat(np.asarray(block_values, dtype=float), block_sizes)

def _curvature_penalty_matrix(x_m):
    """Sparse K such that g.T @ K @ g approximates integral (g''(x))² dx."""
    x_m = np.asarray(x_m, dtype=float)
    x_unit = (x_m - x_m.min()) / (x_m.max() - x_m.min())
    n_nodes = len(x_unit)
    h = np.diff(x_unit)
    if np.any(h <= 0):
        raise ValueError('Profile node coordinates must be strictly increasing.')
    rows, columns, values = [], [], []
    for index in range(1, n_nodes - 1):
        h_left, h_right = h[index - 1], h[index]
        integration_weight = np.sqrt((h_left + h_right) / 2.0)
        row = index - 1
        rows.extend([row, row, row])
        columns.extend([index - 1, index, index + 1])
        values.extend([
            integration_weight * 2.0 / (h_left * (h_left + h_right)),
            integration_weight * -2.0 * (1.0 / h_left + 1.0 / h_right) / (h_left + h_right),
            integration_weight * 2.0 / (h_right * (h_left + h_right)),
        ])
    curvature_operator = sparse.csr_matrix((values, (rows, columns)), shape=(n_nodes - 2, n_nodes))
    penalty = (curvature_operator.T @ curvature_operator).tocsr()
    # This normalisation changes only the numerical scale of lambda, not the fitted family.
    spectral_scale = float(eigsh(penalty, k=1, which='LM', return_eigenvectors=False)[0])
    return penalty / spectral_scale

def fit_joint_stage_monotone_profile(
    x_m, node_index, acquisition_index, wse_m, weights, lambda_value,
    initial_profile_m, minimum_slope_m_per_m, penalty,
):
    """Block-coordinate convex solve for g(x) and acquisition offsets a.

    Given a, g is solved by projected gradient under the minimum-gradient
    constraint. Given g, each a is its closed-form weighted mean residual.
    Alternating these two exact convex blocks converges to the joint constrained
    optimum. Offsets are centred after every update, making g the mean-stage
    profile without imposing static-P50 endpoint heads.
    """
    x_m = np.asarray(x_m, dtype=float)
    n_nodes = len(x_m)
    n_acquisitions = int(acquisition_index.max()) + 1
    sum_weight_by_node = np.bincount(node_index, weights=weights, minlength=n_nodes)
    sum_weight_by_acquisition = np.bincount(acquisition_index, weights=weights, minlength=n_acquisitions)
    if np.any(sum_weight_by_acquisition <= 0):
        raise ValueError('Every acquisition must retain positive observation weight.')

    initial_profile_m = np.asarray(initial_profile_m, dtype=float)
    if initial_profile_m.shape != (n_nodes,):
        raise ValueError('Initial reference profile must have one value for every fixed corridor-grid node.')
    # PAVA is applied to q=g-S_min*x. This exactly enforces the minimum
    # gradient, while allowing multi-pass data to determine total head loss.
    profile = _pava_nondecreasing(
        initial_profile_m - minimum_slope_m_per_m * x_m
    ) + minimum_slope_m_per_m * x_m
    offsets = np.zeros(n_acquisitions, dtype=float)
    if penalty.shape != (n_nodes, n_nodes):
        raise ValueError('Cached curvature penalty does not match the fixed corridor grid.')
    # Largest eigenvalue is one after normalisation, so this is a valid conservative step.
    step_size = 1.0 / (2.0 * sum_weight_by_node.max() + 2.0 * lambda_value)

    for outer_iteration in range(STAGE_MODEL_MAX_OUTER_ITERATIONS):
        profile_before = profile.copy()
        offsets_before = offsets.copy()
        adjusted_sum = np.bincount(
            node_index, weights=weights * (wse_m - offsets[acquisition_index]), minlength=n_nodes,
        )
        for inner_iteration in range(STAGE_MODEL_MAX_INNER_ITERATIONS):
            gradient = (
                2.0 * (sum_weight_by_node * profile - adjusted_sum)
                + 2.0 * lambda_value * (penalty @ profile)
            )
            candidate = _pava_nondecreasing(
                profile - step_size * gradient - minimum_slope_m_per_m * x_m
            ) + minimum_slope_m_per_m * x_m
            if np.max(np.abs(candidate - profile)) <= STAGE_MODEL_TOLERANCE_M:
                profile = candidate
                break
            profile = candidate

        offset_sum = np.bincount(
            acquisition_index, weights=weights * (wse_m - profile[node_index]), minlength=n_acquisitions,
        )
        offsets = offset_sum / sum_weight_by_acquisition
        # Identifiability: transfer the mean acquisition offset into g, leaving
        # mean(a_j)=0. This does not alter slopes or fitted observations.
        offset_shift = offsets.mean()
        offsets -= offset_shift
        profile += offset_shift

        maximum_change = max(
            float(np.max(np.abs(profile - profile_before))),
            float(np.max(np.abs(offsets - offsets_before))),
        )
        if maximum_change <= STAGE_MODEL_TOLERANCE_M:
            break
    else:
        outer_iteration = STAGE_MODEL_MAX_OUTER_ITERATIONS - 1

    residuals = wse_m - profile[node_index] - offsets[acquisition_index]
    return {
        'profile_m': profile, 'offsets_m': offsets, 'residuals_m': residuals,
        'penalty': penalty, 'outer_iterations': outer_iteration + 1, 'inner_iterations': inner_iteration + 1,
        'weighted_mse_m2': float(np.average(np.square(residuals), weights=weights)),
        'weighted_rms_m': float(np.sqrt(np.average(np.square(residuals), weights=weights))),
        'n_observed_profile_nodes': int(np.count_nonzero(sum_weight_by_node)),
    }

# Construct and normalise the sparse curvature penalty once per example. Every
# lambda trial and CV fold uses this immutable cached operator.
stage_model_penalty = _curvature_penalty_matrix(stage_model_x_m)

stage_model_result = fit_joint_stage_monotone_profile(
    stage_model_x_m, stage_model_node_index, stage_model_acquisition_index,
    stage_model_wse_m, stage_model_weights, STAGE_MODEL_LAMBDA,
    stage_model_reference_profile_m, stage_model_minimum_slope_m_per_m, stage_model_penalty,
)
stage_model_profile = pd.DataFrame({
    'reference_section_dist_out_m': stage_model_x_m,
    'joint_monotone_wse_m': stage_model_result['profile_m'],
})
stage_model_fitted_endpoint_heads_m = stage_model_result['profile_m'][stage_model_endpoint_node_index]
stage_model_fitted_head_loss_m = float(np.diff(stage_model_fitted_endpoint_heads_m)[0])
stage_model_static_p50_head_loss_m = float(np.diff(endpoint_reference_heads_m)[0])
stage_model_fitted_global_slope_m_per_m = stage_model_fitted_head_loss_m / stage_model_network_length_m
stage_model_head_loss_difference_m = stage_model_fitted_head_loss_m - stage_model_static_p50_head_loss_m
stage_model_head_loss_relative_difference = (
    stage_model_head_loss_difference_m / stage_model_static_p50_head_loss_m
)
stage_model_profile['joint_interval_slope_m_per_m'] = np.nan
stage_model_profile.loc[stage_model_profile.index[1:], 'joint_interval_slope_m_per_m'] = np.diff(
    stage_model_profile['joint_monotone_wse_m']
) / np.diff(stage_model_profile['reference_section_dist_out_m'])
stage_model_floor_active = stage_model_profile['joint_interval_slope_m_per_m'].iloc[1:].le(
    stage_model_minimum_slope_m_per_m + 1e-12
)
stage_model_observations['joint_fitted_wse_m'] = stage_model_result['profile_m'][stage_model_node_index]
stage_model_observations['acquisition_stage_offset_m'] = stage_model_result['offsets_m'][stage_model_acquisition_index]
stage_model_observations['joint_residual_m'] = stage_model_result['residuals_m']
stage_model_observations['stage_aligned_wse_m'] = (
    stage_model_observations['wse_sm'] - stage_model_observations['acquisition_stage_offset_m']
)

print(
    f"Joint monotone fit: {len(stage_model_profile)} fixed-corridor profile nodes "
    f"({stage_model_result['n_observed_profile_nodes']} directly observed), "
    f"{len(stage_model_acquisitions)} acquisitions, {len(stage_model_observations)} raw observations; "
    f"weighted RMS={stage_model_result['weighted_rms_m']:.3f} m, "
    f"weighted MSE=s/N={stage_model_result['weighted_mse_m2']:.4f} m²; "
    f"outer iterations={stage_model_result['outer_iterations']}."
)
print(
    f"Full-corridor S_ref={stage_model_reference_slope_m_per_m:.6g} m/m; "
    f"S_min={stage_model_minimum_slope_m_per_m:.6g} m/m "
    f"({STAGE_MODEL_MINIMUM_SLOPE_FRACTION:.0%} of S_ref); "
    f"floor-active intervals={int(stage_model_floor_active.sum())}/{len(stage_model_floor_active)}."
)
print(
    f"Fitted multi-pass head loss={stage_model_fitted_head_loss_m:.3f} m "
    f"({stage_model_fitted_global_slope_m_per_m:.6g} m/m); static-P50 head loss="
    f"{stage_model_static_p50_head_loss_m:.3f} m; difference="
    f"{stage_model_head_loss_difference_m:+.3f} m "
    f"({stage_model_head_loss_relative_difference:+.1%})."
)

stage_offset_summary = (
    stage_model_observations.groupby(['acquisition_id', 'cycle', 'pass'], as_index=False)
    .agg(
        first_node_time=('time_str', 'min'), n_nodes=('node_id', 'size'),
        stage_offset_m=('acquisition_stage_offset_m', 'first'),
        residual_rms_m=('joint_residual_m', lambda values: float(np.sqrt(np.mean(np.square(values))))),
    )
    .sort_values('stage_offset_m')
)
display(stage_offset_summary)

joint_profile_fig = go.Figure()
for acquisition_id, group in stage_model_observations.groupby('acquisition_id', sort=True):
    group = group.sort_values('reference_section_dist_out_m')
    label = f"cycle {int(group['cycle'].iloc[0])}, pass {int(group['pass'].iloc[0])}, {str(group['time_str'].min())[:10]}"
    joint_profile_fig.add_scatter(
        x=group['reference_section_dist_out_m'] / 1_000.0, y=group['stage_aligned_wse_m'],
        mode='lines+markers', name=label, marker={'size': 3}, line={'width': 1}, opacity=0.45,
        customdata=group[['node_id', 'acquisition_stage_offset_m', 'joint_residual_m', 'wse_sm_u']].to_numpy(),
        hovertemplate=(
            'distance=%{x:.3f} km<br>aligned WSE=%{y:.3f} m<br>node=%{customdata[0]}'
            '<br>offset=%{customdata[1]:.3f} m<br>residual=%{customdata[2]:.3f} m'
            '<br>wse_sm_u=%{customdata[3]:.3f} m<extra></extra>'
        ),
    )
joint_profile_fig.add_scatter(
    x=stage_model_profile['reference_section_dist_out_m'] / 1_000.0,
    y=stage_model_profile['joint_monotone_wse_m'],
    mode='lines', name='joint monotone profile (S_min = 1% S_ref)', line={'color': 'black', 'width': 4},
)
joint_profile_fig.add_scatter(
    x=profile_candidates['reference_section_dist_out_m'] / 1_000.0, y=profile_candidates['sword_wse'],
    mode='markers', name='current SWORD P50', marker={'color': 'rgba(220,60,60,0.45)', 'size': 4},
)
joint_profile_fig.update_layout(
    title='Stage-aligned raw RiverSP acquisitions and minimum-gradient joint profile',
    xaxis_title='Distance to outlet [km]', yaxis_title='WSE [m]', height=760,
    legend={'itemclick': 'toggle', 'itemdoubleclick': 'toggleothers'},
)
joint_profile_fig.show()

joint_residual_fig = go.Figure()
joint_residual_fig.add_scatter(
    x=stage_model_observations['reference_section_dist_out_m'] / 1_000.0,
    y=stage_model_observations['joint_residual_m'],
    mode='markers', marker={'size': 4, 'color': stage_model_observations['wse_sm_u'], 'colorscale': 'Viridis', 'showscale': True, 'colorbar': {'title': 'wse_sm_u [m]'}},
    name='raw-observation residual',
)
joint_residual_fig.add_hline(y=0.0, line={'color': 'black', 'width': 1})
joint_residual_fig.update_layout(
    title='Joint-model residuals: inspect spatial clusters before accepting a profile',
    xaxis_title='Distance to outlet [km]', yaxis_title='Residual [m]', height=520,
)
joint_residual_fig.show()


## 4. Build the accepted external-anchor registry

Existing assembled-bubble boundaries are reused. Internal compound-member boundaries are
explicitly excluded as longitudinal anchors, even when they lie close to the single
centreline. For every accepted external anchor \(b\), project it to the frozen corridor
coordinate \(x_b\) and assign \(H_b=g(x_b)\). These are the only base-domain fixed heads;
all internal compound heads are solved in Section 6.

In [ ]:
node_reference_dist, node_reference_source = _project_nodes_to_reference_section(
    nodes, support=support, config=SlopeConfig(reference_section_summary_path=str(reference_summary_path))
)
nodes_for_distance = nodes.to_crs(support['centerline_crs']) if str(nodes.crs) != str(support['centerline_crs']) else nodes
degree = pd.concat([links.groupby('id_us_node').size().rename('out_degree'), links.groupby('id_ds_node').size().rename('in_degree')], axis=1).fillna(0)
node_registry = nodes[['id_node', nodes.geometry.name]].copy()
node_registry['reference_section_dist_out_m'] = node_reference_dist
node_registry['centerline_offset_m'] = nodes_for_distance.geometry.distance(reference_line)
node_registry = node_registry.join(degree, on='id_node').fillna({'in_degree': 0, 'out_degree': 0})
node_registry['is_network_source'] = node_registry['in_degree'].eq(0)
node_registry['is_network_sink'] = node_registry['out_degree'].eq(0)
node_registry['is_near_reference_section'] = node_registry['centerline_offset_m'].le(REFERENCE_NODE_MAX_OFFSET_M)

root_metadata = unit_summary[['unit_id', 'compound_bubble_role', 'class', 'is_compound']].rename(columns={'unit_id': 'bubble_root_unit_id'})
reference_domains = bubble_metrics.merge(root_metadata, on='bubble_root_unit_id', how='left', validate='1:1')
for column in ('upstream_boundary_node', 'downstream_boundary_node'):
    reference_domains[column] = pd.to_numeric(reference_domains[column], errors='raise').astype(int)
boundary_rows = []
for row in reference_domains.itertuples(index=False):
    boundary_rows.extend([
        {'id_node': int(row.upstream_boundary_node), 'compound_bubble_id': int(row.compound_bubble_id), 'role': 'upstream'},
        {'id_node': int(row.downstream_boundary_node), 'compound_bubble_id': int(row.compound_bubble_id), 'role': 'downstream'},
    ])
boundary_registry = pd.DataFrame(boundary_rows)
internal_member_nodes = set(pd.to_numeric(unit_summary.loc[unit_summary['compound_bubble_role'].eq('bubble_member'), 'bifurcation'], errors='coerce').dropna().astype(int)) | set(pd.to_numeric(unit_summary.loc[unit_summary['compound_bubble_role'].eq('bubble_member'), 'confluence'], errors='coerce').dropna().astype(int))
node_registry['is_external_domain_boundary'] = node_registry['id_node'].isin(set(boundary_registry['id_node']))
node_registry['is_internal_compound_member_boundary'] = node_registry['id_node'].isin(internal_member_nodes)
node_registry['accepted_reference_anchor'] = node_registry['is_network_source'] | node_registry['is_network_sink'] | (node_registry['is_external_domain_boundary'] & node_registry['is_near_reference_section'])
node_registry['profile_wse_m'] = np.interp(node_registry['reference_section_dist_out_m'], xp, yp)
accepted_registry = node_registry.loc[node_registry['accepted_reference_anchor']].copy()
print(f'Accepted external anchors: {len(accepted_registry)}; internal member boundaries excluded: {node_registry.is_internal_compound_member_boundary.sum()}')
display(accepted_registry.drop(columns=nodes.geometry.name).sort_values('reference_section_dist_out_m', ascending=False))


## 5. Profile-direct unassigned connectors and QA screen

For a connector with no unit ownership, interpolate its endpoint heads from \(g(x)\) and set \(S=(H_u-H_d)/L\). It receives no separate optimiser: the frozen profile is the sole longitudinal data-supported head field.

The historic factor-five screen compares this direct slope against bounded physical support only as a review signal. Nearest-graph-anchor targets and the proposed three-span correction are excluded from production because they can borrow a distant compound reach and manufacture a false error.

In [ ]:
def preferred_length(frame: pd.DataFrame) -> pd.Series:
    for column in ('len', 'len_adj', 'link_length_m'):
        if column in frame.columns:
            values = pd.to_numeric(frame[column], errors='coerce')
            if values.notna().all() and values.gt(0).all():
                return values.astype(float)
    raise ValueError('No complete positive actual link-length field is available.')


def parse_ids(value) -> set[int]:
    if value is None or pd.isna(value) or not str(value).strip():
        return set()
    return {int(float(token.strip())) for token in str(value).split(',') if token.strip()}


def nearest_external_anchor(graph, starts, accepted_ids, blocked_nodes, *, upstream: bool):
    queue = deque((int(node), 0) for node in sorted(set(starts)))
    visited = set(blocked_nodes)
    candidates = []
    while queue:
        node, distance = queue.popleft()
        if node in visited:
            continue
        visited.add(node)
        if node in accepted_ids:
            candidates.append((node, distance))
            continue
        neighbours = graph.predecessors(node) if upstream else graph.successors(node)
        queue.extend((int(next_node), distance + 1) for next_node in neighbours if next_node not in visited)
    if not candidates:
        return None
    min_distance = min(distance for _, distance in candidates)
    nearest = sorted(node for node, distance in candidates if distance == min_distance)
    return nearest[0] if len(nearest) == 1 else None


network = nx.DiGraph()
for row in links.itertuples(index=False):
    network.add_edge(int(row.id_us_node), int(row.id_ds_node), id_link=int(row.id_link))
work_links = links[['id_link', 'id_us_node', 'id_ds_node', 'unit_ids', 'len']].copy()
current_slope_fields = [column for column in ['id_link', 'slope_used', 'raw_slope', 'slope_source_method'] if column in link_context.columns]
work_links = work_links.merge(link_context[current_slope_fields], on='id_link', how='left', validate='1:1')
work_links['actual_length_m'] = preferred_length(work_links)
work_links['is_unassigned'] = work_links['unit_ids'].map(parse_ids).map(len).eq(0)
unassigned = work_links.loc[work_links['is_unassigned']].copy()
accepted_ids = set(accepted_registry['id_node'].astype(int))
profile_by_node = node_registry.set_index('id_node')[['profile_wse_m', 'reference_section_dist_out_m']]

records = []
unassigned_undirected = nx.Graph()
for row in unassigned.itertuples(index=False):
    unassigned_undirected.add_edge(int(row.id_us_node), int(row.id_ds_node), id_link=int(row.id_link))
for component_number, component_nodes in enumerate(nx.connected_components(unassigned_undirected), start=1):
    component_nodes = set(map(int, component_nodes))
    component_links = unassigned.loc[unassigned['id_us_node'].isin(component_nodes) & unassigned['id_ds_node'].isin(component_nodes)].copy()
    entering = [(int(row.id_us_node), int(row.id_ds_node)) for row in links.itertuples(index=False) if int(row.id_us_node) not in component_nodes and int(row.id_ds_node) in component_nodes]
    leaving = [(int(row.id_us_node), int(row.id_ds_node)) for row in links.itertuples(index=False) if int(row.id_us_node) in component_nodes and int(row.id_ds_node) not in component_nodes]
    entry_nodes = sorted({target for _, target in entering})
    exit_nodes = sorted({source for source, _ in leaving})
    record = {'connector_component_id': f'C{component_number:03d}', 'n_links': len(component_links), 'component_link_ids': ','.join(map(str, component_links.id_link.astype(int))), 'entry_nodes': ','.join(map(str, entry_nodes)), 'exit_nodes': ','.join(map(str, exit_nodes))}
    if len(entry_nodes) != 1 or len(exit_nodes) != 1:
        record['screen_status'] = 'non_simple_or_ambiguous_interface'
        records.append(record); continue
    a, b = entry_nodes[0], exit_nodes[0]
    u = nearest_external_anchor(network, [source for source, _ in entering], accepted_ids, component_nodes, upstream=True)
    d = nearest_external_anchor(network, [target for _, target in leaving], accepted_ids, component_nodes, upstream=False)
    record.update({'outer_upstream_anchor': u, 'connector_upstream_node': a, 'connector_downstream_node': b, 'outer_downstream_anchor': d})
    if u is None or d is None or len(component_links) != 1 or not {u, a, b, d}.issubset(profile_by_node.index):
        record['screen_status'] = 'insufficient_simple_connector_context'
        records.append(record); continue
    hu, ha, hb, hd = (float(profile_by_node.loc[node, 'profile_wse_m']) for node in (u, a, b, d))
    du, da, db, dd = (float(profile_by_node.loc[node, 'reference_section_dist_out_m']) for node in (u, a, b, d))
    d_ua, d_bd = du - da, db - dd
    length = float(component_links.iloc[0]['actual_length_m'])
    slope = (ha - hb) / length
    target = ((hu - ha) + (hb - hd)) / (d_ua + d_bd) if d_ua > 0 and d_bd > 0 else np.nan
    ratio = slope / target if np.isfinite(target) and target > 0 and slope > 0 else np.nan
    record.update({
        'connector_length_m': length,
        # Existing final slope, not a new connector-solve result.
        'current_final_slope_used': float(component_links.iloc[0]['slope_used']) if pd.notna(component_links.iloc[0].get('slope_used')) else np.nan,
        'current_final_slope_source_method': component_links.iloc[0].get('slope_source_method', pd.NA),
        'current_raw_slope': float(component_links.iloc[0]['raw_slope']) if pd.notna(component_links.iloc[0].get('raw_slope')) else np.nan,
        # Diagnostic profile estimate and support target used only for screening.
        'profile_connector_slope': slope, 'support_target_slope': target,
        'log_slope_ratio': np.log(ratio) if np.isfinite(ratio) else np.nan,
        'factor_from_target': max(ratio, 1 / ratio) if np.isfinite(ratio) else np.nan,
    })
    record['screen_status'] = 'nonpositive_profile_slope' if not np.isfinite(slope) or slope <= 0 else ('provisional_factor_screen' if record['factor_from_target'] > PROVISIONAL_CONNECTOR_FACTOR else 'within_provisional_screen')
    records.append(record)

connector_screen = pd.DataFrame(records)
display(connector_screen.sort_values(['screen_status', 'connector_component_id']))
print(connector_screen['screen_status'].value_counts(dropna=False).to_string())


### 5.1 Full-example connector-factor calibration (QA only)

This is the reproducible replacement for the earlier exploratory factor-five check.
It evaluates every canonical independent base network in the 16-example study set,
including corrected Examples 13, 45, and 48.

For every context-complete, positive simple connector, calculate

\[
r_c=\log(S_{c,0}/S_{c,\mathrm{target}}).
\]

Across the valid population, calculate

\[
	ilde r=\operatorname{median}(r_c),\qquad
\sigma_r=1.4826\operatorname{MAD}(r_c).
\]

The central three-sigma envelope is
\(\exp(	ilde r-3\sigma_r)\) to \(\exp(	ilde r+3\sigma_r)\).
For a symmetric practical rule around a factor of one, the required factor is

\[
F_\mathrm{symmetric}=\exp(|	ilde r|+3\sigma_r).
\]

The cell also reports the observed coverage under factor five. A final production
threshold is a documented scientific choice after inspecting these results; it is not
automatically written into the workflow.


In [ ]:
CALIBRATION_EXAMPLE_IDS = [
    '02', '03', '05', '11', '13', '15', '21', '22',
    '37', '41', '45', '46', '48', '54', '55', '56',
]


def simple_connector_screen_for_example(example_id: str) -> tuple[pd.DataFrame, dict]:
    # Reproduce section 5 for one frozen canonical base state, without writes.
    exp_dir = OUTPUT_ROOT / f'{example_id}_indep__anchored'
    base_dir = exp_dir / 'states' / 'S000_base'
    local_links = gpd.read_file(one_path(base_dir / 'variant' / 'directed', '*directed_links.gpkg'))
    local_nodes = gpd.read_file(one_path(base_dir / 'variant' / 'directed', '*directed_nodes.gpkg'))
    local_units = pd.read_csv(base_dir / 'hierarchy' / 'unit_summary.csv')
    local_context = pd.read_csv(base_dir / 'hydraulic_context' / 'link_hydraulic_context.csv')
    local_bubbles = pd.read_csv(base_dir / 'hierarchy' / 'bubble_equivalent_metrics.csv')
    local_support = _load_reference_section_support(str(exp_dir / 'reference_section' / 'reference_section_summary.json'))
    local_line = local_support['line']

    reaches = pd.to_numeric(local_nodes['sword_reach_id'], errors='coerce').dropna().astype(int).unique().tolist()
    regions = local_nodes['sword_region'].dropna().astype(str).unique().tolist()
    sword = load_sword_node_reference(
        SWORD_NODE_SOURCE, sword_wse_field='wse_obs_p50',
        regions=regions or None, reach_ids=reaches or None,
    ).to_crs(local_support['centerline_crs'])
    sword['offset_m'] = sword.geometry.distance(local_line)
    station = sword.geometry.map(local_line.project)
    sword['dist_out_m'] = np.where(local_support['outlet_at_start'], station, local_line.length - station)
    profile = sword.loc[
        sword['sword_wse_fill_method'].astype(str).eq('requested_field')
        & pd.to_numeric(sword['sword_wse'], errors='coerce').notna()
        & sword['offset_m'].le(SWORD_CENTERLINE_MAX_OFFSET_M)
    ].drop_duplicates('sword_node_id').groupby('dist_out_m', as_index=False).agg(wse=('sword_wse', 'mean')).sort_values('dist_out_m')
    if len(profile) < 2:
        raise RuntimeError('fewer than two valid profile anchors')
    profile['monotone_wse_m'] = _fit_monotone_increasing_profile(profile['dist_out_m'].to_numpy(float), profile['wse'].to_numpy(float))

    dist_out, _ = _project_nodes_to_reference_section(
        local_nodes, support=local_support,
        config=SlopeConfig(reference_section_summary_path=str(exp_dir / 'reference_section' / 'reference_section_summary.json')),
    )
    projected_nodes = local_nodes.to_crs(local_support['centerline_crs']) if str(local_nodes.crs) != str(local_support['centerline_crs']) else local_nodes
    degree = pd.concat([local_links.groupby('id_us_node').size().rename('out_degree'), local_links.groupby('id_ds_node').size().rename('in_degree')], axis=1).fillna(0)
    node_frame = local_nodes[['id_node']].copy()
    node_frame['reference_section_dist_out_m'] = dist_out
    node_frame['offset_m'] = projected_nodes.geometry.distance(local_line)
    node_frame = node_frame.join(degree, on='id_node').fillna({'in_degree': 0, 'out_degree': 0})

    boundary_ids = set(pd.to_numeric(local_bubbles['upstream_boundary_node'], errors='coerce').dropna().astype(int)) | set(pd.to_numeric(local_bubbles['downstream_boundary_node'], errors='coerce').dropna().astype(int))
    accepted = set(node_frame.loc[
        node_frame['in_degree'].eq(0) | node_frame['out_degree'].eq(0) | (node_frame['id_node'].isin(boundary_ids) & node_frame['offset_m'].le(REFERENCE_NODE_MAX_OFFSET_M)),
        'id_node',
    ].astype(int))
    node_frame['profile_wse_m'] = np.interp(node_frame['reference_section_dist_out_m'], profile['dist_out_m'], profile['monotone_wse_m'])
    profile_by_node = node_frame.set_index('id_node')[['profile_wse_m', 'reference_section_dist_out_m']]

    work = local_links[['id_link', 'id_us_node', 'id_ds_node', 'unit_ids', 'len']].copy()
    local_slope_fields = [column for column in ['id_link', 'slope_used', 'raw_slope', 'slope_source_method'] if column in local_context.columns]
    work = work.merge(local_context[local_slope_fields], on='id_link', how='left', validate='1:1')
    work['actual_length_m'] = preferred_length(work)
    unassigned_mask = work['unit_ids'].map(parse_ids).map(len).eq(0)
    unassigned_local = work.loc[unassigned_mask].copy()
    graph = nx.DiGraph()
    for row in local_links.itertuples(index=False):
        graph.add_edge(int(row.id_us_node), int(row.id_ds_node), id_link=int(row.id_link))
    undirected = nx.Graph()
    for row in unassigned_local.itertuples(index=False):
        undirected.add_edge(int(row.id_us_node), int(row.id_ds_node), id_link=int(row.id_link))

    records = []
    for number, component in enumerate(nx.connected_components(undirected), start=1):
        component = set(map(int, component))
        component_links = unassigned_local.loc[unassigned_local['id_us_node'].isin(component) & unassigned_local['id_ds_node'].isin(component)].copy()
        entering = [(int(row.id_us_node), int(row.id_ds_node)) for row in local_links.itertuples(index=False) if int(row.id_us_node) not in component and int(row.id_ds_node) in component]
        leaving = [(int(row.id_us_node), int(row.id_ds_node)) for row in local_links.itertuples(index=False) if int(row.id_us_node) in component and int(row.id_ds_node) not in component]
        entries, exits = sorted({b for _, b in entering}), sorted({a for a, _ in leaving})
        record = {'example_id': example_id, 'connector_component_id': f'C{number:03d}', 'n_links': len(component_links), 'component_link_ids': ','.join(map(str, component_links.id_link.astype(int))), 'screen_status': ''}
        if len(entries) != 1 or len(exits) != 1:
            record['screen_status'] = 'non_simple_or_ambiguous_interface'; records.append(record); continue
        a, b = entries[0], exits[0]
        u = nearest_external_anchor(graph, [a0 for a0, _ in entering], accepted, component, upstream=True)
        d = nearest_external_anchor(graph, [b0 for _, b0 in leaving], accepted, component, upstream=False)
        record.update({'outer_upstream_anchor': u, 'connector_upstream_node': a, 'connector_downstream_node': b, 'outer_downstream_anchor': d})
        if u is None or d is None or len(component_links) != 1 or not {u, a, b, d}.issubset(profile_by_node.index):
            record['screen_status'] = 'insufficient_simple_connector_context'; records.append(record); continue
        hu, ha, hb, hd = (float(profile_by_node.loc[node, 'profile_wse_m']) for node in (u, a, b, d))
        du, da, db, dd = (float(profile_by_node.loc[node, 'reference_section_dist_out_m']) for node in (u, a, b, d))
        d_ua, d_bd = du - da, db - dd
        length = float(component_links.iloc[0]['actual_length_m'])
        slope = (ha - hb) / length
        target = ((hu - ha) + (hb - hd)) / (d_ua + d_bd) if d_ua > 0 and d_bd > 0 else np.nan
        ratio = slope / target if np.isfinite(slope) and slope > 0 and np.isfinite(target) and target > 0 else np.nan
        record.update({
            'connector_length_m': length,
            'current_final_slope_used': float(component_links.iloc[0]['slope_used']) if pd.notna(component_links.iloc[0].get('slope_used')) else np.nan,
            'current_final_slope_source_method': component_links.iloc[0].get('slope_source_method', pd.NA),
            'current_raw_slope': float(component_links.iloc[0]['raw_slope']) if pd.notna(component_links.iloc[0].get('raw_slope')) else np.nan,
            'profile_connector_slope': slope, 'support_target_slope': target,
            'log_slope_ratio': np.log(ratio) if np.isfinite(ratio) else np.nan,
            'factor_from_target': max(ratio, 1 / ratio) if np.isfinite(ratio) else np.nan,
        })
        record['screen_status'] = 'nonpositive_profile_slope' if not np.isfinite(slope) or slope <= 0 else 'positive_context_complete'
        records.append(record)
    return pd.DataFrame(records), {'example_id': example_id, 'n_profile_anchors': len(profile), 'n_accepted_anchors': len(accepted)}


calibration_frames, calibration_runs, calibration_failures = [], [], []
for example in CALIBRATION_EXAMPLE_IDS:
    try:
        frame, run = simple_connector_screen_for_example(example)
        calibration_frames.append(frame); calibration_runs.append(run)
    except Exception as exc:
        calibration_failures.append({'example_id': example, 'error': f'{type(exc).__name__}: {exc}'})

connector_calibration = pd.concat(calibration_frames, ignore_index=True) if calibration_frames else pd.DataFrame()
calibration_run_summary = pd.DataFrame(calibration_runs)
calibration_failure_summary = pd.DataFrame(calibration_failures)
valid_connector_ratios = connector_calibration.loc[
    connector_calibration['screen_status'].eq('positive_context_complete')
    & pd.to_numeric(connector_calibration['log_slope_ratio'], errors='coerce').notna(),
    'log_slope_ratio',
].astype(float)
if valid_connector_ratios.empty:
    raise RuntimeError('No positive context-complete connector ratios were available for calibration.')
centre = float(valid_connector_ratios.median())
mad = float(np.median(np.abs(valid_connector_ratios - centre)))
sigma = float(1.4826 * mad)
lower_factor, upper_factor = float(np.exp(centre - 3 * sigma)), float(np.exp(centre + 3 * sigma))
symmetric_factor = float(np.exp(abs(centre) + 3 * sigma))
factor_five_coverage = float((np.exp(np.abs(valid_connector_ratios)) <= 5.0).mean())
calibration_summary = pd.DataFrame([{
    'n_examples_requested': len(CALIBRATION_EXAMPLE_IDS), 'n_examples_completed': len(calibration_runs),
    'n_examples_failed': len(calibration_failures), 'n_positive_context_complete': len(valid_connector_ratios),
    'median_log_ratio': centre, 'robust_sigma_log_ratio': sigma,
    'three_sigma_lower_ratio': lower_factor, 'three_sigma_upper_ratio': upper_factor,
    'symmetric_three_sigma_factor': symmetric_factor,
    'factor_five_coverage_fraction': factor_five_coverage,
    'factor_five_coverage_percent': 100 * factor_five_coverage,
}])

display(calibration_summary)
print('Connector classification counts')
display(connector_calibration.groupby(['screen_status']).size().rename('n_components').reset_index())
if not calibration_failure_summary.empty:
    print('Examples not included because their diagnostic could not be completed:')
    display(calibration_failure_summary)
display(connector_calibration.loc[
    connector_calibration['screen_status'].eq('nonpositive_profile_slope')
    | connector_calibration['factor_from_target'].gt(5),
].sort_values(['example_id', 'factor_from_target'], ascending=[True, False]))


## 6. Base simple-unit and compound-domain graph-potential solver

External heads come only from the frozen profile. For every realised simple-unit or compound-bubble DAG \(D\),
\[S_{\rm ref,D}=\frac{H_{\rm source}-H_{\rm sink}}{L_{\rm reference,D}},\qquad S_e=\frac{H_{u,e}-H_{d,e}}{L_e},\]
\[\min_H\sum_{e\in D}\left(\frac{S_e-S_{\rm ref,D}}{S_{\rm ref,D}}\right)^2\quad\text{subject to}\quad S_e\ge0.01S_{\rm ref,D}.\]

`solve_boundary_conditioned_domain` uses realised `len` (with only the established fallback), equal link weights, SLSQP with an analytic Jacobian, and an exhaustive path-drop audit. Equal weights prevent short links from becoming cheap locations for concentrated head loss. The lower bound is a physical constraint aligned with the profile floor, not a numerical epsilon.

In [ ]:
# def links_for_selected_domain(kind: str, identifier: int):
#     kind = str(kind).strip().lower()
#     if kind == 'compound_bubble':
#         # Reuse the current context's unique link owner rather than broad,
#         # overlapping membership strings. This makes the visual review use
#         # precisely the same one-domain-per-link rule as section 8.
#         owner = pd.to_numeric(link_context['compound_bubble_id_link'], errors='coerce')
#         membership = owner.eq(int(identifier))
#         selected = links.loc[membership].copy()
#         domain = reference_domains.loc[reference_domains['compound_bubble_id'].eq(int(identifier))].copy()
#         if len(domain) != 1:
#             raise ValueError(f'Expected exactly one assembled bubble {identifier}, found {len(domain)}.')
#         row = domain.iloc[0]
#         source, sink = int(row.upstream_boundary_node), int(row.downstream_boundary_node)
#         label = f'compound bubble {identifier}'
#     elif kind == 'simple_unit':
#         membership = links['unit_ids'].map(parse_ids).map(lambda values: int(identifier) in values)
#         # A link owned by an assembled bubble is not re-used by a simple
#         # unit visualisation.
#         membership &= pd.to_numeric(link_context['compound_bubble_id_link'], errors='coerce').isna()
#         selected = links.loc[membership].copy()
#         domain = unit_summary.loc[pd.to_numeric(unit_summary['unit_id'], errors='coerce').eq(int(identifier))].copy()
#         if len(domain) != 1:
#             raise ValueError(f'Expected exactly one unit {identifier}, found {len(domain)}.')
#         row = domain.iloc[0]
#         if bool(row.get('is_compound', False)):
#             raise ValueError('Use DEMO_DOMAIN_KIND="compound_bubble" for a compound unit.')
#         source, sink = int(row.bifurcation), int(row.confluence)
#         label = f'simple unit {identifier}'
#     else:
#         raise ValueError("DEMO_DOMAIN_KIND must be 'compound_bubble' or 'simple_unit'.")
#     if selected.empty:
#         raise ValueError(f'No realised links belong to {label}.')
#     selected['actual_length_m'] = preferred_length(selected)
#     return selected, source, sink, label


# def path_drop_audit(graph, drop_by_link, source: int, sink: int) -> dict:
#     if not nx.is_directed_acyclic_graph(nx.DiGraph(graph)):
#         raise ValueError('A strictly decreasing scalar head field is impossible on a directed cycle.')
#     order = list(nx.topological_sort(nx.DiGraph(graph)))
#     min_drop = {node: np.inf for node in graph.nodes}; max_drop = {node: -np.inf for node in graph.nodes}; n_paths = {node: 0 for node in graph.nodes}
#     min_drop[source] = max_drop[source] = 0.0; n_paths[source] = 1
#     for node in order:
#         if n_paths[node] == 0:
#             continue
#         for _, target, key in graph.out_edges(node, keys=True):
#             drop = float(drop_by_link[int(key)])
#             min_drop[target] = min(min_drop[target], min_drop[node] + drop)
#             max_drop[target] = max(max_drop[target], max_drop[node] + drop)
#             n_paths[target] += n_paths[node]
#     return {'n_paths': int(n_paths[sink]), 'minimum_path_drop_m': float(min_drop[sink]), 'maximum_path_drop_m': float(max_drop[sink])}


# def solve_boundary_conditioned_domain(selected_links, source, sink, source_head, sink_head, reference_distance_m):
#     graph = nx.MultiDiGraph()
#     for row in selected_links.itertuples(index=False):
#         graph.add_edge(int(row.id_us_node), int(row.id_ds_node), key=int(row.id_link), length_m=float(row.actual_length_m))
#     simple_graph = nx.DiGraph(graph)
#     if not nx.is_directed_acyclic_graph(simple_graph) or not nx.has_path(simple_graph, source, sink):
#         raise ValueError('Selected domain must be an acyclic connected source-to-sink graph.')
#     internal = sorted(set(graph.nodes).difference({source, sink}))
#     index = {node: i for i, node in enumerate(internal)}
#     lengths = selected_links['actual_length_m'].to_numpy(float)
#     fixed_drop, incidence = np.zeros(len(selected_links)), np.zeros((len(selected_links), len(internal)))
#     for edge_index, row in enumerate(selected_links.itertuples(index=False)):
#         for node, sign in ((int(row.id_us_node), 1.0), (int(row.id_ds_node), -1.0)):
#             if node in index:
#                 incidence[edge_index, index[node]] += sign
#             else:
#                 fixed_drop[edge_index] += sign * (source_head if node == source else sink_head)
#     total_drop = float(source_head - sink_head)
#     if total_drop <= 0 or reference_distance_m <= 0:
#         raise ValueError('Boundary heads and reference distance must define a positive local slope.')
#     s_ref = total_drop / reference_distance_m
#     longest_to_sink = {sink: 0.0}
#     for node in reversed(list(nx.topological_sort(simple_graph))):
#         if node != sink:
#             longest_to_sink[node] = max(float(data['length_m']) + longest_to_sink[target] for _, target, _, data in graph.out_edges(node, keys=True, data=True))
#     floor = min(NUMERICAL_SLOPE_FLOOR, 0.1 * total_drop / longest_to_sink[source])
#     def drops(x): return fixed_drop + incidence @ x
#     def objective(x): return float(np.sum(lengths * ((drops(x) / lengths - s_ref) / s_ref) ** 2))
#     def jacobian(x): return 2.0 * incidence.T @ ((drops(x) / lengths - s_ref) / s_ref**2)
#     if internal:
#         x0 = np.asarray([sink_head + total_drop * longest_to_sink[node] / longest_to_sink[source] for node in internal])
#         result = minimize(objective, x0, jac=jacobian, method='SLSQP', constraints=[LinearConstraint(incidence, floor * lengths - fixed_drop, np.full(len(lengths), np.inf))], options={'ftol': 1e-12, 'maxiter': 5000})
#         if not result.success:
#             raise RuntimeError(result.message)
#         solved = result.x; message = str(result.message)
#     else:
#         solved = np.asarray([]); message = 'no_internal_nodes_direct_boundary_solution'
#         if np.any(drops(solved) < floor * lengths):
#             raise RuntimeError('Fixed boundary heads do not provide positive drops for every direct link.')
#     heads = {source: source_head, sink: sink_head, **{node: float(solved[i]) for node, i in index.items()}}
#     candidate = selected_links.copy()
#     candidate['head_us_m'] = candidate['id_us_node'].map(heads); candidate['head_ds_m'] = candidate['id_ds_node'].map(heads)
#     candidate['candidate_slope'] = (candidate['head_us_m'] - candidate['head_ds_m']) / candidate['actual_length_m']
#     drops_by_link = dict(zip(candidate['id_link'].astype(int), candidate['candidate_slope'] * candidate['actual_length_m']))
#     audit = path_drop_audit(graph, drops_by_link, source, sink)
#     audit.update({'boundary_head_loss_m': total_drop, 'local_reference_slope': s_ref, 'max_abs_path_residual_m': max(abs(audit['minimum_path_drop_m'] - total_drop), abs(audit['maximum_path_drop_m'] - total_drop)), 'objective_value': objective(solved), 'numerical_slope_floor': floor, 'optimizer_message': message})
#     return candidate, pd.DataFrame([audit]), heads


# selected_links, source, sink, domain_label = links_for_selected_domain(DEMO_DOMAIN_KIND, DEMO_DOMAIN_ID)
# heads_by_anchor = accepted_registry.set_index('id_node')['profile_wse_m']
# if source not in heads_by_anchor.index or sink not in heads_by_anchor.index:
#     raise ValueError(f'{domain_label} boundaries {source}->{sink} are not both accepted external anchors.')
# source_head, sink_head = float(heads_by_anchor[source]), float(heads_by_anchor[sink])
# distances = node_registry.set_index('id_node')['reference_section_dist_out_m']
# reference_distance = float(distances[source] - distances[sink])
# candidate_links, candidate_audit, candidate_heads = solve_boundary_conditioned_domain(selected_links, source, sink, source_head, sink_head, reference_distance)

# old_fields = [column for column in ['id_link', 'slope_used', 'raw_slope', 'slope_source_method', 'slope_section_ref'] if column in link_context.columns]
# comparison = candidate_links.merge(link_context[old_fields], on='id_link', how='left', validate='1:1')
# comparison['candidate_to_current_ratio'] = comparison['candidate_slope'] / pd.to_numeric(comparison['slope_used'], errors='coerce')
# comparison['abs_log_change'] = np.abs(np.log(comparison['candidate_to_current_ratio']))
# display(pd.DataFrame([{'domain': domain_label, 'source_node': source, 'sink_node': sink, 'source_head_m': source_head, 'sink_head_m': sink_head, 'reference_distance_m': reference_distance}]))
# display(candidate_audit)
# display(comparison[['id_link', 'id_us_node', 'id_ds_node', 'actual_length_m', 'slope_used', 'raw_slope', 'slope_source_method', 'candidate_slope', 'candidate_to_current_ratio', 'head_us_m', 'head_ds_m']].sort_values('id_link'))
# assert candidate_audit.loc[0, 'max_abs_path_residual_m'] < 1e-8

# fig = go.Figure()
# for row in comparison.itertuples(index=False):
#     xy = np.asarray(row.geometry.coords)
#     ratio = float(row.candidate_to_current_ratio) if np.isfinite(row.candidate_to_current_ratio) else np.nan
#     color = '#E45756' if np.isfinite(ratio) and (ratio > 1.25 or ratio < 0.8) else '#4C78A8'
#     fig.add_scatter(x=xy[:, 0], y=xy[:, 1], mode='lines', line={'width': 5, 'color': color}, name=str(int(row.id_link)), showlegend=False, hovertemplate=f'link={int(row.id_link)}<br>current slope={float(row.slope_used):.8g}<br>candidate slope={float(row.candidate_slope):.8g}<br>candidate/current={ratio:.4g}<br>length={float(row.actual_length_m):.1f} m<extra>{domain_label}</extra>')
# node_geometry = nodes.set_index('id_node').loc[sorted(candidate_heads), nodes.geometry.name]
# fig.add_scatter(x=[geometry.x for geometry in node_geometry], y=[geometry.y for geometry in node_geometry], mode='markers+text', marker={'size': 9, 'color': 'black'}, text=[f'{node}<br>{candidate_heads[node]:.3f} m' for node in sorted(candidate_heads)], textposition='top center', name='candidate heads')
# fig.update_layout(title=f'{domain_label}: candidate/current slope comparison (red = >25% change)', xaxis={'scaleanchor': 'y', 'scaleratio': 1}, xaxis_title='Projected x', yaxis_title='Projected y')
# fig.show()

# scatter = go.Figure()
# scatter.add_scatter(x=comparison['slope_used'], y=comparison['candidate_slope'], mode='markers+text', text=comparison['id_link'].astype(str), textposition='top center', marker={'size': 9, 'color': comparison['abs_log_change'], 'colorscale': 'Viridis', 'colorbar': {'title': '|log ratio|'}}, hovertemplate='link=%{text}<br>current=%{x:.8g}<br>candidate=%{y:.8g}<extra></extra>')
# finite = pd.concat([comparison['slope_used'], comparison['candidate_slope']]).replace([np.inf, -np.inf], np.nan).dropna()
# if not finite.empty:
#     lo, hi = float(finite.min()), float(finite.max())
#     scatter.add_scatter(x=[lo, hi], y=[lo, hi], mode='lines', line={'dash': 'dash', 'color': 'black'}, name='1:1')
# scatter.update_layout(title=f'{domain_label}: current versus candidate slope', xaxis={'type': 'log', 'title': 'current slope_used'}, yaxis={'type': 'log', 'title': 'candidate slope'})
# scatter.show()


## 7. All-base validation, celerity sensitivity, and routing impact

This is an **in-memory diagnostic only**. It does not modify a hydraulic context,
RAPID preparation, or any experiment product.

For every standalone simple unit and every assembled compound bubble in the 16
study bases, it computes the proposed boundary-conditioned slope field, compares
it with the current final `slope_used`, and translates that difference to an
approximate link-level RAPID effect using the stored celerity caps and the routing
width field, including explicit below-minimum / within-range / above-maximum cap
transitions. Compound members are deliberately not solved separately: they are
represented once by their assembled bubble. Unassigned connectors remain an
explicitly reported, not-yet-refitted scope until their three-span conditioning
rule is promoted. A realised link with conflicting membership in more than one
candidate base domain is also excluded and reported; this screen never selects
one arbitrarily.

Equivalent resistance is intentionally absent from this screen. Its current
Manning/geometry edge resistance uses length, width, roughness and \(k_b\), then
solves a normalized unit-head problem; it has no `slope_used` input. Therefore a
slope-only refit produces an expected equivalent-resistance ratio of exactly one.


In [ ]:
# # This diagnostic is deliberately separate from production hydraulic-context generation.
# # It is safe to rerun: it reads frozen base products and keeps all results in memory.
# BASE_IMPACT_EXAMPLE_IDS = CALIBRATION_EXAMPLE_IDS.copy()


# def as_bool(value) -> bool:
#     return str(value).strip().lower() in {'true', '1', 'yes'}


# def canonical_base_inputs(example_id: str) -> dict:
#     exp_dir = OUTPUT_ROOT / f'{example_id}_indep__anchored'
#     state_dir = exp_dir / 'states' / 'S000_base'
#     variant_dir = state_dir / 'variant'
#     directed_dir = variant_dir / 'directed'
#     hydraulic_dir = state_dir / 'hydraulic_context'
#     local_links = gpd.read_file(one_path(directed_dir, '*directed_links.gpkg'))
#     # Direction products intentionally exclude the routing-width family. Merge
#     # it only for the read-only RAPID-response translation below.
#     local_widths = gpd.read_file(variant_dir / 'widths' / 'links_with_width_families.gpkg')
#     width_fields = [column for column in ['id_link', 'wid_adj_routing', 'wid_adj_wet', 'wid_adj_orig'] if column in local_widths.columns]
#     local_links = local_links.merge(local_widths[width_fields], on='id_link', how='left', validate='1:1')
#     local_nodes = gpd.read_file(one_path(directed_dir, '*directed_nodes.gpkg'))
#     local_context = pd.read_csv(hydraulic_dir / 'link_hydraulic_context.csv')
#     local_units = pd.read_csv(state_dir / 'hierarchy' / 'unit_summary.csv')
#     local_bubbles = pd.read_csv(state_dir / 'hierarchy' / 'bubble_equivalent_metrics.csv')
#     local_support = _load_reference_section_support(str(exp_dir / 'reference_section' / 'reference_section_summary.json'))
#     local_line = local_support['line']
#     return {
#         'example_id': example_id, 'experiment_dir': exp_dir, 'state_dir': state_dir,
#         'links': local_links, 'nodes': local_nodes, 'context': local_context,
#         'units': local_units, 'bubbles': local_bubbles, 'support': local_support,
#         'reference_line': local_line,
#     }


# def base_reference_heads(data: dict) -> tuple[pd.Series, pd.Series, pd.DataFrame]:
#     local_links, local_nodes = data['links'], data['nodes']
#     local_support, local_line = data['support'], data['reference_line']
#     reaches = pd.to_numeric(local_nodes['sword_reach_id'], errors='coerce').dropna().astype(int).unique().tolist()
#     regions = local_nodes['sword_region'].dropna().astype(str).unique().tolist()
#     sword = load_sword_node_reference(
#         SWORD_NODE_SOURCE, sword_wse_field='wse_obs_p50', regions=regions or None, reach_ids=reaches or None,
#     ).to_crs(local_support['centerline_crs'])
#     sword['offset_m'] = sword.geometry.distance(local_line)
#     station = sword.geometry.map(local_line.project)
#     sword['dist_out_m'] = np.where(local_support['outlet_at_start'], station, local_line.length - station)
#     profile = sword.loc[
#         sword['sword_wse_fill_method'].astype(str).eq('requested_field')
#         & pd.to_numeric(sword['sword_wse'], errors='coerce').notna()
#         & sword['offset_m'].le(SWORD_CENTERLINE_MAX_OFFSET_M)
#     ].drop_duplicates('sword_node_id').groupby('dist_out_m', as_index=False).agg(wse=('sword_wse', 'mean')).sort_values('dist_out_m')
#     if len(profile) < 2:
#         raise RuntimeError('fewer than two valid SWORD profile anchors')
#     profile['monotone_wse_m'] = _fit_monotone_increasing_profile(profile['dist_out_m'].to_numpy(float), profile['wse'].to_numpy(float))
#     dist_out, _ = _project_nodes_to_reference_section(
#         local_nodes, support=local_support,
#         config=SlopeConfig(reference_section_summary_path=str(data['experiment_dir'] / 'reference_section' / 'reference_section_summary.json')),
#     )
#     projected_nodes = local_nodes.to_crs(local_support['centerline_crs']) if str(local_nodes.crs) != str(local_support['centerline_crs']) else local_nodes
#     degree = pd.concat([local_links.groupby('id_us_node').size().rename('out_degree'), local_links.groupby('id_ds_node').size().rename('in_degree')], axis=1).fillna(0)
#     node_frame = local_nodes[['id_node']].copy()
#     node_frame['reference_section_dist_out_m'] = dist_out
#     node_frame['offset_m'] = projected_nodes.geometry.distance(local_line)
#     node_frame = node_frame.join(degree, on='id_node').fillna({'in_degree': 0, 'out_degree': 0})
#     boundary_ids = set(pd.to_numeric(data['bubbles']['upstream_boundary_node'], errors='coerce').dropna().astype(int)) | set(pd.to_numeric(data['bubbles']['downstream_boundary_node'], errors='coerce').dropna().astype(int))
#     node_frame['accepted_reference_anchor'] = (
#         node_frame['in_degree'].eq(0) | node_frame['out_degree'].eq(0)
#         | (node_frame['id_node'].isin(boundary_ids) & node_frame['offset_m'].le(REFERENCE_NODE_MAX_OFFSET_M))
#     )
#     node_frame['profile_wse_m'] = np.interp(node_frame['reference_section_dist_out_m'], profile['dist_out_m'], profile['monotone_wse_m'])
#     return node_frame.set_index('id_node')['profile_wse_m'], node_frame.set_index('id_node')['reference_section_dist_out_m'], node_frame


# def proposed_base_domains(data: dict, heads: pd.Series, distances: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
#     local_links, units, bubbles, context = data['links'], data['units'], data['bubbles'], data['context']
#     # Reuse the current workflow's conservative compound-bubble assignment.
#     # `compound_bubble_id_link` is only populated where `_apply_compound_bubble_qc`
#     # resolved one unique bubble. It is the appropriate ownership source, rather
#     # than re-expanding overlapping hierarchy membership strings.
#     context_fields = [column for column in ['id_link', 'compound_bubble_id_link', 'bubble_slope_qc_reason'] if column in context.columns]
#     ownership = local_links[['id_link', 'unit_ids']].merge(context[context_fields], on='id_link', how='left', validate='1:1')
#     ownership['compound_bubble_id_link'] = pd.to_numeric(ownership.get('compound_bubble_id_link'), errors='coerce')
#     standalone = units.loc[units['compound_bubble_role'].astype(str).eq('standalone') & ~units['is_compound'].map(as_bool)].copy()
#     standalone_ids = set(standalone['unit_id'].astype(int))
#     ownership['standalone_unit_candidates'] = ownership['unit_ids'].map(parse_ids).map(lambda ids: sorted(ids & standalone_ids))
#     ownership['ownership_kind'] = np.select(
#         [
#             ownership['compound_bubble_id_link'].notna(),
#             ownership.get('bubble_slope_qc_reason', pd.Series('', index=ownership.index)).astype(str).eq('ambiguous_link_assignment'),
#             ownership['standalone_unit_candidates'].map(len).eq(1),
#             ownership['standalone_unit_candidates'].map(len).gt(1),
#             ownership['unit_ids'].map(parse_ids).map(len).eq(0),
#         ],
#         ['compound_bubble', 'ambiguous_compound_membership', 'simple_unit', 'ambiguous_simple_membership', 'unassigned_connector_or_boundary'],
#         default='not_owned_by_base_refit_domain',
#     )
#     ownership['ownership_domain_id'] = pd.NA
#     bubble_mask = ownership['ownership_kind'].eq('compound_bubble')
#     ownership.loc[bubble_mask, 'ownership_domain_id'] = ownership.loc[bubble_mask, 'compound_bubble_id_link'].astype(int).astype(str)
#     simple_mask = ownership['ownership_kind'].eq('simple_unit')
#     ownership.loc[simple_mask, 'ownership_domain_id'] = ownership.loc[simple_mask, 'standalone_unit_candidates'].map(lambda values: str(values[0]))
#     ownership['example_id'] = data['example_id']

#     domains = []
#     standalone_by_id = standalone.set_index('unit_id')
#     for unit_id in sorted(pd.to_numeric(ownership.loc[ownership['ownership_kind'].eq('simple_unit'), 'ownership_domain_id'], errors='coerce').dropna().astype(int).unique()):
#         row = standalone_by_id.loc[int(unit_id)]
#         selected_ids = set(ownership.loc[(ownership['ownership_kind'].eq('simple_unit')) & ownership['ownership_domain_id'].eq(str(unit_id)), 'id_link'].astype(int))
#         domains.append({'domain_kind': 'simple_unit', 'domain_id': int(unit_id), 'source': int(row.bifurcation), 'sink': int(row.confluence), 'links': local_links.loc[local_links['id_link'].isin(selected_ids)].copy()})
#     bubbles_by_id = bubbles.set_index('compound_bubble_id')
#     for bubble_id in sorted(pd.to_numeric(ownership.loc[ownership['ownership_kind'].eq('compound_bubble'), 'ownership_domain_id'], errors='coerce').dropna().astype(int).unique()):
#         if int(bubble_id) not in bubbles_by_id.index:
#             ownership.loc[(ownership['ownership_kind'].eq('compound_bubble')) & ownership['ownership_domain_id'].eq(str(bubble_id)), 'ownership_kind'] = 'compound_owner_missing_boundary_record'
#             continue
#         row = bubbles_by_id.loc[int(bubble_id)]
#         selected_ids = set(ownership.loc[(ownership['ownership_kind'].eq('compound_bubble')) & ownership['ownership_domain_id'].eq(str(bubble_id)), 'id_link'].astype(int))
#         domains.append({'domain_kind': 'compound_bubble', 'domain_id': int(bubble_id), 'source': int(row.upstream_boundary_node), 'sink': int(row.downstream_boundary_node), 'links': local_links.loc[local_links['id_link'].isin(selected_ids)].copy()})

#     results, failures = [], []
#     for domain in domains:
#         label = f"{domain['domain_kind']}:{domain['domain_id']}"
#         selected, source, sink = domain['links'], domain['source'], domain['sink']
#         try:
#             if selected.empty:
#                 raise ValueError('no realised links')
#             if source not in heads.index or sink not in heads.index:
#                 raise ValueError('one or both boundary heads are not accepted reference anchors')
#             reference_distance = float(distances[source] - distances[sink])
#             candidate, audit, _ = solve_boundary_conditioned_domain_fast(
#                 selected.assign(actual_length_m=preferred_length(selected)), source, sink,
#                 float(heads[source]), float(heads[sink]), reference_distance,
#             )
#             candidate['example_id'] = data['example_id']
#             candidate['domain_kind'] = domain['domain_kind']; candidate['domain_id'] = domain['domain_id']
#             candidate['domain_label'] = label
#             candidate['domain_max_path_residual_m'] = float(audit.iloc[0]['max_abs_path_residual_m'])
#             candidate['domain_local_reference_slope'] = float(audit.iloc[0]['local_reference_slope'])
#             # The full-study table deliberately has no geometry: examples use
#             # different projected CRSs and visual review remains per-example.
#             results.append(pd.DataFrame(candidate.drop(columns=[candidate.geometry.name], errors='ignore')))
#         except Exception as exc:
#             failures.append({'example_id': data['example_id'], 'domain_label': label, 'domain_kind': domain['domain_kind'], 'domain_id': domain['domain_id'], 'error': f'{type(exc).__name__}: {exc}'})
#     return (pd.concat(results, ignore_index=True) if results else pd.DataFrame()), pd.DataFrame(failures), ownership


# def stored_rapid_config(experiment_dir: Path) -> dict:
#     manifest = json.loads((experiment_dir / 'rapid_prep_manifest.json').read_text())
#     config = dict(manifest.get('prep_config', {}))
#     config['kb_value'] = float(manifest.get('resolved_kb_summary', {}).get('kb_value', config.get('kb_value', 20.0)))
#     return config


# def add_link_level_rapid_impact(candidate: pd.DataFrame, data: dict) -> pd.DataFrame:
#     context_fields = [c for c in ['id_link', 'slope_used'] if c in data['context'].columns]
#     frame = candidate.merge(data['context'][context_fields], on='id_link', how='left', validate='m:1')
#     frame['current_slope_used'] = pd.to_numeric(frame['slope_used'], errors='coerce')
#     frame['slope_ratio_candidate_to_current'] = frame['candidate_slope'] / frame['current_slope_used']
#     frame['raw_celerity_ratio_candidate_to_current'] = np.sqrt(frame['slope_ratio_candidate_to_current'])
#     frame['raw_k_ratio_candidate_to_current'] = 1.0 / frame['raw_celerity_ratio_candidate_to_current']
#     try:
#         config = stored_rapid_config(data['experiment_dir'])
#     except FileNotFoundError:
#         # Direction-corrected examples not yet prepared for RAPID still belong
#         # in the slope-consistency screen. Retain their candidate slopes and
#         # make the unavailable routing translation explicit rather than dropping
#         # the entire example.
#         frame['rapid_parameter_translation_status'] = 'rapid_prep_manifest_unavailable'
#         for column in [
#             'approx_current_celerity_mps_raw', 'approx_candidate_celerity_mps_raw',
#             'approx_current_celerity_mps_capped', 'approx_candidate_celerity_mps_capped',
#             'approx_capped_k_ratio_candidate_to_current',
#         ]:
#             frame[column] = np.nan
#         frame['approx_current_celerity_cap_status'] = 'rapid_config_unavailable'
#         frame['approx_candidate_celerity_cap_status'] = 'rapid_config_unavailable'
#         frame['approx_celerity_cap_transition'] = 'rapid_config_unavailable'
#         frame['approx_capped_celerity_changed'] = pd.NA
#         return frame
#     frame['rapid_parameter_translation_status'] = 'computed_from_stored_rapid_prep_manifest'
#     width_field = str(config.get('width_field', 'wid_adj_routing'))
#     if width_field not in frame.columns:
#         raise ValueError(f'Configured RAPID width field is absent from link data: {width_field}')
#     width = pd.to_numeric(frame[width_field], errors='coerce').clip(lower=float(config.get('min_width', 1.0)))
#     n_value, kb_value = float(config.get('n_manning', 0.35)), float(config['kb_value'])
#     c_factor = (5.0 / (3.0 * n_value)) * (width ** (2.0 / 3.0) / kb_value ** (2.0 / 3.0))
#     frame['approx_current_celerity_mps_raw'] = c_factor * np.sqrt(frame['current_slope_used'])
#     frame['approx_candidate_celerity_mps_raw'] = c_factor * np.sqrt(frame['candidate_slope'])
#     if bool(config.get('use_celerity_capping', False)):
#         lower, upper = float(config['min_celerity_mps']), float(config['max_celerity_mps'])
#         frame['approx_current_celerity_cap_status'] = np.select(
#             [frame['approx_current_celerity_mps_raw'].lt(lower), frame['approx_current_celerity_mps_raw'].gt(upper)],
#             ['below_min_cap', 'above_max_cap'], default='within_cap_range',
#         )
#         frame['approx_candidate_celerity_cap_status'] = np.select(
#             [frame['approx_candidate_celerity_mps_raw'].lt(lower), frame['approx_candidate_celerity_mps_raw'].gt(upper)],
#             ['below_min_cap', 'above_max_cap'], default='within_cap_range',
#         )
#         frame['approx_current_celerity_mps_capped'] = frame['approx_current_celerity_mps_raw'].clip(lower=lower, upper=upper)
#         frame['approx_candidate_celerity_mps_capped'] = frame['approx_candidate_celerity_mps_raw'].clip(lower=lower, upper=upper)
#     else:
#         frame['approx_current_celerity_cap_status'] = 'cap_disabled'
#         frame['approx_candidate_celerity_cap_status'] = 'cap_disabled'
#         frame['approx_current_celerity_mps_capped'] = frame['approx_current_celerity_mps_raw']
#         frame['approx_candidate_celerity_mps_capped'] = frame['approx_candidate_celerity_mps_raw']
#     frame['approx_celerity_cap_transition'] = frame['approx_current_celerity_cap_status'] + ' -> ' + frame['approx_candidate_celerity_cap_status']
#     frame['approx_capped_k_ratio_candidate_to_current'] = frame['approx_current_celerity_mps_capped'] / frame['approx_candidate_celerity_mps_capped']
#     # This records a change in the *capped celerity value*, not a transition
#     # between cap categories. A separate cap-category audit can be added after
#     # subreach widths are incorporated.
#     frame['approx_capped_celerity_changed'] = ~np.isclose(frame['approx_current_celerity_mps_capped'], frame['approx_candidate_celerity_mps_capped'], rtol=1e-12, atol=1e-12)
#     return frame


# def solve_boundary_conditioned_domain_fast(selected_links, source, sink, source_head, sink_head, reference_distance_m):
#     """Exact unconstrained quadratic solution, retained only when it is positive.

#     It has the identical target to the displayed constrained QP. When the
#     unconstrained optimum obeys every positive-drop constraint, it is also the
#     constrained optimum. Other domains are reported for constrained treatment;
#     the batch diagnostic never approximates them.
#     """
#     graph = nx.MultiDiGraph()
#     for row in selected_links.itertuples(index=False):
#         graph.add_edge(int(row.id_us_node), int(row.id_ds_node), key=int(row.id_link), length_m=float(row.actual_length_m))
#     simple_graph = nx.DiGraph(graph)
#     if not nx.is_directed_acyclic_graph(simple_graph) or not nx.has_path(simple_graph, source, sink):
#         raise ValueError('Selected domain must be an acyclic connected source-to-sink graph.')
#     total_drop = float(source_head - sink_head)
#     if total_drop <= 0 or reference_distance_m <= 0:
#         raise ValueError('Boundary heads and reference distance must define a positive local slope.')
#     internal = sorted(set(graph.nodes).difference({source, sink}))
#     index = {node: i for i, node in enumerate(internal)}
#     lengths = selected_links['actual_length_m'].to_numpy(float)
#     fixed, incidence = np.zeros(len(selected_links)), np.zeros((len(selected_links), len(internal)))
#     for edge_index, row in enumerate(selected_links.itertuples(index=False)):
#         for node, sign in ((int(row.id_us_node), 1.0), (int(row.id_ds_node), -1.0)):
#             if node in index:
#                 incidence[edge_index, index[node]] += sign
#             else:
#                 fixed[edge_index] += sign * (source_head if node == source else sink_head)
#     s_ref = total_drop / reference_distance_m
#     floor = min(NUMERICAL_SLOPE_FLOOR, 0.1 * s_ref)
#     if internal:
#         normal = incidence.T @ (incidence / lengths[:, None])
#         rhs = incidence.T @ ((s_ref * lengths - fixed) / lengths)
#         try:
#             solved = np.linalg.solve(normal, rhs)
#         except np.linalg.LinAlgError:
#             solved = np.linalg.lstsq(normal, rhs, rcond=None)[0]
#     else:
#         solved = np.asarray([])
#     drops = fixed + incidence @ solved
#     if np.any(drops < floor * lengths):
#         raise ValueError('unconstrained quadratic optimum violates positive-drop constraints; requires constrained QP')
#     heads = {source: source_head, sink: sink_head, **{node: float(solved[i]) for node, i in index.items()}}
#     candidate = selected_links.copy()
#     candidate['head_us_m'] = candidate['id_us_node'].map(heads); candidate['head_ds_m'] = candidate['id_ds_node'].map(heads)
#     candidate['candidate_slope'] = (candidate['head_us_m'] - candidate['head_ds_m']) / candidate['actual_length_m']
#     drops_by_link = dict(zip(candidate['id_link'].astype(int), candidate['candidate_slope'] * candidate['actual_length_m']))
#     audit = path_drop_audit(graph, drops_by_link, source, sink)
#     objective = float(np.sum(lengths * ((drops / lengths - s_ref) / s_ref) ** 2))
#     audit.update({'boundary_head_loss_m': total_drop, 'local_reference_slope': s_ref, 'max_abs_path_residual_m': max(abs(audit['minimum_path_drop_m'] - total_drop), abs(audit['maximum_path_drop_m'] - total_drop)), 'objective_value': objective, 'numerical_slope_floor': floor, 'optimizer_message': 'fast_unconstrained_qp_positive'})
#     return candidate, pd.DataFrame([audit]), heads


# impact_frames, impact_failures, coverage_rows, ownership_frames = [], [], [], []
# for example_id in BASE_IMPACT_EXAMPLE_IDS:
#     try:
#         data = canonical_base_inputs(example_id)
#         heads, distances, node_frame = base_reference_heads(data)
#         candidate, failures, ownership = proposed_base_domains(data, heads, distances)
#         ownership_frames.append(ownership)
#         if not failures.empty:
#             impact_failures.append(failures)
#         if candidate.empty:
#             raise RuntimeError('no base domains could be solved')
#         impact = add_link_level_rapid_impact(candidate, data)
#         if impact.duplicated('id_link').any():
#             raise RuntimeError('Canonical ownership unexpectedly produced duplicate candidate slopes.')
#         all_links = set(data['links']['id_link'].astype(int))
#         solved_links = set(impact['id_link'].astype(int))
#         ownership_counts = ownership['ownership_kind'].value_counts()
#         eligible = ownership.loc[ownership['ownership_kind'].isin(['simple_unit', 'compound_bubble']), 'id_link'].astype(int)
#         coverage_rows.append({
#             'example_id': example_id, 'n_base_links': len(all_links), 'n_proposed_scope_links': len(solved_links),
#             'n_eligible_owned_links': int(eligible.nunique()), 'n_eligible_but_unsolved_links': int(len(set(eligible) - solved_links)),
#             'n_outside_scope_links': len(all_links - solved_links),
#             'n_compound_bubble_owned_links': int(ownership_counts.get('compound_bubble', 0)),
#             'n_simple_unit_owned_links': int(ownership_counts.get('simple_unit', 0)),
#             'n_unassigned_connector_or_boundary_links': int(ownership_counts.get('unassigned_connector_or_boundary', 0)),
#             'n_ambiguous_membership_links': int(ownership_counts.get('ambiguous_compound_membership', 0) + ownership_counts.get('ambiguous_simple_membership', 0)),
#             'n_accepted_reference_anchors': int(node_frame['accepted_reference_anchor'].sum()),
#         })
#         impact_frames.append(impact)
#     except Exception as exc:
#         impact_failures.append(pd.DataFrame([{'example_id': example_id, 'domain_label': pd.NA, 'domain_kind': pd.NA, 'domain_id': pd.NA, 'error': f'{type(exc).__name__}: {exc}'}]))

# base_slope_impact = pd.concat(impact_frames, ignore_index=True) if impact_frames else pd.DataFrame()
# base_slope_impact_failures = pd.concat(impact_failures, ignore_index=True) if impact_failures else pd.DataFrame()
# base_slope_impact_coverage = pd.DataFrame(coverage_rows)
# base_slope_link_ownership = pd.concat(ownership_frames, ignore_index=True) if ownership_frames else pd.DataFrame()
# if base_slope_impact.empty:
#     raise RuntimeError('The all-base impact screen did not solve any domains.')

# # Full slope-consistency assessment is available even where a direction-corrected
# # experiment has not yet received a RAPID preparation manifest.
# finite_slope = base_slope_impact.replace([np.inf, -np.inf], np.nan).dropna(
#     subset=['slope_ratio_candidate_to_current', 'current_slope_used', 'candidate_slope', 'domain_local_reference_slope']
# ).copy()
# finite_slope['current_abs_log_local_reference_residual'] = np.abs(np.log(finite_slope['current_slope_used'] / finite_slope['domain_local_reference_slope']))
# finite_slope['candidate_abs_log_local_reference_residual'] = np.abs(np.log(finite_slope['candidate_slope'] / finite_slope['domain_local_reference_slope']))
# finite_slope['candidate_closer_to_local_reference'] = finite_slope['candidate_abs_log_local_reference_residual'].lt(finite_slope['current_abs_log_local_reference_residual'])
# slope_consistency_by_example = finite_slope.groupby('example_id', as_index=False).agg(
#     n_links=('id_link', 'size'),
#     current_median_abs_log_local_reference_residual=('current_abs_log_local_reference_residual', 'median'),
#     candidate_median_abs_log_local_reference_residual=('candidate_abs_log_local_reference_residual', 'median'),
#     current_p95_abs_log_local_reference_residual=('current_abs_log_local_reference_residual', lambda x: x.quantile(.95)),
#     candidate_p95_abs_log_local_reference_residual=('candidate_abs_log_local_reference_residual', lambda x: x.quantile(.95)),
#     pct_links_closer_to_local_reference=('candidate_closer_to_local_reference', lambda x: 100 * x.mean()),
#     candidate_max_path_residual_m=('domain_max_path_residual_m', 'max'),
# )
# overall_slope_consistency = pd.DataFrame([{
#     'n_examples_with_slope_results': int(finite_slope['example_id'].nunique()),
#     'n_links': len(finite_slope),
#     'current_median_abs_log_local_reference_residual': finite_slope['current_abs_log_local_reference_residual'].median(),
#     'candidate_median_abs_log_local_reference_residual': finite_slope['candidate_abs_log_local_reference_residual'].median(),
#     'current_p95_abs_log_local_reference_residual': finite_slope['current_abs_log_local_reference_residual'].quantile(.95),
#     'candidate_p95_abs_log_local_reference_residual': finite_slope['candidate_abs_log_local_reference_residual'].quantile(.95),
#     'pct_links_closer_to_local_reference': 100 * finite_slope['candidate_closer_to_local_reference'].mean(),
#     'max_candidate_path_residual_m': finite_slope['domain_max_path_residual_m'].max(),
#     'expected_equivalent_resistance_ratio': 1.0,
# }])

# # A "neighbourhood" comparison is meaningful only at a strictly serial node;
# # braiding/junction nodes deliberately have no expected equal-slope relation.
# serial_rows = []
# for domain_key, group in finite_slope.groupby(['example_id', 'domain_kind', 'domain_id'], sort=False):
#     incoming = {int(node): frame for node, frame in group.groupby('id_ds_node', sort=False)}
#     outgoing = {int(node): frame for node, frame in group.groupby('id_us_node', sort=False)}
#     for node in sorted(set(incoming) & set(outgoing)):
#         if len(incoming[node]) != 1 or len(outgoing[node]) != 1:
#             continue
#         left, right = incoming[node].iloc[0], outgoing[node].iloc[0]
#         serial_rows.append({
#             'example_id': domain_key[0], 'domain_kind': domain_key[1], 'domain_id': domain_key[2], 'serial_node_id': node,
#             'current_abs_log_slope_jump': abs(np.log(float(left.current_slope_used) / float(right.current_slope_used))),
#             'candidate_abs_log_slope_jump': abs(np.log(float(left.candidate_slope) / float(right.candidate_slope))),
#         })
# serial_neighbourhood_audit = pd.DataFrame(serial_rows)
# if not serial_neighbourhood_audit.empty:
#     serial_neighbourhood_audit['candidate_smoother_at_serial_node'] = serial_neighbourhood_audit['candidate_abs_log_slope_jump'].lt(serial_neighbourhood_audit['current_abs_log_slope_jump'])
#     serial_neighbourhood_summary = pd.DataFrame([{
#         'n_serial_nodes': len(serial_neighbourhood_audit),
#         'current_median_abs_log_slope_jump': serial_neighbourhood_audit['current_abs_log_slope_jump'].median(),
#         'candidate_median_abs_log_slope_jump': serial_neighbourhood_audit['candidate_abs_log_slope_jump'].median(),
#         'current_p95_abs_log_slope_jump': serial_neighbourhood_audit['current_abs_log_slope_jump'].quantile(.95),
#         'candidate_p95_abs_log_slope_jump': serial_neighbourhood_audit['candidate_abs_log_slope_jump'].quantile(.95),
#         'pct_serial_nodes_smoother': 100 * serial_neighbourhood_audit['candidate_smoother_at_serial_node'].mean(),
#     }])
# else:
#     serial_neighbourhood_summary = pd.DataFrame()

# # This is the visual-review queue. Pick a row's domain kind/id in section 1
# # to inspect its exact network and current/candidate slopes in Plotly.
# visual_review_queue = finite_slope.assign(
#     abs_log_slope_change=np.abs(np.log(finite_slope['slope_ratio_candidate_to_current'])),
#     local_reference_residual_change=(finite_slope['candidate_abs_log_local_reference_residual'] - finite_slope['current_abs_log_local_reference_residual']),
# ).sort_values(['abs_log_slope_change', 'local_reference_residual_change'], ascending=[False, False]).groupby('example_id', as_index=False, group_keys=False).head(5)

# finite = base_slope_impact.replace([np.inf, -np.inf], np.nan).dropna(subset=['slope_ratio_candidate_to_current', 'approx_capped_k_ratio_candidate_to_current'])
# impact_summary = finite.groupby('example_id', as_index=False).agg(
#     n_links=('id_link', 'size'),
#     median_slope_ratio=('slope_ratio_candidate_to_current', 'median'),
#     p05_slope_ratio=('slope_ratio_candidate_to_current', lambda x: x.quantile(.05)),
#     p95_slope_ratio=('slope_ratio_candidate_to_current', lambda x: x.quantile(.95)),
#     pct_slope_change_over_25pct=('slope_ratio_candidate_to_current', lambda x: 100 * ((x < .8) | (x > 1.25)).mean()),
#     median_uncapped_k_ratio=('raw_k_ratio_candidate_to_current', 'median'),
#     p05_uncapped_k_ratio=('raw_k_ratio_candidate_to_current', lambda x: x.quantile(.05)),
#     p95_uncapped_k_ratio=('raw_k_ratio_candidate_to_current', lambda x: x.quantile(.95)),
#     median_capped_k_ratio=('approx_capped_k_ratio_candidate_to_current', 'median'),
#     pct_capped_k_change_over_25pct=('approx_capped_k_ratio_candidate_to_current', lambda x: 100 * ((x < .8) | (x > 1.25)).mean()),
#     pct_capped_celerity_changed=('approx_capped_celerity_changed', lambda x: 100 * x.mean()),
#     n_current_below_min_cap=('approx_current_celerity_cap_status', lambda x: int(x.eq('below_min_cap').sum())),
#     n_candidate_below_min_cap=('approx_candidate_celerity_cap_status', lambda x: int(x.eq('below_min_cap').sum())),
#     n_current_above_max_cap=('approx_current_celerity_cap_status', lambda x: int(x.eq('above_max_cap').sum())),
#     n_candidate_above_max_cap=('approx_candidate_celerity_cap_status', lambda x: int(x.eq('above_max_cap').sum())),
#     n_left_min_cap=('approx_celerity_cap_transition', lambda x: int(x.str.startswith('below_min_cap ->').sum() - x.eq('below_min_cap -> below_min_cap').sum())),
#     n_entered_min_cap=('approx_celerity_cap_transition', lambda x: int(x.str.endswith('-> below_min_cap').sum() - x.eq('below_min_cap -> below_min_cap').sum())),
#     n_left_max_cap=('approx_celerity_cap_transition', lambda x: int(x.str.startswith('above_max_cap ->').sum() - x.eq('above_max_cap -> above_max_cap').sum())),
#     n_entered_max_cap=('approx_celerity_cap_transition', lambda x: int(x.str.endswith('-> above_max_cap').sum() - x.eq('above_max_cap -> above_max_cap').sum())),
#     max_path_residual_m=('domain_max_path_residual_m', 'max'),
# )
# overall_impact_summary = pd.DataFrame([{
#     'n_examples_completed': int(finite['example_id'].nunique()), 'n_links_in_proposed_scope': len(finite),
#     'median_slope_ratio': finite['slope_ratio_candidate_to_current'].median(),
#     'p01_slope_ratio': finite['slope_ratio_candidate_to_current'].quantile(.01),
#     'p99_slope_ratio': finite['slope_ratio_candidate_to_current'].quantile(.99),
#     'pct_slope_change_over_25pct': 100 * ((finite['slope_ratio_candidate_to_current'] < .8) | (finite['slope_ratio_candidate_to_current'] > 1.25)).mean(),
#     'median_uncapped_k_ratio': finite['raw_k_ratio_candidate_to_current'].median(),
#     'p01_uncapped_k_ratio': finite['raw_k_ratio_candidate_to_current'].quantile(.01),
#     'p99_uncapped_k_ratio': finite['raw_k_ratio_candidate_to_current'].quantile(.99),
#     'median_capped_k_ratio': finite['approx_capped_k_ratio_candidate_to_current'].median(),
#     'p01_capped_k_ratio': finite['approx_capped_k_ratio_candidate_to_current'].quantile(.01),
#     'p99_capped_k_ratio': finite['approx_capped_k_ratio_candidate_to_current'].quantile(.99),
#     'pct_capped_celerity_changed': 100 * finite['approx_capped_celerity_changed'].mean(),
#     'n_current_below_min_cap': int(finite['approx_current_celerity_cap_status'].eq('below_min_cap').sum()),
#     'n_candidate_below_min_cap': int(finite['approx_candidate_celerity_cap_status'].eq('below_min_cap').sum()),
#     'n_current_above_max_cap': int(finite['approx_current_celerity_cap_status'].eq('above_max_cap').sum()),
#     'n_candidate_above_max_cap': int(finite['approx_candidate_celerity_cap_status'].eq('above_max_cap').sum()),
#     'n_left_min_cap': int(((finite['approx_current_celerity_cap_status'].eq('below_min_cap')) & ~finite['approx_candidate_celerity_cap_status'].eq('below_min_cap')).sum()),
#     'n_entered_min_cap': int((~finite['approx_current_celerity_cap_status'].eq('below_min_cap') & finite['approx_candidate_celerity_cap_status'].eq('below_min_cap')).sum()),
#     'n_left_max_cap': int(((finite['approx_current_celerity_cap_status'].eq('above_max_cap')) & ~finite['approx_candidate_celerity_cap_status'].eq('above_max_cap')).sum()),
#     'n_entered_max_cap': int((~finite['approx_current_celerity_cap_status'].eq('above_max_cap') & finite['approx_candidate_celerity_cap_status'].eq('above_max_cap')).sum()),
#     'max_path_residual_m': finite['domain_max_path_residual_m'].max(),
#     'expected_equivalent_resistance_ratio': 1.0,
# }])

# print('All eligible base-domain slope-consistency assessment (includes examples without RAPID prep manifests):')
# display(overall_slope_consistency)
# display(slope_consistency_by_example.sort_values('example_id'))
# display(serial_neighbourhood_summary)
# print('Highest-priority visual review rows; set their domain kind/id in section 1:')
# display(visual_review_queue[['example_id', 'domain_kind', 'domain_id', 'id_link', 'slope_ratio_candidate_to_current', 'current_abs_log_local_reference_residual', 'candidate_abs_log_local_reference_residual', 'domain_max_path_residual_m']])
# print('RAPID-parameter translation subset (only examples with a stored rapid_prep_manifest):')
# display(overall_impact_summary)
# display(base_slope_impact_coverage.sort_values('example_id'))
# display(impact_summary.sort_values('example_id'))
# cap_transition_summary = finite.groupby(['example_id', 'approx_celerity_cap_transition']).size().rename('n_links').reset_index()
# print('Approximate link-level celerity-cap transitions (from raw celerity and each stored example cap):')
# display(cap_transition_summary.sort_values(['example_id', 'approx_celerity_cap_transition']))
# display(base_slope_link_ownership.groupby(['example_id', 'ownership_kind']).size().rename('n_links').reset_index().sort_values(['example_id', 'ownership_kind']))
# if not base_slope_impact_failures.empty:
#     print('Domains excluded from the proposed-slope impact scope (nothing was modified):')
#     display(base_slope_impact_failures)

# # Two compact diagnostic plots: distributional stability and potential routing response.
# fig = go.Figure()
# fig.add_histogram(x=np.log10(finite_slope['current_slope_used']), name='current final slope', opacity=.60, nbinsx=70)
# fig.add_histogram(x=np.log10(finite_slope['candidate_slope']), name='proposed candidate slope', opacity=.60, nbinsx=70)
# fig.update_layout(barmode='overlay', title='All-base proposed-scope slope distribution', xaxis_title='log10(slope)', yaxis_title='link count')
# fig.show()

# fig = go.Figure()
# fig.add_scatter(
#     x=finite['slope_ratio_candidate_to_current'], y=finite['approx_capped_k_ratio_candidate_to_current'],
#     mode='markers', text=finite['example_id'].astype(str) + ':' + finite['id_link'].astype(str),
#     marker={'size': 6, 'opacity': .55, 'color': finite['domain_kind'].eq('compound_bubble').astype(int), 'colorscale': [[0, '#4C78A8'], [1, '#E45756']], 'showscale': False},
#     hovertemplate='%{text}<br>slope ratio=%{x:.4g}<br>approx capped-K ratio=%{y:.4g}<extra></extra>', name='link',
# )
# fig.add_hline(y=1.0, line_dash='dash', line_color='black'); fig.add_vline(x=1.0, line_dash='dash', line_color='black')
# fig.update_layout(title='All-base slope change translated to approximate capped RAPID K change', xaxis={'type': 'log', 'title': 'candidate/current slope'}, yaxis={'type': 'log', 'title': 'approximate candidate/current K'})
# fig.show()


## 8. Child-collapse solver: retained boundary-conditioned formulation

No child-collapse slope logic is changed by this notebook. The working
`collapsed_unit_path_relative_solver` remains the production method: it fixes all
inherited boundary heads, uses each boundary-pair/path target
\(S_{p}=\Delta H_p/L_p\), and constrains positive directed drops.

The future shared solver interface must reproduce that result before any refactor. The
only special branch left outside this scope is the existing one-hop parent-interface
conflict solver, which handles reversed or otherwise inconsistent parent matching.


In [ ]:
child_solver_columns = [
    'id_link', 'slope_used', 'slope_source_method', 'collapsed_profile_repair_applied',
    'path_relative_target_paths', 'path_relative_target_slopes',
    'path_relative_objective_before', 'path_relative_objective_after',
]
display(link_context[[column for column in child_solver_columns if column in link_context.columns]].head(20))
print('This cell is provenance-only: no child-state slope values are recomputed here.')


## 9. Decision and promotion record

The staged candidate workflow passed all required acceptance checks: every promoted base
link has a positive `len`-based slope; reconstructed link heads reproduce the stored
slope; all 440 solved domains meet their prescribed path head loss; and the explicit
\(0.01S_{\rm ref}\) domain constraint has no violations. Four Example 15 links are
recorded as intentional physical-floor-bound links, not hidden numerical artefacts.

On 2026-10-01 the validated base contexts for all 16 examples were promoted with
recoverable backups and immutable reference-profile artifacts. The promotion changes
only base hydraulic contexts. Required downstream work is to refresh child hydraulic
contexts, recompute equivalent resistance, and prepare/run RAPID; topology, masks,
directions, widths, and collapse geometries are unchanged.

In [ ]:
# Focused review of flagged simple connectors. Run the frozen-profile connector
# calibration cell first; this cell only reads its in-memory results.
CONNECTOR_REVIEW_CASES = [
    ('22', 'C001'),  # long, floor-controlled candidate
    ('11', 'C005'),
    ('46', 'C049'),
    ('46', 'C003'),
]
if 'frozen_connector_calibration' not in globals() or 'frozen_reference_profiles' not in globals():
    raise RuntimeError('Run the frozen-profile connector calibration cell first.')

connector_review_rows = []
for review_example_id, review_component_id in CONNECTOR_REVIEW_CASES:
    review_record = frozen_connector_calibration.loc[
        frozen_connector_calibration['example_id'].astype(str).eq(review_example_id)
        & frozen_connector_calibration['connector_component_id'].eq(review_component_id)
    ]
    if len(review_record) != 1:
        raise RuntimeError(f'Could not resolve {review_example_id}/{review_component_id}.')
    review_record = review_record.iloc[0]
    review_exp_dir = OUTPUT_ROOT / f'{review_example_id}_indep__anchored'
    review_state_dir = review_exp_dir / 'states' / 'S000_base'
    review_links = gpd.read_file(one_path(review_state_dir / 'variant' / 'directed', '*directed_links.gpkg'))
    review_nodes = gpd.read_file(one_path(review_state_dir / 'variant' / 'directed', '*directed_nodes.gpkg'))
    review_context = pd.read_csv(review_state_dir / 'hydraulic_context' / 'link_hydraulic_context.csv')
    review_support = _load_reference_section_support(
        str(review_exp_dir / 'reference_section' / 'reference_section_summary.json')
    )
    review_distance, _ = _project_nodes_to_reference_section(
        review_nodes, support=review_support,
        config=SlopeConfig(reference_section_summary_path=str(
            review_exp_dir / 'reference_section' / 'reference_section_summary.json'
        )),
    )
    review_node_frame = review_nodes[['id_node']].copy()
    review_node_frame['reference_section_dist_out_m'] = review_distance
    review_profile = frozen_reference_profiles.loc[
        frozen_reference_profiles['example_id'].astype(str).eq(review_example_id),
        ['reference_section_dist_out_m', 'joint_monotone_wse_m'],
    ].sort_values('reference_section_dist_out_m')
    review_node_frame['profile_wse_m'] = np.interp(
        review_node_frame['reference_section_dist_out_m'],
        review_profile['reference_section_dist_out_m'],
        review_profile['joint_monotone_wse_m'],
    )
    review_nodes_by_id = review_node_frame.set_index('id_node')
    review_anchor_ids = [
        int(review_record['outer_upstream_anchor']),
        int(review_record['connector_upstream_node']),
        int(review_record['connector_downstream_node']),
        int(review_record['outer_downstream_anchor']),
    ]
    review_anchor_labels = ['U: outer upstream', 'A: connector upstream', 'B: connector downstream', 'D: outer downstream']
    review_anchor_frame = review_nodes_by_id.loc[review_anchor_ids].copy()
    review_anchor_frame['label'] = review_anchor_labels
    review_link_id = int(str(review_record['component_link_ids']).split(',')[0])
    review_current = review_context.loc[review_context['id_link'].eq(review_link_id)]
    review_current_slope = float(review_current['slope_used'].iloc[0]) if 'slope_used' in review_current.columns and len(review_current) == 1 else np.nan
    review_raw_slope = float(review_current['raw_slope'].iloc[0]) if 'raw_slope' in review_current.columns and len(review_current) == 1 else np.nan
    actual_drop = float(review_record['profile_connector_slope'] * review_record['connector_length_m'])
    target_drop = float(review_record['support_target_slope'] * review_record['connector_length_m'])
    connector_review_rows.append({
        'example_id': review_example_id, 'connector_component_id': review_component_id,
        'link_id': review_link_id, 'connector_length_km': review_record['connector_length_m'] / 1_000.0,
        'profile_connector_slope_m_per_m': review_record['profile_connector_slope'],
        'support_target_slope_m_per_m': review_record['support_target_slope'],
        'factor_from_target': review_record['factor_from_target'],
        'actual_profile_drop_m': actual_drop, 'support_target_drop_m': target_drop,
        'additional_drop_needed_m': target_drop - actual_drop,
        'current_slope_used_m_per_m': review_current_slope,
        'current_raw_slope_m_per_m': review_raw_slope,
    })

    x_min = review_anchor_frame['reference_section_dist_out_m'].min() - 0.10 * max(
        review_anchor_frame['reference_section_dist_out_m'].max()
        - review_anchor_frame['reference_section_dist_out_m'].min(), 5_000.0
    )
    x_max = review_anchor_frame['reference_section_dist_out_m'].max() + 0.10 * max(
        review_anchor_frame['reference_section_dist_out_m'].max()
        - review_anchor_frame['reference_section_dist_out_m'].min(), 5_000.0
    )
    local_profile = review_profile.loc[
        review_profile['reference_section_dist_out_m'].between(x_min, x_max)
    ]
    fig = go.Figure()
    fig.add_scatter(
        x=local_profile['reference_section_dist_out_m'] / 1_000.0,
        y=local_profile['joint_monotone_wse_m'], mode='lines',
        name='frozen reference profile', line={'color': '#1F77B4', 'width': 3},
    )
    fig.add_scatter(
        x=review_anchor_frame['reference_section_dist_out_m'] / 1_000.0,
        y=review_anchor_frame['profile_wse_m'], mode='markers+text',
        text=review_anchor_frame['label'], textposition='top center',
        name='connector/support nodes', marker={'size': 9, 'color': '#F58518'},
    )
    a_x, b_x = review_anchor_frame['reference_section_dist_out_m'].iloc[1:3] / 1_000.0
    a_h = review_anchor_frame['profile_wse_m'].iloc[1]
    fig.add_scatter(
        x=[a_x, b_x], y=[a_h, a_h - target_drop], mode='lines',
        name='connector if it matched support target',
        line={'color': '#E45756', 'width': 2, 'dash': 'dash'},
    )
    fig.update_layout(
        title=(f'Example {review_example_id} / {review_component_id}: factor='
               f'{review_record["factor_from_target"]:.2f}, length='
               f'{review_record["connector_length_m"] / 1_000.0:.2f} km'),
        xaxis_title='Distance to outlet [km]', yaxis_title='Frozen-profile WSE [m]',
        height=520, legend={'orientation': 'h', 'y': -0.22},
    )
    fig.show()

connector_case_review = pd.DataFrame(connector_review_rows)
display(connector_case_review)


In [ ]:
# Physical-window audit for all genuine factor-five connector flags.
# Read-only: excludes the 32 network endpoint links and does not refit profiles.
from rapid_tools.connector_support_windows import (
    DEFAULT_SUPPORT_WINDOWS_M, audit_connector_physical_support,
)

if 'frozen_connector_calibration' not in globals() or 'frozen_reference_profiles' not in globals():
    raise RuntimeError('Run the frozen-profile connector calibration cell first.')

PHYSICAL_SUPPORT_WINDOWS_M = DEFAULT_SUPPORT_WINDOWS_M  # 1, 2, and 5 km on each side
PHYSICAL_SUPPORT_FACTOR_LIMIT = 5.0
window_flagged_connectors = frozen_connector_calibration.loc[
    frozen_connector_calibration['screen_status'].eq('positive_context_complete')
    & frozen_connector_calibration['factor_from_target'].gt(PHYSICAL_SUPPORT_FACTOR_LIMIT)
].copy().sort_values(['example_id', 'connector_component_id'])
assert len(window_flagged_connectors) == 14, len(window_flagged_connectors)

physical_window_rows, physical_window_failures = [], []
for connector in window_flagged_connectors.itertuples(index=False):
    try:
        example_id = str(connector.example_id)
        exp_dir = OUTPUT_ROOT / f'{example_id}_indep__anchored'
        state_dir = exp_dir / 'states' / 'S000_base'
        nodes = gpd.read_file(one_path(state_dir / 'variant' / 'directed', '*directed_nodes.gpkg'))
        support = _load_reference_section_support(
            str(exp_dir / 'reference_section' / 'reference_section_summary.json')
        )
        distance_out, _ = _project_nodes_to_reference_section(
            nodes, support=support,
            config=SlopeConfig(reference_section_summary_path=str(
                exp_dir / 'reference_section' / 'reference_section_summary.json'
            )),
        )
        node_x = pd.Series(distance_out.to_numpy(float), index=nodes['id_node'].astype(int))
        profile = frozen_reference_profiles.loc[
            frozen_reference_profiles['example_id'].astype(str).eq(example_id),
            ['reference_section_dist_out_m', 'joint_monotone_wse_m'],
        ]
        a = int(connector.connector_upstream_node)
        b = int(connector.connector_downstream_node)
        result = audit_connector_physical_support(
            profile, upstream_x_m=float(node_x.loc[a]), downstream_x_m=float(node_x.loc[b]),
            connector_length_m=float(connector.connector_length_m),
            connector_slope_m_per_m=float(connector.profile_connector_slope),
            windows_m=PHYSICAL_SUPPORT_WINDOWS_M,
            factor_limit=PHYSICAL_SUPPORT_FACTOR_LIMIT,
        )
        result.update({
            'example_id': example_id,
            'connector_component_id': connector.connector_component_id,
            'link_ids': connector.component_link_ids,
            'outer_upstream_anchor': int(connector.outer_upstream_anchor),
            'connector_upstream_node': a,
            'connector_downstream_node': b,
            'outer_downstream_anchor': int(connector.outer_downstream_anchor),
            'immediate_anchor_target_slope_m_per_m': float(connector.support_target_slope),
            'immediate_anchor_factor': float(connector.factor_from_target),
        })
        physical_window_rows.append(result)
    except Exception as error:
        physical_window_failures.append({
            'example_id': str(connector.example_id),
            'connector_component_id': connector.connector_component_id,
            'error': f'{type(error).__name__}: {error}',
        })

physical_window_audit = pd.DataFrame(physical_window_rows).sort_values(
    ['example_id', 'connector_component_id']
).reset_index(drop=True)
physical_window_failure_audit = pd.DataFrame(physical_window_failures)
if not physical_window_failure_audit.empty:
    display(physical_window_failure_audit)
if len(physical_window_audit) != 14:
    raise RuntimeError('Every genuine factor-five connector must receive a physical-window audit.')

physical_window_display_fields = [
    'example_id', 'connector_component_id', 'link_ids', 'connector_length_m',
    'connector_profile_slope_m_per_m', 'physical_support_target_slope_m_per_m',
    'connector_factor_from_physical_support', 'physical_support_spread_factor',
    'head_adjustment_to_factor_limit_m', 'physical_support_status',
    'upstream_support_slope_1000m', 'downstream_support_slope_1000m',
    'upstream_support_slope_2000m', 'downstream_support_slope_2000m',
    'upstream_support_slope_5000m', 'downstream_support_slope_5000m',
    'immediate_anchor_target_slope_m_per_m', 'immediate_anchor_factor',
]
display(physical_window_audit[physical_window_display_fields])
display(physical_window_audit.groupby('physical_support_status').size().rename(
    'n_connectors'
).reset_index())

plot = go.Figure()
for status, frame in physical_window_audit.groupby('physical_support_status', sort=True):
    plot.add_scatter(
        x=frame['connector_length_m'], y=frame['connector_factor_from_physical_support'],
        mode='markers', name=status,
        text=(frame['example_id'].astype(str) + '/' + frame['connector_component_id']
              + ' (link ' + frame['link_ids'].astype(str) + ')'),
        hovertemplate='%{text}<br>length=%{x:.1f} m<br>physical factor=%{y:.2f}<extra></extra>',
    )
plot.add_hline(y=PHYSICAL_SUPPORT_FACTOR_LIMIT, line_dash='dash', line_color='#E45756',
               annotation_text='factor-five screen')
plot.update_xaxes(type='log', title='Connector length [m]')
plot.update_yaxes(type='log', title='Connector factor from physical support')
plot.update_layout(
    title='All 14 genuine factor-five flags: bounded 1/2/5 km physical support',
    height=520, legend={'orientation': 'h', 'y': -0.22},
)
plot.show()


## 11. Batch fitting, convergence, and final QA

The cells below reproduce the 16-example selection/convergence and connector QA. Production acceptance requires positive profile intervals; positive `len`-based slopes; exact reconstruction from promoted heads; exact domain path loss; explicit physical-floor reporting; and separate global/local/topological review.

In [ ]:
# Batch comparison only: all work happens in a temporary namespace, so the selected
# example above is not overwritten. Run Sections 0--12 at least once before this cell.
import contextlib
import io
import nbformat
import re

BATCH_COMPARISON_EXAMPLE_IDS = [
    '02', '03', '05', '11', '13', '15', '21', '22',
    '37', '41', '45', '46', '48', '54', '55', '56',
]

# Targeted continuation: example 22 alone needs its smooth-side CV bracket
# extended. Restore the 16-example list above for a later full evaluation.
# BATCH_COMPARISON_EXAMPLE_IDS = ['22']
BATCH_COMPARISON_LAMBDA = 3_000.0
BATCH_COMPARISON_WEIGHT_MODE = 'equal'
# Broad regularisation path (31 values, about twofold spacing). The one-SE
# rule deliberately favours smoother fits, so retain substantial headroom beyond
# the L-curve knee rather than truncating candidates on the smooth side.
BATCH_JOINT_LAMBDA_GRID = np.logspace(-1, 8, 31)
BATCH_JOINT_CV_FOLDS = 3
# Begin with a compact CV bracket around the L-curve knee. If the one-SE
# choice is still the smoothest tested value, extend only that example's
# bracket in three-value increments until it is bracketed or the full path
# is exhausted. This avoids spending extra CV fits on settled examples.
BATCH_JOINT_CV_INITIAL_CANDIDATES = 7
BATCH_JOINT_CV_EXTENSION_CANDIDATES = 3

# Numerical tolerance for checking the intrinsic minimum-gradient constraint.
# The fitted floor itself is eta*S_ref, defined in the joint-profile cell.
FLOOR_ACTIVITY_TOLERANCE_M_PER_M = 1e-10

batch_notebook = nbformat.read(
    REPO_ROOT / 'RAPID' / 'notebooks' / 'slope_investigation.ipynb',
    as_version=4,
)
batch_source = {cell.get('id'): cell.source for cell in batch_notebook.cells}
required_batch_cells = [
    'f3f2018b', 'b3690346', '9a7865a3', 'direct-riversp-profile-test',
    'profile-four-candidate-comparison', 'natural-spline-grid-code',
    'raw-riversp-acquisition-audit', 'joint-stage-profile-prototype',
]
missing_batch_cells = [cell_id for cell_id in required_batch_cells if cell_id not in batch_source]
if missing_batch_cells:
    raise RuntimeError('Notebook cell identifiers changed: %s' % missing_batch_cells)

previous_show = go.Figure.show
go.Figure.show = lambda *args, **kwargs: None
batch_profile_rows, batch_joint_profile_rows, batch_joint_observation_rows = [], [], []
batch_summary_rows, batch_slope_rows, batch_cv_rows, batch_failures = [], [], [], []
try:
    for batch_example_id in BATCH_COMPARISON_EXAMPLE_IDS:
        batch_ns = dict(globals())
        batch_ns['display'] = lambda *args, **kwargs: None
        try:
            config_source = re.sub(
                r"EXAMPLE_ID = '[^']+'",
                "EXAMPLE_ID = '%s'" % batch_example_id,
                batch_source['f3f2018b'],
            )
            with contextlib.redirect_stdout(io.StringIO()):
                for source in [
                    config_source,
                    batch_source['b3690346'],
                    batch_source['9a7865a3'],
                    batch_source['direct-riversp-profile-test'],
                    batch_source['profile-four-candidate-comparison'],
                    batch_source['natural-spline-grid-code'],
                    batch_source['raw-riversp-acquisition-audit'],
                ]:
                    exec(compile(source, '<batch-%s>' % batch_example_id, 'exec'), batch_ns)
                joint_source = batch_source['joint-stage-profile-prototype']
                joint_source = joint_source.replace(
                    "STAGE_MODEL_WEIGHT_MODE = 'equal'",
                    "STAGE_MODEL_WEIGHT_MODE = '%s'" % BATCH_COMPARISON_WEIGHT_MODE,
                ).replace(
                    'STAGE_MODEL_LAMBDA = 1_000.0',
                    'STAGE_MODEL_LAMBDA = %r' % BATCH_COMPARISON_LAMBDA,
                )
                exec(compile(joint_source, '<joint-%s>' % batch_example_id, 'exec'), batch_ns)

            # --- Joint raw-pass tuning: cached sparse path, targeted whole-acquisition CV. ---
            stage_obs = batch_ns['stage_model_observations'].copy()
            stage_obs['_joint_weight'] = batch_ns['stage_model_weights']
            lambda_values = np.unique(np.r_[BATCH_JOINT_LAMBDA_GRID, BATCH_COMPARISON_LAMBDA]).astype(float)
            fit_joint = batch_ns['fit_joint_stage_monotone_profile']
            full_joint_trials = {}
            for lambda_value in lambda_values:
                result = fit_joint(
                    batch_ns['stage_model_x_m'], batch_ns['stage_model_node_index'],
                    batch_ns['stage_model_acquisition_index'], batch_ns['stage_model_wse_m'],
                    batch_ns['stage_model_weights'], float(lambda_value),
                    batch_ns['stage_model_reference_profile_m'],
                    batch_ns['stage_model_minimum_slope_m_per_m'],
                    batch_ns['stage_model_penalty'],
                )
                profile = result['profile_m']
                full_joint_trials[float(lambda_value)] = {
                    'result': result,
                    'weighted_mse_m2': result['weighted_mse_m2'],
                    'roughness': float(profile @ (result['penalty'] @ profile)),
                }

            full_trial_table = pd.DataFrame([
                {'lambda_value': lambda_value, 'weighted_mse_m2': trial['weighted_mse_m2'], 'roughness': trial['roughness']}
                for lambda_value, trial in full_joint_trials.items()
            ]).sort_values('lambda_value').reset_index(drop=True)

            # L-curve knee only brackets CV candidates; validation selects the final lambda.
            lcurve_x = np.log10(np.maximum(full_trial_table['roughness'].to_numpy(float), 1e-300))
            lcurve_y = np.log10(np.maximum(full_trial_table['weighted_mse_m2'].to_numpy(float), 1e-300))
            chord = np.array([lcurve_x[-1] - lcurve_x[0], lcurve_y[-1] - lcurve_y[0]])
            chord_norm = float(np.hypot(*chord))
            if chord_norm <= 1e-12:
                lcurve_knee_index = len(full_trial_table) // 2
            else:
                lcurve_distance = np.abs(
                    chord[0] * (lcurve_y - lcurve_y[0]) - chord[1] * (lcurve_x - lcurve_x[0])
                ) / chord_norm
                lcurve_knee_index = int(np.argmax(lcurve_distance))
            candidate_count = min(BATCH_JOINT_CV_INITIAL_CANDIDATES, len(full_trial_table))
            # One weaker-than-knee candidate and the remaining candidates on
            # the smoother side. This brackets the one-SE choice without
            # silently making the largest candidate an artificial ceiling.
            candidate_start = min(max(lcurve_knee_index - 1, 0), len(full_trial_table) - candidate_count)
            cv_candidate_indices = list(range(candidate_start, candidate_start + candidate_count))

            acquisition_order = (
                stage_obs.groupby('acquisition_id', as_index=False)['time_str'].min()
                .sort_values(['time_str', 'acquisition_id']).reset_index(drop=True)
            )
            acquisition_order['cv_fold'] = np.arange(len(acquisition_order)) % BATCH_JOINT_CV_FOLDS
            stage_obs['_cv_fold'] = stage_obs['acquisition_id'].map(
                acquisition_order.set_index('acquisition_id')['cv_fold']
            ).astype(int)

            def _acquisition_cv_scores(lambda_value):
                score_rows = []
                for held_out_fold in range(BATCH_JOINT_CV_FOLDS):
                    train = stage_obs.loc[stage_obs['_cv_fold'].ne(held_out_fold)].copy()
                    test = stage_obs.loc[stage_obs['_cv_fold'].eq(held_out_fold)].copy()
                    train_acquisitions = pd.Index(sorted(train['acquisition_id'].unique()))
                    train_acquisition_lookup = {value: index for index, value in enumerate(train_acquisitions)}
                    train_result = fit_joint(
                        batch_ns['stage_model_x_m'],
                        train['_stage_model_node_index'].to_numpy(int),
                        train['acquisition_id'].map(train_acquisition_lookup).to_numpy(int),
                        train['wse_sm'].to_numpy(float),
                        train['_joint_weight'].to_numpy(float),
                        float(lambda_value),
                        batch_ns['stage_model_reference_profile_m'],
                        batch_ns['stage_model_minimum_slope_m_per_m'],
                        batch_ns['stage_model_penalty'],
                    )
                    predicted_profile = train_result['profile_m'][
                        test['_stage_model_node_index'].to_numpy(int)
                    ]
                    residual_before_offset = test['wse_sm'].to_numpy(float) - predicted_profile
                    test_acquisition_index, _ = pd.factorize(test['acquisition_id'], sort=True)
                    test_weight = test['_joint_weight'].to_numpy(float)
                    held_out_offsets = np.bincount(
                        test_acquisition_index, weights=test_weight * residual_before_offset,
                    ) / np.bincount(test_acquisition_index, weights=test_weight)
                    residual = residual_before_offset - held_out_offsets[test_acquisition_index]
                    held_out = pd.DataFrame({
                        'acquisition_id': test['acquisition_id'].to_numpy(),
                        'weight': test_weight, 'squared_residual': np.square(residual),
                    })
                    for acquisition_id, group in held_out.groupby('acquisition_id', sort=False):
                        score_rows.append({
                            'lambda_value': float(lambda_value),
                            'acquisition_id': acquisition_id,
                            'weighted_mse_m2': float(np.average(
                                group['squared_residual'], weights=group['weight'],
                            )),
                        })
                return pd.DataFrame(score_rows)

            # Score the initial bracket, then extend only if the one-SE rule
            # selects its smoothest value. Cached scores are never recomputed.
            cv_scores_by_lambda = {}
            cv_extension_rounds = 0
            while True:
                cv_lambda_values = full_trial_table['lambda_value'].iloc[cv_candidate_indices].to_numpy(float)
                for value in cv_lambda_values:
                    value = float(value)
                    if value not in cv_scores_by_lambda:
                        cv_scores_by_lambda[value] = _acquisition_cv_scores(value)
                cv_scores_by_acquisition = pd.concat(
                    [cv_scores_by_lambda[float(value)] for value in cv_lambda_values],
                    ignore_index=True,
                )
                cv_summary = (
                    cv_scores_by_acquisition.groupby('lambda_value', as_index=False)
                    .agg(
                        cv_mean_mse_m2=('weighted_mse_m2', 'mean'),
                        cv_sd_mse_m2=('weighted_mse_m2', 'std'),
                        n_acquisitions=('weighted_mse_m2', 'count'),
                    )
                )
                cv_summary['cv_se_mse_m2'] = (
                    cv_summary['cv_sd_mse_m2'].fillna(0.0) / np.sqrt(cv_summary['n_acquisitions'])
                )
                minimum_cv_row = cv_summary.loc[cv_summary['cv_mean_mse_m2'].idxmin()]
                minimum_cv_lambda = float(minimum_cv_row['lambda_value'])
                one_se_threshold_m2 = float(
                    minimum_cv_row['cv_mean_mse_m2'] + minimum_cv_row['cv_se_mse_m2']
                )
                # Within one standard error of the minimum, retain the largest lambda:
                # the smoothest profile that predicts held-out acquisitions comparably well.
                selected_joint_lambda = float(cv_summary.loc[
                    cv_summary['cv_mean_mse_m2'].le(one_se_threshold_m2), 'lambda_value'
                ].max())
                selected_at_upper_edge = selected_joint_lambda == float(np.max(cv_lambda_values))
                next_candidate_index = cv_candidate_indices[-1] + 1
                if not selected_at_upper_edge or next_candidate_index >= len(full_trial_table):
                    break
                extension_stop = min(
                    next_candidate_index + BATCH_JOINT_CV_EXTENSION_CANDIDATES,
                    len(full_trial_table),
                )
                cv_candidate_indices.extend(range(next_candidate_index, extension_stop))
                cv_extension_rounds += 1
            selected_joint_trial = full_joint_trials[selected_joint_lambda]
            joint_profile = pd.DataFrame({
                'reference_section_dist_out_m': batch_ns['stage_model_x_m'],
                'joint_monotone_wse_m': selected_joint_trial['result']['profile_m'],
            })
            batch_joint_profile_rows.append(joint_profile.assign(example_id=batch_example_id))
            selected_offset_lookup = dict(zip(
                batch_ns['stage_model_acquisitions'], selected_joint_trial['result']['offsets_m'],
            ))
            batch_joint_observation_rows.append(
                stage_obs[[
                    'acquisition_id', 'cycle', 'pass', 'time_str', 'node_id',
                    'reference_section_dist_out_m', 'wse_sm', 'wse_sm_u',
                ]].assign(
                    example_id=batch_example_id,
                    acquisition_stage_offset_m=stage_obs['acquisition_id'].map(selected_offset_lookup),
                    stage_aligned_wse_m=lambda frame: (
                        frame['wse_sm'] - frame['acquisition_stage_offset_m']
                    ),
                )
            )
            batch_cv_rows.append(cv_summary.assign(
                example_id=batch_example_id,
                minimum_cv_lambda=minimum_cv_lambda,
                one_se_threshold_m2=one_se_threshold_m2,
                selected_lambda=selected_joint_lambda,
            ))
            joint_tuning_summary = {
                'selected_joint_lambda': selected_joint_lambda,
                'minimum_cv_lambda': minimum_cv_lambda,
                'lcurve_knee_lambda': float(full_trial_table['lambda_value'].iloc[lcurve_knee_index]),
                'minimum_cv_mse_m2': float(minimum_cv_row['cv_mean_mse_m2']),
                'selected_joint_cv_mse_m2': float(cv_summary.loc[
                    cv_summary['lambda_value'].eq(selected_joint_lambda), 'cv_mean_mse_m2'
                ].iloc[0]),
                'one_se_threshold_m2': one_se_threshold_m2,
                'selected_lambda_is_cv_grid_upper_edge': bool(
                    selected_at_upper_edge
                ),
                'cv_extension_rounds': cv_extension_rounds,
                'cv_grid_exhausted': bool(
                    selected_at_upper_edge and cv_candidate_indices[-1] == len(full_trial_table) - 1
                ),
                'cv_lambda_values': ', '.join('%g' % value for value in cv_lambda_values),
                'full_lambda_values': ', '.join('%g' % value for value in lambda_values),
            }


            previous_profile = batch_ns['profile_table'][
                ['reference_section_dist_out_m', 'monotone_wse_m', 'wse_obs_p10', 'wse_obs_p90']
            ].rename(columns={'monotone_wse_m': 'previous_p50_monotone_wse_m'})
            # joint_profile was selected above by whole-acquisition validation.
            # This is the previously proposed natural smoothing-spline method. Where
            # the displayed grid has a strictly monotone candidate, select its first
            # admissible m=s/N. For a remaining problem case retain the least-deficit
            # candidate and label it clearly rather than pretending it is admissible.
            selected_smoothing = batch_ns['selected_natural_grid_row']
            if selected_smoothing.empty:
                selected_smoothing = batch_ns['natural_spline_grid'].sort_values(
                    ['monotonicity_deficit_m_per_m', 'max_PAVA_adjustment_m', 'm_equals_s_over_N']
                ).head(1)
                smoothing_status = 'least-deficit (not admissible)'
            else:
                smoothing_status = 'first admissible monotone'
            smoothing_m = float(selected_smoothing['m_equals_s_over_N'].iloc[0])
            smoothing_profile = pd.DataFrame({
                'reference_section_dist_out_m': batch_ns['candidate_x'],
                's_over_n_smoothing_wse_m': batch_ns['natural_grid_profiles'][f'm={smoothing_m:g}'],
            })
            compared = previous_profile.merge(
                smoothing_profile, on='reference_section_dist_out_m', validate='1:1',
            ).merge(
                joint_profile, on='reference_section_dist_out_m', validate='1:1',
            ).sort_values('reference_section_dist_out_m')
            compared['example_id'] = batch_example_id
            compared['difference_new_minus_previous_m'] = (
                compared['joint_monotone_wse_m'] - compared['previous_p50_monotone_wse_m']
            )
            compared['smoothing_m_equals_s_over_n'] = smoothing_m
            compared['smoothing_selection_status'] = smoothing_status
            batch_profile_rows.append(compared)

            # Audit the intrinsic numerical floor. Intervals at the floor are not
            # discarded: they are retained, explicitly labelled, and quantify where
            # raw support plus smoothing would otherwise create a plateau.
            joint_profile_intervals = pd.DataFrame({
                'example_id': batch_example_id,
                'interval_start_m': joint_profile['reference_section_dist_out_m'].iloc[:-1].to_numpy(float),
                'interval_end_m': joint_profile['reference_section_dist_out_m'].iloc[1:].to_numpy(float),
            })
            joint_profile_intervals['interval_length_m'] = (
                joint_profile_intervals['interval_end_m'] - joint_profile_intervals['interval_start_m']
            )
            joint_profile_intervals['profile_slope_m_per_m'] = np.diff(
                joint_profile['joint_monotone_wse_m'].to_numpy(float)
            ) / (
                joint_profile_intervals['interval_length_m'].to_numpy(float)
            )
            joint_global_reference_slope = batch_ns['stage_model_reference_slope_m_per_m']
            joint_minimum_slope = batch_ns['stage_model_minimum_slope_m_per_m']
            fitted_endpoint_heads = selected_joint_trial['result']['profile_m'][
                batch_ns['stage_model_endpoint_node_index']
            ]
            joint_fitted_head_loss_m = float(np.diff(fitted_endpoint_heads)[0])
            joint_static_p50_head_loss_m = float(np.diff(batch_ns['endpoint_reference_heads_m'])[0])
            joint_fitted_global_slope = float(
                joint_fitted_head_loss_m / batch_ns['stage_model_network_length_m']
            )
            joint_head_loss_difference_m = joint_fitted_head_loss_m - joint_static_p50_head_loss_m
            joint_head_loss_relative_difference = (
                joint_head_loss_difference_m / joint_static_p50_head_loss_m
            )
            joint_endpoint_difference_m = (
                fitted_endpoint_heads - batch_ns['endpoint_reference_heads_m']
            )
            joint_profile_intervals['slope_ratio_to_joint_global'] = (
                joint_profile_intervals['profile_slope_m_per_m'] / joint_global_reference_slope
            )
            joint_profile_intervals['minimum_slope_m_per_m'] = joint_minimum_slope
            joint_profile_intervals['is_floor_active'] = joint_profile_intervals['profile_slope_m_per_m'].le(
                joint_minimum_slope + FLOOR_ACTIVITY_TOLERANCE_M_PER_M
            )
            joint_profile_intervals['is_above_floor'] = ~joint_profile_intervals['is_floor_active']
            batch_slope_rows.append(joint_profile_intervals)

            distance = compared['reference_section_dist_out_m'].to_numpy(float)
            previous_wse = compared['previous_p50_monotone_wse_m'].to_numpy(float)
            joint_wse = compared['joint_monotone_wse_m'].to_numpy(float)
            above_floor_slopes = joint_profile_intervals.loc[
                joint_profile_intervals['is_above_floor'], 'profile_slope_m_per_m'
            ]
            above_floor_slope_ratios = joint_profile_intervals.loc[
                joint_profile_intervals['is_above_floor'], 'slope_ratio_to_joint_global'
            ]
            batch_summary_rows.append({
                'example_id': batch_example_id,
                'n_profile_nodes': len(joint_profile),
                'n_directly_observed_profile_nodes': selected_joint_trial['result']['n_observed_profile_nodes'],
                'n_raw_observations': len(batch_ns['stage_model_observations']),
                'n_acquisitions': len(batch_ns['stage_model_acquisitions']),
                'smoothing_m_equals_s_over_n': smoothing_m,
                'smoothing_selection_status': smoothing_status,
                'joint_selected_lambda': joint_tuning_summary['selected_joint_lambda'],
                'joint_lambda_selection_rule': 'one-standard-error',
                'joint_minimum_cv_lambda': joint_tuning_summary['minimum_cv_lambda'],
                'joint_lcurve_knee_lambda': joint_tuning_summary['lcurve_knee_lambda'],
                'joint_minimum_cv_mse_m2': joint_tuning_summary['minimum_cv_mse_m2'],
                'joint_selected_cv_mse_m2': joint_tuning_summary['selected_joint_cv_mse_m2'],
                'joint_one_se_threshold_m2': joint_tuning_summary['one_se_threshold_m2'],
                'joint_selected_lambda_is_cv_grid_upper_edge': (
                    joint_tuning_summary['selected_lambda_is_cv_grid_upper_edge']
                ),
                'joint_cv_extension_rounds': joint_tuning_summary['cv_extension_rounds'],
                'joint_cv_grid_exhausted': joint_tuning_summary['cv_grid_exhausted'],
                'joint_cv_lambda_values': joint_tuning_summary['cv_lambda_values'],
                'previous_global_slope_m_per_m': (previous_wse[-1] - previous_wse[0]) / (distance[-1] - distance[0]),
                's_over_n_global_slope_m_per_m': (
                    compared['s_over_n_smoothing_wse_m'].iloc[-1] - compared['s_over_n_smoothing_wse_m'].iloc[0]
                ) / (distance[-1] - distance[0]),
                'joint_static_p50_head_loss_m': joint_static_p50_head_loss_m,
                'joint_fitted_head_loss_m': joint_fitted_head_loss_m,
                'joint_head_loss_difference_m': joint_head_loss_difference_m,
                'joint_head_loss_relative_difference': joint_head_loss_relative_difference,
                'joint_outlet_head_difference_m': joint_endpoint_difference_m[0],
                'joint_inlet_head_difference_m': joint_endpoint_difference_m[1],
                'joint_reference_slope_m_per_m': joint_global_reference_slope,
                'joint_global_slope_m_per_m': joint_fitted_global_slope,
                'joint_minimum_slope_m_per_m': joint_minimum_slope,
                'joint_minimum_slope_fraction_of_reference': batch_ns['STAGE_MODEL_MINIMUM_SLOPE_FRACTION'],
                's_over_n_rms_difference_from_previous_m': float(np.sqrt(np.mean(np.square(
                    compared['s_over_n_smoothing_wse_m'].to_numpy(float) - previous_wse
                )))),
                'profile_rms_difference_m': float(np.sqrt(np.mean(np.square(joint_wse - previous_wse)))),
                'profile_max_abs_difference_m': float(np.max(np.abs(joint_wse - previous_wse))),
                'joint_weighted_mse_m2': selected_joint_trial['result']['weighted_mse_m2'],
                'joint_weighted_rms_m': selected_joint_trial['result']['weighted_rms_m'],
                'n_joint_profile_intervals': len(joint_profile_intervals),
                'n_joint_floor_active_intervals': int(joint_profile_intervals['is_floor_active'].sum()),
                'floor_active_length_m': float(joint_profile_intervals.loc[
                    joint_profile_intervals['is_floor_active'], 'interval_length_m'
                ].sum()),
                'minimum_above_floor_joint_slope_m_per_m': (
                    above_floor_slopes.min() if not above_floor_slopes.empty else np.nan
                ),
                'minimum_above_floor_joint_slope_ratio_to_global': (
                    above_floor_slope_ratios.min() if not above_floor_slope_ratios.empty else np.nan
                ),
                'p05_above_floor_joint_slope_m_per_m': (
                    above_floor_slopes.quantile(0.05) if not above_floor_slopes.empty else np.nan
                ),
                'status': 'completed',
            })
        except Exception as error:
            batch_failures.append({
                'example_id': batch_example_id,
                'error': '%s: %s' % (type(error).__name__, error),
            })
finally:
    go.Figure.show = previous_show

all_example_profile_comparison = (
    pd.concat(batch_profile_rows, ignore_index=True) if batch_profile_rows else pd.DataFrame()
)
all_example_joint_profile = (
    pd.concat(batch_joint_profile_rows, ignore_index=True) if batch_joint_profile_rows else pd.DataFrame()
)
all_example_joint_observations = (
    pd.concat(batch_joint_observation_rows, ignore_index=True) if batch_joint_observation_rows else pd.DataFrame()
)
all_example_joint_cv = (
    pd.concat(batch_cv_rows, ignore_index=True) if batch_cv_rows else pd.DataFrame()
)
all_example_profile_summary = (
    pd.DataFrame(batch_summary_rows).sort_values('example_id') if batch_summary_rows else pd.DataFrame()
)
all_example_profile_failures = pd.DataFrame(batch_failures)
joint_profile_slope_diagnostic = (
    pd.concat(batch_slope_rows, ignore_index=True) if batch_slope_rows else pd.DataFrame()
)
print('Completed: %d; failed: %d.' % (
    len(all_example_profile_summary), len(all_example_profile_failures),
))
display(all_example_profile_summary)

# Population audit of the implemented minimum-gradient constraint. This reports
# how much of each fitted corridor is actively conditioned, rather than treating
# a data-gap plateau as an observed hydraulic low slope.
if not joint_profile_slope_diagnostic.empty:
    all_interval_slopes = joint_profile_slope_diagnostic['profile_slope_m_per_m'].dropna()
    above_floor_population = joint_profile_slope_diagnostic.loc[
        joint_profile_slope_diagnostic['is_above_floor'], 'profile_slope_m_per_m'
    ].dropna()
    def slope_population_row(label, values, units):
        return {
            'population': label, 'units': units, 'n': len(values),
            'minimum': values.min(), 'p01': values.quantile(0.01),
            'p05': values.quantile(0.05), 'p10': values.quantile(0.10),
            'median': values.median(),
        }
    joint_profile_slope_population_summary = pd.DataFrame([
        slope_population_row('all fitted fixed-grid intervals', all_interval_slopes, 'm/m'),
        slope_population_row('fitted intervals above the numerical floor', above_floor_population, 'm/m'),
    ])
    display(joint_profile_slope_population_summary)
    display(all_example_profile_summary[[
        'example_id', 'joint_lambda_selection_rule', 'joint_minimum_cv_lambda',
        'joint_selected_lambda', 'joint_selected_lambda_is_cv_grid_upper_edge',
        'joint_cv_extension_rounds', 'joint_cv_grid_exhausted',
        'joint_static_p50_head_loss_m',
        'joint_fitted_head_loss_m', 'joint_head_loss_difference_m',
        'joint_head_loss_relative_difference', 'joint_outlet_head_difference_m',
        'joint_inlet_head_difference_m', 'joint_minimum_slope_m_per_m',
        'n_joint_floor_active_intervals', 'floor_active_length_m',
        'minimum_above_floor_joint_slope_m_per_m',
        'minimum_above_floor_joint_slope_ratio_to_global',
        'p05_above_floor_joint_slope_m_per_m',
    ]])
if not all_example_profile_failures.empty:
    display(all_example_profile_failures)

all_example_profile_fig = go.Figure()
completed_example_ids = all_example_profile_summary['example_id'].astype(str).tolist()
for example_position, batch_example_id in enumerate(completed_example_ids):
    compared = all_example_profile_comparison.loc[
        all_example_profile_comparison['example_id'].eq(batch_example_id)
    ].sort_values('reference_section_dist_out_m')
    x_km = compared['reference_section_dist_out_m'] / 1_000.0
    visible = example_position == 0
    # Invisible lower envelope edge plus filled upper edge yields a selectable P10--P90 band.
    all_example_profile_fig.add_scatter(
        x=x_km, y=compared['wse_obs_p10'], mode='lines',
        line={'width': 0}, hoverinfo='skip', showlegend=False, visible=visible,
    )
    all_example_profile_fig.add_scatter(
        x=x_km, y=compared['wse_obs_p90'], mode='lines',
        name='%s: SWORD P10--P90' % batch_example_id,
        line={'width': 0}, fill='tonexty', fillcolor='rgba(120,120,120,0.20)',
        visible=visible,
    )
    all_example_profile_fig.add_scatter(
        x=x_km, y=compared['previous_p50_monotone_wse_m'], mode='lines',
        name='%s: old P50/PAVA monotone' % batch_example_id,
        line={'color': '#D62728', 'width': 3, 'dash': 'dot'}, visible=visible,
    )
    all_example_profile_fig.add_scatter(
        x=x_km, y=compared['s_over_n_smoothing_wse_m'], mode='lines',
        name='%s: P50 natural spline, m=%g' % (batch_example_id, compared['smoothing_m_equals_s_over_n'].iloc[0]),
        line={'color': '#F58518', 'width': 3}, visible=visible,
    )
    all_example_profile_fig.add_scatter(
        x=x_km, y=compared['joint_monotone_wse_m'], mode='lines',
        name='%s: raw-pass profile (one-SE acquisition-CV, lambda=%g)' % (
            batch_example_id,
            all_example_profile_summary.loc[all_example_profile_summary['example_id'].eq(batch_example_id), 'joint_selected_lambda'].iloc[0],
        ),
        line={'color': '#1F77B4', 'width': 3}, visible=visible,
    )

profile_buttons = []
for example_position, batch_example_id in enumerate(completed_example_ids):
    visible = [False] * (5 * len(completed_example_ids))
    visible[5 * example_position:5 * example_position + 5] = [True] * 5
    profile_buttons.append({
        'label': 'Example %s' % batch_example_id,
        'method': 'update',
        'args': [
            {'visible': visible},
            {'title': 'Example %s: PAVA, P50 natural-spline, and one-SE raw-pass profile' % batch_example_id},
        ],
    })
if completed_example_ids:
    all_example_profile_fig.update_layout(
        title='Example %s: PAVA, P50 natural-spline, and one-SE raw-pass profile' % completed_example_ids[0],
        xaxis_title='Distance to outlet [km]', yaxis_title='WSE [m]', height=720,
        updatemenus=[{
            'buttons': profile_buttons, 'x': 0.0, 'y': 1.18,
            'xanchor': 'left', 'yanchor': 'top',
        }],
        legend={'orientation': 'h', 'x': 0.0, 'y': -0.22, 'itemclick': 'toggle', 'itemdoubleclick': 'toggleothers'},
        margin={'t': 120, 'b': 135, 'r': 40},
    )
    all_example_profile_fig.show()



In [ ]:
# Run the final batch-comparison cell first. This screens every example for
# local fitted-slope changes, then lets one selected example be inspected
# against its stage-aligned individual raw acquisitions.
KINKS_PER_EXAMPLE = 5
FOCUSED_KINK_EXAMPLE_ID = '22'  # Change and rerun this cell for any example.

if all_example_joint_profile.empty or all_example_joint_observations.empty:
    raise RuntimeError('Run the final batch-comparison cell before this diagnostic.')

kink_rows = []
summary_lookup = all_example_profile_summary.set_index('example_id')
for example_id, profile in all_example_joint_profile.groupby('example_id', sort=True):
    profile = profile.sort_values('reference_section_dist_out_m').reset_index(drop=True)
    x_m = profile['reference_section_dist_out_m'].to_numpy(float)
    slope = np.diff(profile['joint_monotone_wse_m'].to_numpy(float)) / np.diff(x_m)
    slope_change = np.diff(slope)
    reference_slope = float(summary_lookup.loc[example_id, 'joint_reference_slope_m_per_m'])
    minimum_slope = float(summary_lookup.loc[example_id, 'joint_minimum_slope_m_per_m'])
    for index, change in enumerate(slope_change, start=1):
        left_slope, right_slope = slope[index - 1], slope[index]
        left_floor = left_slope <= minimum_slope + FLOOR_ACTIVITY_TOLERANCE_M_PER_M
        right_floor = right_slope <= minimum_slope + FLOOR_ACTIVITY_TOLERANCE_M_PER_M
        kink_rows.append({
            'example_id': example_id,
            'kink_distance_to_outlet_km': x_m[index] / 1_000.0,
            'left_slope_m_per_m': left_slope,
            'right_slope_m_per_m': right_slope,
            'slope_change_m_per_m': change,
            'absolute_slope_change_over_reference': abs(change) / reference_slope,
            'left_interval_floor_active': left_floor,
            'right_interval_floor_active': right_floor,
            'candidate_type': (
                'floor transition' if left_floor != right_floor else 'above-floor slope change'
            ),
        })

all_profile_kink_candidates = pd.DataFrame(kink_rows)
all_profile_kink_candidates['rank_within_example'] = (
    all_profile_kink_candidates.groupby('example_id')['absolute_slope_change_over_reference']
    .rank(method='first', ascending=False).astype(int)
)
reported_profile_kinks = all_profile_kink_candidates.loc[
    all_profile_kink_candidates['rank_within_example'].le(KINKS_PER_EXAMPLE)
].sort_values(['example_id', 'rank_within_example'])
display(reported_profile_kinks)

focused_kinks = reported_profile_kinks.loc[
    reported_profile_kinks['example_id'].eq(FOCUSED_KINK_EXAMPLE_ID)
].sort_values('rank_within_example')
focused_profile = all_example_joint_profile.loc[
    all_example_joint_profile['example_id'].eq(FOCUSED_KINK_EXAMPLE_ID)
].sort_values('reference_section_dist_out_m')
focused_observations = all_example_joint_observations.loc[
    all_example_joint_observations['example_id'].eq(FOCUSED_KINK_EXAMPLE_ID)
].sort_values(['acquisition_id', 'reference_section_dist_out_m'])
focused_comparison = all_example_profile_comparison.loc[
    all_example_profile_comparison['example_id'].eq(FOCUSED_KINK_EXAMPLE_ID)
].sort_values('reference_section_dist_out_m')
if focused_profile.empty:
    raise ValueError(f'No completed profile for Example {FOCUSED_KINK_EXAMPLE_ID}.')

focused_kink_fig = go.Figure()
if not focused_comparison.empty:
    x_km = focused_comparison['reference_section_dist_out_m'] / 1_000.0
    focused_kink_fig.add_scatter(
        x=x_km, y=focused_comparison['wse_obs_p10'], mode='lines',
        line={'width': 0}, hoverinfo='skip', showlegend=False,
    )
    focused_kink_fig.add_scatter(
        x=x_km, y=focused_comparison['wse_obs_p90'], mode='lines',
        name='static SWORD P10--P90', line={'width': 0}, fill='tonexty',
        fillcolor='rgba(120,120,120,0.20)',
    )
for acquisition_id, group in focused_observations.groupby('acquisition_id', sort=True):
    group = group.sort_values('reference_section_dist_out_m')
    focused_kink_fig.add_scatter(
        x=group['reference_section_dist_out_m'] / 1_000.0, y=group['stage_aligned_wse_m'],
        mode='lines+markers', marker={'size': 3}, line={'width': 1}, opacity=0.35,
        name=f"cycle {int(group['cycle'].iloc[0])}, pass {int(group['pass'].iloc[0])}",
        customdata=group[['node_id', 'acquisition_stage_offset_m', 'wse_sm_u']].to_numpy(),
        hovertemplate=(
            'distance=%{x:.3f} km<br>stage-aligned WSE=%{y:.3f} m'
            '<br>node=%{customdata[0]}<br>offset=%{customdata[1]:.3f} m'
            '<br>wse_sm_u=%{customdata[2]:.3f} m<extra></extra>'
        ),
    )
focused_kink_fig.add_scatter(
    x=focused_profile['reference_section_dist_out_m'] / 1_000.0,
    y=focused_profile['joint_monotone_wse_m'], mode='lines',
    name='selected one-SE raw-pass profile', line={'color': 'black', 'width': 4},
)
for _, kink in focused_kinks.iterrows():
    focused_kink_fig.add_vline(
        x=kink['kink_distance_to_outlet_km'], line={'color': '#D62728', 'width': 1.5, 'dash': 'dash'},
        annotation_text=f"#{int(kink['rank_within_example'])}: {kink['candidate_type']}",
        annotation_position='top',
    )
focused_kink_fig.update_layout(
    title=(f'Example {FOCUSED_KINK_EXAMPLE_ID}: individual stage-aligned passes and '
           'largest fitted slope-change candidates'),
    xaxis_title='Distance to outlet [km]', yaxis_title='WSE [m]', height=780,
    legend={'itemclick': 'toggle', 'itemdoubleclick': 'toggleothers'},
)
focused_kink_fig.show()


In [ ]:
# Incremental continuation for the examples that still selected the upper lambda edge.
# Run the completed 16-example batch cell first, then run this cell. Re-run only
# if the last line reports remaining upper-edge examples. Existing baseline CV
# summaries are reused; this evaluates eight new upper-tail lambdas per case.
from rapid_tools.slope_profile_upper_tail import run_upper_tail_continuation

upper_tail_result = run_upper_tail_continuation(
    globals(),
    notebook_path=REPO_ROOT / 'RAPID' / 'notebooks' / 'slope_investigation.ipynb',
    lambdas_per_run=8,
    log10_step=0.3,
    max_lambda=1e14,
)

print('Completed:', len(upper_tail_result['summary']), '; failed:', len(upper_tail_result['failures']))
display(upper_tail_result['summary'])
if not upper_tail_result['failures'].empty:
    display(upper_tail_result['failures'])
remaining_upper_tail_examples = upper_tail_result['status'].loc[
    upper_tail_result['status']['selected_at_current_upper_edge'].fillna(False), 'example_id'
].astype(str).tolist()
print('Run this continuation cell again only for:', ', '.join(remaining_upper_tail_examples) or 'none')


In [ ]:
# Profile-convergence audit: compare the original one-SE selection retained by
# the 16-example batch with the latest upper-tail profile. This does not freeze
# or promote a profile automatically; it makes that decision auditable.
PROFILE_CONVERGENCE_RMS_WSE_TOL_M = 0.01
PROFILE_CONVERGENCE_MAX_WSE_TOL_M = 0.05
PROFILE_CONVERGENCE_RMS_SLOPE_FRACTION_TOL = 0.01
PROFILE_CONVERGENCE_MAX_SLOPE_FRACTION_TOL = 0.05

required_convergence_objects = [
    'all_example_joint_profile', 'all_example_profile_summary',
    'upper_tail_continuation_profiles',
]
missing_convergence_objects = [
    name for name in required_convergence_objects if name not in globals()
]
if missing_convergence_objects:
    raise RuntimeError(
        'Run the batch and at least one upper-tail continuation in this kernel; '
        f'missing: {missing_convergence_objects}'
    )

convergence_rows = []
convergence_interval_rows = []
summary_by_example = all_example_profile_summary.set_index('example_id')
for convergence_example_id, tail_profile in upper_tail_continuation_profiles.groupby('example_id', sort=True):
    base_profile = all_example_joint_profile.loc[
        all_example_joint_profile['example_id'].astype(str).eq(str(convergence_example_id))
    ][['reference_section_dist_out_m', 'joint_monotone_wse_m']].rename(
        columns={'joint_monotone_wse_m': 'base_wse_m'}
    )
    tail_profile = tail_profile[[
        'reference_section_dist_out_m', 'joint_monotone_wse_m'
    ]].rename(columns={'joint_monotone_wse_m': 'tail_wse_m'})
    compared_profiles = base_profile.merge(
        tail_profile, on='reference_section_dist_out_m', how='outer', validate='1:1', indicator=True
    ).sort_values('reference_section_dist_out_m').reset_index(drop=True)
    if not compared_profiles['_merge'].eq('both').all():
        raise RuntimeError(
            f'Fixed profile grids differ for example {convergence_example_id}; '
            'convergence requires identical corridor coordinates.'
        )
    x_m = compared_profiles['reference_section_dist_out_m'].to_numpy(float)
    base_wse_m = compared_profiles['base_wse_m'].to_numpy(float)
    tail_wse_m = compared_profiles['tail_wse_m'].to_numpy(float)
    wse_difference_m = tail_wse_m - base_wse_m
    base_slope = np.diff(base_wse_m) / np.diff(x_m)
    tail_slope = np.diff(tail_wse_m) / np.diff(x_m)
    slope_difference = tail_slope - base_slope
    maximum_slope_change_index = int(np.argmax(np.abs(slope_difference)))
    convergence_interval_rows.append({
        'example_id': str(convergence_example_id),
        'interval_start_km': x_m[maximum_slope_change_index] / 1_000.0,
        'interval_end_km': x_m[maximum_slope_change_index + 1] / 1_000.0,
        'interval_length_m': x_m[maximum_slope_change_index + 1] - x_m[maximum_slope_change_index],
        'base_slope_m_per_m': base_slope[maximum_slope_change_index],
        'upper_tail_slope_m_per_m': tail_slope[maximum_slope_change_index],
        'slope_change_m_per_m': slope_difference[maximum_slope_change_index],
        'head_change_across_interval_m': (
            (tail_wse_m[maximum_slope_change_index + 1] - tail_wse_m[maximum_slope_change_index])
            - (base_wse_m[maximum_slope_change_index + 1] - base_wse_m[maximum_slope_change_index])
        ),
    })
    reference_slope = float(
        summary_by_example.loc[convergence_example_id, 'joint_reference_slope_m_per_m']
    )
    convergence_rows.append({
        'example_id': str(convergence_example_id),
        'n_profile_nodes': len(compared_profiles),
        'base_lambda': float(summary_by_example.loc[convergence_example_id, 'joint_selected_lambda']),
        'upper_tail_lambda': float(
            upper_tail_status.loc[
                upper_tail_status['example_id'].astype(str).eq(str(convergence_example_id)),
                'selected_lambda'
            ].iloc[0]
        ),
        'wse_rms_difference_m': float(np.sqrt(np.mean(np.square(wse_difference_m)))),
        'wse_max_abs_difference_m': float(np.max(np.abs(wse_difference_m))),
        'outlet_head_change_m': float(wse_difference_m[0]),
        'inlet_head_change_m': float(wse_difference_m[-1]),
        'slope_rms_difference_m_per_m': float(np.sqrt(np.mean(np.square(slope_difference)))),
        'slope_max_abs_difference_m_per_m': float(np.max(np.abs(slope_difference))),
        'slope_rms_difference_over_reference': float(
            np.sqrt(np.mean(np.square(slope_difference))) / reference_slope
        ),
        'slope_max_abs_difference_over_reference': float(
            np.max(np.abs(slope_difference)) / reference_slope
        ),
    })

profile_convergence_audit = pd.DataFrame(convergence_rows).sort_values('example_id')
profile_convergence_audit['within_proposed_wse_tolerance'] = (
    profile_convergence_audit['wse_rms_difference_m'].le(PROFILE_CONVERGENCE_RMS_WSE_TOL_M)
    & profile_convergence_audit['wse_max_abs_difference_m'].le(PROFILE_CONVERGENCE_MAX_WSE_TOL_M)
)
profile_convergence_audit['within_proposed_slope_tolerance'] = (
    profile_convergence_audit['slope_rms_difference_over_reference'].le(
        PROFILE_CONVERGENCE_RMS_SLOPE_FRACTION_TOL
    )
    & profile_convergence_audit['slope_max_abs_difference_over_reference'].le(
        PROFILE_CONVERGENCE_MAX_SLOPE_FRACTION_TOL
    )
)
profile_convergence_audit['converged_under_proposed_tolerances'] = (
    profile_convergence_audit['within_proposed_wse_tolerance']
    & profile_convergence_audit['within_proposed_slope_tolerance']
)
profile_convergence_max_slope_intervals = pd.DataFrame(convergence_interval_rows).sort_values('example_id')

display(profile_convergence_audit)
display(profile_convergence_max_slope_intervals)
print(
    'Converged under proposed diagnostic tolerances: %d/%d' % (
        profile_convergence_audit['converged_under_proposed_tolerances'].sum(),
        len(profile_convergence_audit),
    )
)


In [ ]:
# Recalibrate the connector screen with the frozen raw-pass profile choice.
# This is read-only: it does not modify hydraulic contexts or connector heads.
required_frozen_profile_objects = [
    'all_example_joint_profile', 'all_example_profile_summary',
    'upper_tail_continuation_profiles', 'upper_tail_status',
]
missing_frozen_profile_objects = [
    name for name in required_frozen_profile_objects if name not in globals()
]
if missing_frozen_profile_objects:
    raise RuntimeError(
        'Run the batch and upper-tail continuation first in this kernel; '
        f'missing: {missing_frozen_profile_objects}'
    )

FROZEN_CONNECTOR_EXAMPLE_IDS = [
    '02', '03', '05', '11', '13', '15', '21', '22',
    '37', '41', '45', '46', '48', '54', '55', '56',
]

# Four profiles were bounded within the original one-SE grid. The other twelve
# use their reviewed upper-tail minimum-curvature profile.
upper_tail_profile_ids = set(upper_tail_continuation_profiles['example_id'].astype(str))
frozen_profile_rows, frozen_profile_registry_rows = [], []
for frozen_example_id in FROZEN_CONNECTOR_EXAMPLE_IDS:
    if frozen_example_id in upper_tail_profile_ids:
        profile = upper_tail_continuation_profiles.loc[
            upper_tail_continuation_profiles['example_id'].astype(str).eq(frozen_example_id)
        ].copy()
        profile_decision = 'converged_upper_tail_minimum_curvature'
        selected_lambda = float(upper_tail_status.loc[
            upper_tail_status['example_id'].astype(str).eq(frozen_example_id), 'selected_lambda'
        ].iloc[0])
    else:
        profile = all_example_joint_profile.loc[
            all_example_joint_profile['example_id'].astype(str).eq(frozen_example_id)
        ].copy()
        profile_decision = 'finite_one_standard_error_selection'
        selected_lambda = float(all_example_profile_summary.loc[
            all_example_profile_summary['example_id'].astype(str).eq(frozen_example_id),
            'joint_selected_lambda'
        ].iloc[0])
    if profile.empty:
        raise RuntimeError(f'No frozen profile is available for example {frozen_example_id}.')
    frozen_profile_rows.append(profile[[
        'example_id', 'reference_section_dist_out_m', 'joint_monotone_wse_m'
    ]])
    frozen_profile_registry_rows.append({
        'example_id': frozen_example_id,
        'profile_decision': profile_decision,
        'selected_lambda': selected_lambda,
        'n_profile_nodes': len(profile),
    })
frozen_reference_profiles = pd.concat(frozen_profile_rows, ignore_index=True)
frozen_profile_registry = pd.DataFrame(frozen_profile_registry_rows).sort_values('example_id')
display(frozen_profile_registry)

def frozen_connector_screen_for_example(example_id: str) -> tuple[pd.DataFrame, dict]:
    exp_dir = OUTPUT_ROOT / f'{example_id}_indep__anchored'
    state_dir = exp_dir / 'states' / 'S000_base'
    links_path = one_path(state_dir / 'variant' / 'directed', '*directed_links.gpkg')
    nodes_path = one_path(state_dir / 'variant' / 'directed', '*directed_nodes.gpkg')
    local_links = gpd.read_file(links_path)
    local_nodes = gpd.read_file(nodes_path)
    local_context = pd.read_csv(state_dir / 'hydraulic_context' / 'link_hydraulic_context.csv')
    local_bubbles = pd.read_csv(state_dir / 'hierarchy' / 'bubble_equivalent_metrics.csv')
    local_support = _load_reference_section_support(
        str(exp_dir / 'reference_section' / 'reference_section_summary.json')
    )
    local_profile = frozen_reference_profiles.loc[
        frozen_reference_profiles['example_id'].astype(str).eq(example_id),
        ['reference_section_dist_out_m', 'joint_monotone_wse_m'],
    ].sort_values('reference_section_dist_out_m')

    distance_out, _ = _project_nodes_to_reference_section(
        local_nodes, support=local_support,
        config=SlopeConfig(
            reference_section_summary_path=str(
                exp_dir / 'reference_section' / 'reference_section_summary.json'
            )
        ),
    )
    projected_nodes = (
        local_nodes.to_crs(local_support['centerline_crs'])
        if str(local_nodes.crs) != str(local_support['centerline_crs']) else local_nodes
    )
    degree = pd.concat([
        local_links.groupby('id_us_node').size().rename('out_degree'),
        local_links.groupby('id_ds_node').size().rename('in_degree'),
    ], axis=1).fillna(0)
    node_frame = local_nodes[['id_node']].copy()
    node_frame['reference_section_dist_out_m'] = distance_out
    node_frame['centerline_offset_m'] = projected_nodes.geometry.distance(local_support['line'])
    node_frame = node_frame.join(degree, on='id_node').fillna({'in_degree': 0, 'out_degree': 0})
    boundary_ids = set(pd.to_numeric(
        local_bubbles['upstream_boundary_node'], errors='coerce'
    ).dropna().astype(int)) | set(pd.to_numeric(
        local_bubbles['downstream_boundary_node'], errors='coerce'
    ).dropna().astype(int))
    accepted = set(node_frame.loc[
        node_frame['in_degree'].eq(0)
        | node_frame['out_degree'].eq(0)
        | (node_frame['id_node'].isin(boundary_ids)
           & node_frame['centerline_offset_m'].le(REFERENCE_NODE_MAX_OFFSET_M)),
        'id_node',
    ].astype(int))
    node_frame['profile_wse_m'] = np.interp(
        node_frame['reference_section_dist_out_m'],
        local_profile['reference_section_dist_out_m'],
        local_profile['joint_monotone_wse_m'],
    )
    profile_by_node = node_frame.set_index('id_node')[
        ['profile_wse_m', 'reference_section_dist_out_m']
    ]

    work = local_links[['id_link', 'id_us_node', 'id_ds_node', 'unit_ids', 'len']].copy()
    slope_fields = [
        column for column in ['id_link', 'slope_used', 'raw_slope', 'slope_source_method']
        if column in local_context.columns
    ]
    work = work.merge(local_context[slope_fields], on='id_link', how='left', validate='1:1')
    work['actual_length_m'] = preferred_length(work)
    unassigned = work.loc[work['unit_ids'].map(parse_ids).map(len).eq(0)].copy()
    graph, undirected = nx.DiGraph(), nx.Graph()
    for row in local_links.itertuples(index=False):
        graph.add_edge(int(row.id_us_node), int(row.id_ds_node), id_link=int(row.id_link))
    for row in unassigned.itertuples(index=False):
        undirected.add_edge(int(row.id_us_node), int(row.id_ds_node), id_link=int(row.id_link))

    records = []
    for number, component in enumerate(nx.connected_components(undirected), start=1):
        component = set(map(int, component))
        component_links = unassigned.loc[
            unassigned['id_us_node'].isin(component) & unassigned['id_ds_node'].isin(component)
        ].copy()
        entering = [(int(row.id_us_node), int(row.id_ds_node)) for row in local_links.itertuples(index=False)
                    if int(row.id_us_node) not in component and int(row.id_ds_node) in component]
        leaving = [(int(row.id_us_node), int(row.id_ds_node)) for row in local_links.itertuples(index=False)
                   if int(row.id_us_node) in component and int(row.id_ds_node) not in component]
        entries, exits = sorted({target for _, target in entering}), sorted({source for source, _ in leaving})
        record = {
            'example_id': example_id, 'connector_component_id': f'C{number:03d}',
            'n_links': len(component_links),
            'component_link_ids': ','.join(map(str, component_links['id_link'].astype(int))),
        }
        if len(entries) != 1 or len(exits) != 1:
            record['screen_status'] = 'non_simple_or_ambiguous_interface'
            records.append(record); continue
        a, b = entries[0], exits[0]
        u = nearest_external_anchor(graph, [source for source, _ in entering], accepted, component, upstream=True)
        d = nearest_external_anchor(graph, [target for _, target in leaving], accepted, component, upstream=False)
        record.update({
            'outer_upstream_anchor': u, 'connector_upstream_node': a,
            'connector_downstream_node': b, 'outer_downstream_anchor': d,
        })
        if u is None or d is None or len(component_links) != 1 or not {u, a, b, d}.issubset(profile_by_node.index):
            record['screen_status'] = 'insufficient_simple_connector_context'
            records.append(record); continue
        hu, ha, hb, hd = (float(profile_by_node.loc[node, 'profile_wse_m']) for node in (u, a, b, d))
        du, da, db, dd = (float(profile_by_node.loc[node, 'reference_section_dist_out_m']) for node in (u, a, b, d))
        d_ua, d_bd = du - da, db - dd
        length = float(component_links.iloc[0]['actual_length_m'])
        slope = (ha - hb) / length
        target = ((hu - ha) + (hb - hd)) / (d_ua + d_bd) if d_ua > 0 and d_bd > 0 else np.nan
        ratio = slope / target if np.isfinite(slope) and slope > 0 and np.isfinite(target) and target > 0 else np.nan
        record.update({
            'connector_length_m': length,
            'profile_connector_slope': slope, 'support_target_slope': target,
            'log_slope_ratio': np.log(ratio) if np.isfinite(ratio) else np.nan,
            'factor_from_target': max(ratio, 1 / ratio) if np.isfinite(ratio) else np.nan,
        })
        record['screen_status'] = (
            'nonpositive_profile_slope'
            if not np.isfinite(slope) or slope <= 0 else 'positive_context_complete'
        )
        records.append(record)
    return pd.DataFrame(records), {
        'example_id': example_id, 'n_profile_nodes': len(local_profile),
        'n_accepted_anchors': len(accepted),
    }

frozen_connector_frames, frozen_connector_runs, frozen_connector_failures = [], [], []
for frozen_example_id in FROZEN_CONNECTOR_EXAMPLE_IDS:
    try:
        frame, run = frozen_connector_screen_for_example(frozen_example_id)
        frozen_connector_frames.append(frame)
        frozen_connector_runs.append(run)
    except Exception as error:
        frozen_connector_failures.append({
            'example_id': frozen_example_id, 'error': f'{type(error).__name__}: {error}'
        })

frozen_connector_calibration = (
    pd.concat(frozen_connector_frames, ignore_index=True)
    if frozen_connector_frames else pd.DataFrame()
)
frozen_connector_run_summary = pd.DataFrame(frozen_connector_runs)
frozen_connector_failure_summary = pd.DataFrame(frozen_connector_failures)
valid_frozen_connector_ratios = frozen_connector_calibration.loc[
    frozen_connector_calibration['screen_status'].eq('positive_context_complete')
    & pd.to_numeric(frozen_connector_calibration['log_slope_ratio'], errors='coerce').notna(),
    'log_slope_ratio',
].astype(float)
if valid_frozen_connector_ratios.empty:
    raise RuntimeError('No positive context-complete frozen-profile connector ratios were available.')
connector_centre = float(valid_frozen_connector_ratios.median())
connector_mad = float(np.median(np.abs(valid_frozen_connector_ratios - connector_centre)))
connector_sigma = float(1.4826 * connector_mad)
connector_lower = float(np.exp(connector_centre - 3 * connector_sigma))
connector_upper = float(np.exp(connector_centre + 3 * connector_sigma))
connector_symmetric_factor = float(np.exp(abs(connector_centre) + 3 * connector_sigma))
connector_factor_five_coverage = float((np.exp(np.abs(valid_frozen_connector_ratios)) <= 5.0).mean())
frozen_connector_calibration_summary = pd.DataFrame([{
    'n_examples_requested': len(FROZEN_CONNECTOR_EXAMPLE_IDS),
    'n_examples_completed': len(frozen_connector_runs),
    'n_examples_failed': len(frozen_connector_failures),
    'n_positive_context_complete': len(valid_frozen_connector_ratios),
    'median_log_ratio': connector_centre,
    'robust_sigma_log_ratio': connector_sigma,
    'three_sigma_lower_ratio': connector_lower,
    'three_sigma_upper_ratio': connector_upper,
    'symmetric_three_sigma_factor': connector_symmetric_factor,
    'factor_five_coverage_fraction': connector_factor_five_coverage,
    'factor_five_coverage_percent': 100.0 * connector_factor_five_coverage,
}])
display(frozen_connector_calibration_summary)
display(frozen_connector_calibration.groupby('screen_status').size().rename('n_components').reset_index())
if not frozen_connector_failure_summary.empty:
    display(frozen_connector_failure_summary)
display(frozen_connector_calibration.loc[
    frozen_connector_calibration['screen_status'].eq('nonpositive_profile_slope')
    | frozen_connector_calibration['factor_from_target'].gt(5.0),
].sort_values(['example_id', 'factor_from_target'], ascending=[True, False]))


## Appendix A. Historical robust-fit trial status

A diagnostic trial combined the proposed count-support weight, \(`min(1, n_obs / 5)`\),
with iterative Huber residual reweighting of an *unconstrained* natural smoothing spline.
It is **not a production candidate in that two-stage form**: on Example 11 it introduced
large negative continuous derivatives before monotonic projection. A final PAVA pass would
hide rather than solve that failure.

The count-support term remains a useful diagnostic. Any robust-residual implementation
must instead be solved jointly with the monotonicity constraint, or follow a reviewed
reconstruction of the filtered pass-level RiverSP P50 inputs. This record prevents an
apparently standard Huber step from being adopted as an untested post-processing patch.
